In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

BASE = '/content/gdrive/MyDrive/Delhi-AQI-Prediction'

print("Google Drive mounted successfully.")
print("Project path:")
print(BASE)

In [ ]:
# ===========================================================================
# R.1 — FINAL RESULTS PACKAGE INITIALIZATION
# ===========================================================================
# Creates a clean reporting structure and inventories existing Phase 13–15
# artifacts. No model is trained, retuned, or evaluated here.
# ===========================================================================

import os
import json
from datetime import datetime, timezone

import pandas as pd

print("=" * 80)
print("R.1 — FINAL RESULTS PACKAGE INITIALIZATION")
print("=" * 80)

# ---------------------------------------------------------------------------
# 1. Project root
# ---------------------------------------------------------------------------
BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

if not os.path.exists(BASE):
    raise FileNotFoundError(
        f"Project root not found:\n{BASE}\n"
        "Mount Google Drive before running this cell."
    )

# ---------------------------------------------------------------------------
# 2. Final results directory
# ---------------------------------------------------------------------------
RESULTS_ROOT = os.path.join(
    BASE,
    "outputs",
    "FINAL_RESULTS"
)

folders = [
    "00_MASTER",
    "01_TRAINING_AND_VALIDATION",
    "01_TRAINING_AND_VALIDATION/tuning_results",
    "02_MODEL_RESULTS",
    "02_MODEL_RESULTS/EXTRA_TREES",
    "02_MODEL_RESULTS/RANDOM_FOREST",
    "02_MODEL_RESULTS/ENSEMBLE",
    "03_MODEL_COMPARISON",
    "04_FINAL_HOLDOUT",
    "05_SHAP_XAI",
    "06_ERROR_ANALYSIS",
    "07_PROTOCOL_AND_FREEZE",
    "08_REPORTS"
]

for folder in folders:
    os.makedirs(
        os.path.join(RESULTS_ROOT, folder),
        exist_ok=True
    )

# ---------------------------------------------------------------------------
# 3. Phase directories that contain source artifacts
# ---------------------------------------------------------------------------
SOURCE_DIRS = [
    os.path.join(BASE, "outputs", "reports"),
    os.path.join(BASE, "outputs", "checkpoints"),
    os.path.join(BASE, "models"),
    os.path.join(BASE, "deployment", "tests"),
    os.path.join(BASE, "deployment", "models")
]

# ---------------------------------------------------------------------------
# 4. Recursively inventory all existing project artifacts
# ---------------------------------------------------------------------------
inventory = []

for source_dir in SOURCE_DIRS:

    if not os.path.exists(source_dir):
        continue

    for root, dirs, files in os.walk(source_dir):

        for filename in files:

            path = os.path.join(root, filename)

            try:
                size_bytes = os.path.getsize(path)
            except OSError:
                size_bytes = None

            extension = (
                os.path.splitext(filename)[1]
                .lower()
            )

            inventory.append({
                "Source_Root": source_dir,
                "Relative_Path": os.path.relpath(
                    path,
                    BASE
                ),
                "Filename": filename,
                "Extension": extension,
                "Size_MB": (
                    size_bytes / (1024 ** 2)
                    if size_bytes is not None
                    else None
                ),
                "Exists": os.path.exists(path)
            })

inventory_df = pd.DataFrame(inventory)

if len(inventory_df) == 0:
    raise RuntimeError(
        "No existing project artifacts were found."
    )

inventory_df = inventory_df.sort_values(
    ["Relative_Path", "Filename"]
).reset_index(drop=True)

# ---------------------------------------------------------------------------
# 5. Save complete inventory
# ---------------------------------------------------------------------------
inventory_path = os.path.join(
    RESULTS_ROOT,
    "00_MASTER",
    "artifact_manifest.csv"
)

inventory_df.to_csv(
    inventory_path,
    index=False
)

# ---------------------------------------------------------------------------
# 6. Identify important phase-specific artifacts
# ---------------------------------------------------------------------------
important_mask = (
    inventory_df["Filename"].str.contains(
        r"phase15|phase14|final|ensemble|shap|xai|freeze|holdout",
        case=False,
        regex=True,
        na=False
    )
)

important_df = inventory_df[
    important_mask
].copy()

important_path = os.path.join(
    RESULTS_ROOT,
    "00_MASTER",
    "important_existing_artifacts.csv"
)

important_df.to_csv(
    important_path,
    index=False
)

# ---------------------------------------------------------------------------
# 7. Save package metadata
# ---------------------------------------------------------------------------
metadata = {
    "package_name": "FINAL_RESULTS",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "purpose": (
        "Organize and visualize completed modeling results "
        "without retraining, retuning, or changing the frozen model."
    ),

    "frozen_model_integrity": {
        "feature_count": 73,
        "feature_contract_changed": False,
        "ensemble_weights_changed": False,
        "models_retrained": False,
        "models_retuned": False,
        "holdout_used_for_new_experiments": False
    },

    "final_ensemble": {
        "extra_trees_weight": 0.594466,
        "random_forest_weight": 0.405534
    },

    "known_final_holdout": {
        "rows": 68352,
        "mae": 3.216121,
        "rmse": 7.658390,
        "r2": 0.996634,
        "within_5": 85.249883,
        "within_10": 94.605864,
        "within_20": 98.090765
    },

    "package_root": RESULTS_ROOT,
    "total_existing_artifacts_found": int(
        len(inventory_df)
    ),
    "important_existing_artifacts_found": int(
        len(important_df)
    )
}

metadata_path = os.path.join(
    RESULTS_ROOT,
    "00_MASTER",
    "results_package_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

# ---------------------------------------------------------------------------
# 8. Display summary
# ---------------------------------------------------------------------------
print("\nResults package created:")
print(RESULTS_ROOT)

print("\nFolders created:")
for folder in folders:
    print(
        "  ✓",
        os.path.join(RESULTS_ROOT, folder)
    )

print("\nExisting artifacts discovered:",
      len(inventory_df))

print("Important Phase 13–15/final artifacts:",
      len(important_df))

print("\nSaved:")
print(inventory_path)
print(important_path)
print(metadata_path)

print("\n" + "=" * 80)
print("R.1 COMPLETE")
print("=" * 80)
print("No models trained.")
print("No models retuned.")
print("No model weights changed.")
print("No holdout re-evaluation performed.")
print("Next: R.2 — Build the master model-results table and comparison plots.")

In [ ]:
# ===========================================================================
# R.2 — MASTER MODEL RESULTS + MODEL COMPARISON
# ===========================================================================
# Purpose:
#   1. Build a clean master record of completed model results.
#   2. Preserve validation-stage model-ranking information.
#   3. Preserve final frozen holdout results.
#   4. Load the final ensemble configuration.
#   5. Generate comparison plots.
#
# IMPORTANT:
#   - No model training
#   - No model tuning
#   - No new holdout evaluation
#   - FINAL_HOLDOUT remains frozen
# ===========================================================================

import os
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("R.2 — MASTER MODEL RESULTS + MODEL COMPARISON")
print("=" * 80)

# ===========================================================================
# 1. PATHS
# ===========================================================================

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

RESULTS_ROOT = os.path.join(
    BASE,
    "outputs",
    "FINAL_RESULTS"
)

MASTER_DIR = os.path.join(
    RESULTS_ROOT,
    "00_MASTER"
)

VALIDATION_DIR = os.path.join(
    RESULTS_ROOT,
    "01_TRAINING_AND_VALIDATION"
)

COMPARISON_DIR = os.path.join(
    RESULTS_ROOT,
    "03_MODEL_COMPARISON"
)

os.makedirs(MASTER_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)
os.makedirs(COMPARISON_DIR, exist_ok=True)

# ===========================================================================
# 2. SAFETY CHECK
# ===========================================================================

if not os.path.exists(RESULTS_ROOT):
    raise FileNotFoundError(
        "FINAL_RESULTS directory does not exist. Run R.1 first."
    )

# ===========================================================================
# 3. FINAL FROZEN HOLDOUT RESULTS
# ===========================================================================
# These values are copied from the already completed Phase 15.12 report.
# They are NOT recalculated here.

final_holdout_results = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees",
        "Model_Type": "Final Component",
        "Evaluation_Set": "FINAL_HOLDOUT",
        "Records": 68352,
        "MAE": 3.284691,
        "RMSE": 7.716191,
        "R2": 0.996583,
        "Accuracy_±5_AQI_%": np.nan,
        "Accuracy_±10_AQI_%": 94.395190,
        "Accuracy_±20_AQI_%": 98.023467,
        "Mean_Error": -0.420950,
        "Median_Error": -0.229714
    },
    {
        "Model": "Tuned Random Forest",
        "Model_Type": "Final Component",
        "Evaluation_Set": "FINAL_HOLDOUT",
        "Records": 68352,
        "MAE": 3.205527,
        "RMSE": 7.690637,
        "R2": 0.996605,
        "Accuracy_±5_AQI_%": np.nan,
        "Accuracy_±10_AQI_%": 94.493212,
        "Accuracy_±20_AQI_%": 98.105396,
        "Mean_Error": -0.443798,
        "Median_Error": -0.215216
    },
    {
        "Model": "Optimized Weighted Ensemble",
        "Model_Type": "FINAL MODEL",
        "Evaluation_Set": "FINAL_HOLDOUT",
        "Records": 68352,
        "MAE": 3.216121,
        "RMSE": 7.658390,
        "R2": 0.996634,
        "Accuracy_±5_AQI_%": 85.249883,
        "Accuracy_±10_AQI_%": 94.605864,
        "Accuracy_±20_AQI_%": 98.090765,
        "Mean_Error": -0.430216,
        "Median_Error": -0.229319
    },
    {
        "Model": "Persistence",
        "Model_Type": "Benchmark",
        "Evaluation_Set": "FINAL_HOLDOUT",
        "Records": 68285,
        "MAE": 3.855634,
        "RMSE": 8.840011,
        "R2": 0.995516,
        "Accuracy_±5_AQI_%": np.nan,
        "Accuracy_±10_AQI_%": 92.261844,
        "Accuracy_±20_AQI_%": 97.532401,
        "Mean_Error": -0.017163,
        "Median_Error": 0.000000
    }
])

final_holdout_path = os.path.join(
    MASTER_DIR,
    "final_holdout_model_results.csv"
)

final_holdout_results.to_csv(
    final_holdout_path,
    index=False
)

# ===========================================================================
# 4. VALIDATION MODEL RANKING
# ===========================================================================
# Phase 15.3 established the chronological validation ranking.
# Only values already recorded are entered here.
# Missing values are deliberately left as NaN rather than invented.

validation_ranking = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees / Extra Trees",
        "Evaluation_Set": "VALIDATION",
        "Validation_RMSE": 8.538368,
        "Recorded": True
    },
    {
        "Model": "HistGradientBoosting",
        "Evaluation_Set": "VALIDATION",
        "Validation_RMSE": 8.898831,
        "Recorded": True
    },
    {
        "Model": "XGBoost",
        "Evaluation_Set": "VALIDATION",
        "Validation_RMSE": 8.989854,
        "Recorded": True
    },
    {
        "Model": "Random Forest",
        "Evaluation_Set": "VALIDATION",
        "Validation_RMSE": 9.504760,
        "Recorded": True
    },
    {
        "Model": "Linear Regression",
        "Evaluation_Set": "VALIDATION",
        "Validation_RMSE": np.nan,
        "Recorded": False
    },
    {
        "Model": "Decision Tree",
        "Evaluation_Set": "VALIDATION",
        "Validation_RMSE": np.nan,
        "Recorded": False
    },
    {
        "Model": "Gradient Boosting",
        "Evaluation_Set": "VALIDATION",
        "Validation_RMSE": np.nan,
        "Recorded": False
    },
    {
        "Model": "AdaBoost",
        "Evaluation_Set": "VALIDATION",
        "Validation_RMSE": np.nan,
        "Recorded": False
    }
])

validation_ranking = validation_ranking.sort_values(
    "Validation_RMSE",
    na_position="last"
).reset_index(drop=True)

validation_ranking["Validation_Rank"] = np.nan

known_mask = validation_ranking["Validation_RMSE"].notna()

validation_ranking.loc[
    known_mask,
    "Validation_Rank"
] = np.arange(
    1,
    known_mask.sum() + 1
)

validation_path = os.path.join(
    VALIDATION_DIR,
    "validation_model_ranking.csv"
)

validation_ranking.to_csv(
    validation_path,
    index=False
)

# ===========================================================================
# 5. FINAL ENSEMBLE CONFIGURATION
# ===========================================================================

final_config_path = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_10_final_ensemble_config.json"
)

if os.path.exists(final_config_path):

    with open(final_config_path, "r") as f:
        final_config = json.load(f)

    components = final_config.get(
        "component_models",
        []
    )

    ensemble_rows = []

    for component in components:

        ensemble_rows.append({
            "Component_Model": component.get("model"),
            "Weight": component.get("weight"),
            "Source_Config": final_config_path
        })

    ensemble_config_df = pd.DataFrame(
        ensemble_rows
    )

else:

    ensemble_config_df = pd.DataFrame([
        {
            "Component_Model": "Tuned Extra Trees",
            "Weight": 0.594466,
            "Source_Config": "Recovered final configuration"
        },
        {
            "Component_Model": "Tuned Random Forest",
            "Weight": 0.405534,
            "Source_Config": "Recovered final configuration"
        }
    ])

ensemble_config_out = os.path.join(
    MASTER_DIR,
    "final_ensemble_configuration.csv"
)

ensemble_config_df.to_csv(
    ensemble_config_out,
    index=False
)

# ===========================================================================
# 6. MASTER FINAL RESULTS TABLE
# ===========================================================================

master_results = final_holdout_results.copy()

master_results["Final_Status"] = master_results[
    "Model"
].map({
    "Optimized Weighted Ensemble": "FINAL MODEL",
    "Tuned Extra Trees": "FINAL COMPONENT",
    "Tuned Random Forest": "FINAL COMPONENT",
    "Persistence": "BENCHMARK"
})

master_results_path = os.path.join(
    MASTER_DIR,
    "final_results_summary.csv"
)

master_results.to_csv(
    master_results_path,
    index=False
)

# ===========================================================================
# 7. COMBINED COMPARISON TABLE
# ===========================================================================

comparison_rows = []

for _, row in validation_ranking.iterrows():

    comparison_rows.append({
        "Model": row["Model"],
        "Validation_RMSE": row["Validation_RMSE"],
        "Final_Holdout_RMSE": np.nan,
        "Final_Holdout_MAE": np.nan,
        "Final_Holdout_R2": np.nan,
        "Final_Holdout_±10_%": np.nan,
        "Final_Holdout_±20_%": np.nan,
        "Stage": "Validation"
    })

for _, row in final_holdout_results.iterrows():

    comparison_rows.append({
        "Model": row["Model"],
        "Validation_RMSE": np.nan,
        "Final_Holdout_RMSE": row["RMSE"],
        "Final_Holdout_MAE": row["MAE"],
        "Final_Holdout_R2": row["R2"],
        "Final_Holdout_±10_%": row["Accuracy_±10_AQI_%"],
        "Final_Holdout_±20_%": row["Accuracy_±20_AQI_%"],
        "Stage": "Final Holdout"
    })

comparison_df = pd.DataFrame(
    comparison_rows
)

comparison_path = os.path.join(
    MASTER_DIR,
    "model_comparison_master.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

# ===========================================================================
# 8. PLOT 1 — FINAL HOLDOUT RMSE
# ===========================================================================

plot_df = final_holdout_results.copy()

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    plot_df["Model"],
    plot_df["RMSE"]
)

plt.ylabel(
    "RMSE"
)

plt.xlabel(
    "Model"
)

plt.title(
    "Final Holdout RMSE Comparison"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

rmse_path = os.path.join(
    COMPARISON_DIR,
    "final_holdout_rmse_comparison.png"
)

plt.savefig(
    rmse_path,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ===========================================================================
# 9. PLOT 2 — FINAL HOLDOUT MAE
# ===========================================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    plot_df["Model"],
    plot_df["MAE"]
)

plt.ylabel(
    "MAE"
)

plt.xlabel(
    "Model"
)

plt.title(
    "Final Holdout MAE Comparison"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

mae_path = os.path.join(
    COMPARISON_DIR,
    "final_holdout_mae_comparison.png"
)

plt.savefig(
    mae_path,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ===========================================================================
# 10. PLOT 3 — FINAL HOLDOUT R2
# ===========================================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    plot_df["Model"],
    plot_df["R2"]
)

plt.ylabel(
    "R²"
)

plt.xlabel(
    "Model"
)

plt.title(
    "Final Holdout R² Comparison"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

r2_path = os.path.join(
    COMPARISON_DIR,
    "final_holdout_r2_comparison.png"
)

plt.savefig(
    r2_path,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ===========================================================================
# 11. PLOT 4 — TOLERANCE ACCURACY
# ===========================================================================

tol_df = final_holdout_results[
    final_holdout_results["Model"].ne("Persistence")
].copy()

x = np.arange(
    len(tol_df)
)

width = 0.35

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    x - width / 2,
    tol_df["Accuracy_±10_AQI_%"],
    width,
    label="Within ±10 AQI"
)

plt.bar(
    x + width / 2,
    tol_df["Accuracy_±20_AQI_%"],
    width,
    label="Within ±20 AQI"
)

plt.xticks(
    x,
    tol_df["Model"],
    rotation=20,
    ha="right"
)

plt.ylabel(
    "Percentage (%)"
)

plt.xlabel(
    "Model"
)

plt.title(
    "Final Holdout Tolerance Accuracy"
)

plt.legend()

plt.tight_layout()

tolerance_path = os.path.join(
    COMPARISON_DIR,
    "final_holdout_tolerance_accuracy.png"
)

plt.savefig(
    tolerance_path,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ===========================================================================
# 12. PLOT 5 — VALIDATION RMSE
# ===========================================================================

known_validation = validation_ranking[
    validation_ranking["Validation_RMSE"].notna()
].copy()

if len(known_validation) > 0:

    known_validation = known_validation.sort_values(
        "Validation_RMSE",
        ascending=True
    )

    plt.figure(
        figsize=(10, 6)
    )

    plt.barh(
        known_validation["Model"],
        known_validation["Validation_RMSE"]
    )

    plt.xlabel(
        "Validation RMSE"
    )

    plt.ylabel(
        "Model"
    )

    plt.title(
        "Recorded Validation RMSE Comparison"
    )

    plt.tight_layout()

    validation_plot_path = os.path.join(
        COMPARISON_DIR,
        "validation_rmse_comparison.png"
    )

    plt.savefig(
        validation_plot_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

else:

    validation_plot_path = None

# ===========================================================================
# 13. MASTER PACKAGE METADATA
# ===========================================================================

metadata = {
    "phase": "R.2",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "purpose": (
        "Master completed model results and comparison visualizations."
    ),

    "model_training_performed": False,
    "model_tuning_performed": False,
    "holdout_re_evaluated": False,

    "final_holdout": {
        "rows": 68352,
        "status": "FROZEN / UNTOUCHED"
    },

    "final_model": {
        "name": "Optimized Weighted Ensemble",
        "components": {
            "Tuned Extra Trees": 0.594466,
            "Tuned Random Forest": 0.405534
        }
    },

    "outputs": {
        "final_results_summary": master_results_path,
        "final_holdout_results": final_holdout_path,
        "validation_model_ranking": validation_path,
        "ensemble_configuration": ensemble_config_out,
        "model_comparison_master": comparison_path,
        "rmse_plot": rmse_path,
        "mae_plot": mae_path,
        "r2_plot": r2_path,
        "tolerance_plot": tolerance_path,
        "validation_rmse_plot": validation_plot_path
    }
}

metadata_path = os.path.join(
    MASTER_DIR,
    "r2_results_metadata.json"
)

with open(metadata_path, "w") as f:
    json.dump(
        metadata,
        f,
        indent=2
    )

# ===========================================================================
# 14. DISPLAY RESULTS
# ===========================================================================

print("\n" + "=" * 80)
print("FINAL HOLDOUT MASTER RESULTS")
print("=" * 80)

display(
    final_holdout_results.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "Accuracy_±5_AQI_%": "{:.4f}",
        "Accuracy_±10_AQI_%": "{:.4f}",
        "Accuracy_±20_AQI_%": "{:.4f}",
        "Mean_Error": "{:.6f}",
        "Median_Error": "{:.6f}"
    })
)

print("\n" + "=" * 80)
print("RECORDED VALIDATION MODEL RANKING")
print("=" * 80)

display(
    validation_ranking.style.format({
        "Validation_RMSE": "{:.6f}",
        "Validation_Rank": "{:.0f}"
    })
)

print("\n" + "=" * 80)
print("FINAL ENSEMBLE CONFIGURATION")
print("=" * 80)

display(
    ensemble_config_df.style.format({
        "Weight": "{:.6f}"
    })
)

print("\n" + "=" * 80)
print("R.2 OUTPUTS")
print("=" * 80)

for label, path in {
    "Master final results": master_results_path,
    "Final holdout results": final_holdout_path,
    "Validation ranking": validation_path,
    "Ensemble configuration": ensemble_config_out,
    "Master comparison": comparison_path,
    "RMSE plot": rmse_path,
    "MAE plot": mae_path,
    "R² plot": r2_path,
    "Tolerance plot": tolerance_path,
    "Validation RMSE plot": validation_plot_path,
    "R.2 metadata": metadata_path
}.items():

    if path is not None:
        status = "FOUND" if os.path.exists(path) else "MISSING"
        print(f"{status:8} | {label}")
        print(f"         | {path}")

# ===========================================================================
# 15. INTEGRITY CHECK
# ===========================================================================

print("\n" + "=" * 80)
print("FROZEN MODEL INTEGRITY")
print("=" * 80)

print("73-feature contract changed : NO")
print("Ensemble weights changed    : NO")
print("Models retrained            : NO")
print("Models retuned              : NO")
print("Holdout re-evaluated        : NO")
print("Final holdout status        : FROZEN / UNTOUCHED")

print("\n" + "=" * 80)
print("R.2 COMPLETE")
print("=" * 80)
print(
    "Next: R.3 — Individual model result folders, "
    "prediction/error plots, and AQI-category confusion matrices."
)

In [ ]:
# ===========================================================================
# R.3 — INDIVIDUAL FINAL-MODEL DIAGNOSTICS
# ===========================================================================
# Uses ONLY the already saved Phase 15.12 final-holdout predictions.
#
# Models:
#   1. Tuned Extra Trees
#   2. Tuned Random Forest
#   3. Optimized Weighted Ensemble
#
# Generates:
#   - Actual vs Predicted
#   - Residual distribution
#   - Residual vs Predicted
#   - Error over time
#   - Actual vs Predicted by AQI range
#   - AQI-category confusion matrix
#   - Clean metrics CSVs
#
# IMPORTANT:
#   - No model training
#   - No model inference
#   - No model tuning
#   - No holdout re-evaluation
#   - Final holdout remains frozen
# ===========================================================================

import os
import json
import glob
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("=" * 80)
print("R.3 — INDIVIDUAL FINAL-MODEL DIAGNOSTICS")
print("=" * 80)

# ===========================================================================
# 1. PATHS
# ===========================================================================

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

FINAL_HOLDOUT_DIR = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_12_final_holdout"
)

RESULTS_ROOT = os.path.join(
    BASE,
    "outputs",
    "FINAL_RESULTS"
)

MODEL_RESULTS_ROOT = os.path.join(
    RESULTS_ROOT,
    "02_MODEL_RESULTS"
)

ET_DIR = os.path.join(
    MODEL_RESULTS_ROOT,
    "EXTRA_TREES"
)

RF_DIR = os.path.join(
    MODEL_RESULTS_ROOT,
    "RANDOM_FOREST"
)

ENSEMBLE_DIR = os.path.join(
    MODEL_RESULTS_ROOT,
    "ENSEMBLE"
)

for directory in [
    ET_DIR,
    RF_DIR,
    ENSEMBLE_DIR
]:
    os.makedirs(
        directory,
        exist_ok=True
    )

if not os.path.exists(FINAL_HOLDOUT_DIR):
    raise FileNotFoundError(
        "Phase 15.12 final holdout directory not found:\n"
        + FINAL_HOLDOUT_DIR
    )

# ===========================================================================
# 2. FIND THE EXISTING HOLDOUT PREDICTION FILE
# ===========================================================================
# We do not assume an exact filename.
# Instead, inspect the Phase 15.12 directory and locate the file that
# contains prediction columns.

candidate_files = []

for pattern in [
    "*.parquet",
    "*.csv"
]:

    candidate_files.extend(
        glob.glob(
            os.path.join(
                FINAL_HOLDOUT_DIR,
                pattern
            )
        )
    )

candidate_files = sorted(
    set(candidate_files)
)

if len(candidate_files) == 0:
    raise FileNotFoundError(
        "No CSV or Parquet files found in:\n"
        + FINAL_HOLDOUT_DIR
    )

print("\nCandidate Phase 15.12 files:")

for path in candidate_files:
    print(
        "  ",
        os.path.basename(path)
    )

# ===========================================================================
# 3. AUTOMATICALLY IDENTIFY PREDICTION FILE
# ===========================================================================

prediction_file = None
prediction_df = None

required_prediction_aliases = {
    "actual": [
        "Actual_AQI_t_plus_1",
        "Actual",
        "y_true",
        "AQI_t_plus_1"
    ],
    "et": [
        "ET_Prediction",
        "Extra_Trees_Prediction",
        "Tuned_Extra_Trees_Prediction"
    ],
    "rf": [
        "RF_Prediction",
        "Random_Forest_Prediction",
        "Tuned_Random_Forest_Prediction"
    ],
    "ensemble": [
        "Ensemble_Prediction",
        "Weighted_Ensemble_Prediction",
        "Final_Ensemble_Prediction"
    ]
}

def find_matching_column(
    columns,
    aliases
):
    lower_map = {
        str(c).lower(): c
        for c in columns
    }

    for alias in aliases:
        if alias.lower() in lower_map:
            return lower_map[alias.lower()]

    return None


for path in candidate_files:

    try:

        if path.lower().endswith(
            ".parquet"
        ):
            test_df = pd.read_parquet(
                path
            )

        else:
            test_df = pd.read_csv(
                path
            )

        actual_col = find_matching_column(
            test_df.columns,
            required_prediction_aliases["actual"]
        )

        et_col = find_matching_column(
            test_df.columns,
            required_prediction_aliases["et"]
        )

        rf_col = find_matching_column(
            test_df.columns,
            required_prediction_aliases["rf"]
        )

        ensemble_col = find_matching_column(
            test_df.columns,
            required_prediction_aliases["ensemble"]
        )

        if all(
            x is not None
            for x in [
                actual_col,
                et_col,
                rf_col,
                ensemble_col
            ]
        ):

            prediction_file = path
            prediction_df = test_df.copy()

            ACTUAL_COL = actual_col
            ET_COL = et_col
            RF_COL = rf_col
            ENSEMBLE_COL = ensemble_col

            break

    except Exception:
        continue


if prediction_file is None:

    raise RuntimeError(
        "Could not automatically identify the Phase 15.12 "
        "final prediction file.\n\n"
        "Available files:\n"
        + "\n".join(candidate_files)
    )

print("\nPrediction file identified:")
print(prediction_file)

print("\nPrediction columns:")
print("Actual    :", ACTUAL_COL)
print("ExtraTrees:", ET_COL)
print("RF        :", RF_COL)
print("Ensemble  :", ENSEMBLE_COL)

# ===========================================================================
# 4. IDENTIFY DATETIME / STATION COLUMNS
# ===========================================================================

datetime_candidates = [
    "Datetime",
    "datetime",
    "DateTime",
    "Timestamp",
    "timestamp"
]

station_candidates = [
    "StationId",
    "station_id",
    "Station"
]

DATETIME_COL = find_matching_column(
    prediction_df.columns,
    datetime_candidates
)

STATION_COL = find_matching_column(
    prediction_df.columns,
    station_candidates
)

if DATETIME_COL is not None:

    prediction_df[DATETIME_COL] = pd.to_datetime(
        prediction_df[DATETIME_COL],
        errors="coerce"
    )

    if prediction_df[DATETIME_COL].isna().any():

        raise ValueError(
            "Invalid datetime values found in final predictions."
        )

if STATION_COL is None:

    prediction_df["_Station_Fallback"] = "ALL_STATIONS"
    STATION_COL = "_Station_Fallback"

# ===========================================================================
# 5. PREPARE CLEAN ANALYSIS DATA
# ===========================================================================

analysis_df = prediction_df[
    [
        c for c in [
            DATETIME_COL,
            STATION_COL,
            ACTUAL_COL,
            ET_COL,
            RF_COL,
            ENSEMBLE_COL
        ]
        if c is not None
    ]
].copy()

analysis_df = analysis_df.rename(
    columns={
        DATETIME_COL: "Datetime",
        STATION_COL: "StationId",
        ACTUAL_COL: "Actual_AQI",
        ET_COL: "ET_Prediction",
        RF_COL: "RF_Prediction",
        ENSEMBLE_COL: "Ensemble_Prediction"
    }
)

analysis_df["Actual_AQI"] = pd.to_numeric(
    analysis_df["Actual_AQI"],
    errors="coerce"
)

for column in [
    "ET_Prediction",
    "RF_Prediction",
    "Ensemble_Prediction"
]:

    analysis_df[column] = pd.to_numeric(
        analysis_df[column],
        errors="coerce"
    )

analysis_df = analysis_df.dropna(
    subset=[
        "Actual_AQI",
        "ET_Prediction",
        "RF_Prediction",
        "Ensemble_Prediction"
    ]
).reset_index(
    drop=True
)

if len(analysis_df) != 68352:

    print(
        "\nWARNING:"
        f" Expected 68,352 rows, found {len(analysis_df):,}."
    )

else:

    print(
        "\nVerified final holdout rows:",
        f"{len(analysis_df):,}"
    )

# ===========================================================================
# 6. AQI CATEGORY FUNCTION
# ===========================================================================
# CPCB-style six-category interpretation used only for the diagnostic matrix.
#
#   0–50       Good
#   51–100     Satisfactory
#   101–200    Moderate
#   201–300    Poor
#   301–400    Very Poor
#   >400       Severe
#
# These categories are NOT used in the regression metrics.

def aqi_category(
    values
):

    values = np.asarray(
        values,
        dtype=float
    )

    categories = np.empty(
        len(values),
        dtype=object
    )

    categories[
        values <= 50
    ] = "Good"

    categories[
        (values > 50)
        & (values <= 100)
    ] = "Satisfactory"

    categories[
        (values > 100)
        & (values <= 200)
    ] = "Moderate"

    categories[
        (values > 200)
        & (values <= 300)
    ] = "Poor"

    categories[
        (values > 300)
        & (values <= 400)
    ] = "Very Poor"

    categories[
        values > 400
    ] = "Severe"

    return categories


category_order = [
    "Good",
    "Satisfactory",
    "Moderate",
    "Poor",
    "Very Poor",
    "Severe"
]

# ===========================================================================
# 7. MODEL DEFINITIONS
# ===========================================================================

models = {
    "Tuned Extra Trees": {
        "prediction_column": "ET_Prediction",
        "output_directory": ET_DIR
    },

    "Tuned Random Forest": {
        "prediction_column": "RF_Prediction",
        "output_directory": RF_DIR
    },

    "Optimized Weighted Ensemble": {
        "prediction_column": "Ensemble_Prediction",
        "output_directory": ENSEMBLE_DIR
    }
}

# ===========================================================================
# 8. DIAGNOSTIC FUNCTION
# ===========================================================================

def create_model_diagnostics(
    data,
    model_name,
    prediction_column,
    output_directory
):

    df = data[
        [
            "Datetime",
            "StationId",
            "Actual_AQI",
            prediction_column
        ]
    ].copy()

    df = df.rename(
        columns={
            prediction_column: "Prediction"
        }
    )

    df["Error"] = (
        df["Actual_AQI"]
        - df["Prediction"]
    )

    df["Absolute_Error"] = np.abs(
        df["Error"]
    )

    df["Squared_Error"] = (
        df["Error"] ** 2
    )

    # -----------------------------------------------------------------------
    # METRICS
    # -----------------------------------------------------------------------

    y_true = df["Actual_AQI"].to_numpy(
        dtype=float
    )

    y_pred = df["Prediction"].to_numpy(
        dtype=float
    )

    mae = float(
        np.mean(
            np.abs(
                y_true - y_pred
            )
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                (y_true - y_pred) ** 2
            )
        )
    )

    ss_res = float(
        np.sum(
            (y_true - y_pred) ** 2
        )
    )

    ss_tot = float(
        np.sum(
            (y_true - np.mean(y_true)) ** 2
        )
    )

    r2 = (
        1.0 - ss_res / ss_tot
        if ss_tot > 0
        else np.nan
    )

    within_5 = float(
        np.mean(
            np.abs(
                y_true - y_pred
            ) <= 5
        ) * 100
    )

    within_10 = float(
        np.mean(
            np.abs(
                y_true - y_pred
            ) <= 10
        ) * 100
    )

    within_20 = float(
        np.mean(
            np.abs(
                y_true - y_pred
            ) <= 20
        ) * 100
    )

    metrics = pd.DataFrame([
        {
            "Model": model_name,
            "Evaluation_Set": "FINAL_HOLDOUT",
            "Records": len(df),
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "Accuracy_±5_AQI_%": within_5,
            "Accuracy_±10_AQI_%": within_10,
            "Accuracy_±20_AQI_%": within_20,
            "Mean_Error": float(
                np.mean(df["Error"])
            ),
            "Median_Error": float(
                np.median(df["Error"])
            ),
            "P90_Absolute_Error": float(
                np.percentile(
                    df["Absolute_Error"],
                    90
                )
            ),
            "P95_Absolute_Error": float(
                np.percentile(
                    df["Absolute_Error"],
                    95
                )
            ),
            "P99_Absolute_Error": float(
                np.percentile(
                    df["Absolute_Error"],
                    99
                )
            ),
            "Maximum_Absolute_Error": float(
                np.max(df["Absolute_Error"])
            )
        }
    ])

    metrics_path = os.path.join(
        output_directory,
        "metrics.csv"
    )

    metrics.to_csv(
        metrics_path,
        index=False
    )

    # -----------------------------------------------------------------------
    # ADD CATEGORY
    # -----------------------------------------------------------------------

    df["Actual_Category"] = aqi_category(
        df["Actual_AQI"]
    )

    df["Predicted_Category"] = aqi_category(
        df["Prediction"]
    )

    # -----------------------------------------------------------------------
    # PLOT 1 — ACTUAL VS PREDICTED
    # -----------------------------------------------------------------------

    plt.figure(
        figsize=(9, 7)
    )

    plt.scatter(
        df["Actual_AQI"],
        df["Prediction"],
        s=8,
        alpha=0.20
    )

    maximum = max(
        float(df["Actual_AQI"].max()),
        float(df["Prediction"].max())
    )

    plt.plot(
        [0, maximum],
        [0, maximum],
        linestyle="--",
        linewidth=1.5
    )

    plt.xlabel(
        "Actual AQI(t+1)"
    )

    plt.ylabel(
        "Predicted AQI(t+1)"
    )

    plt.title(
        f"{model_name} — Actual vs Predicted"
    )

    plt.tight_layout()

    actual_predicted_path = os.path.join(
        output_directory,
        "actual_vs_predicted.png"
    )

    plt.savefig(
        actual_predicted_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # -----------------------------------------------------------------------
    # PLOT 2 — RESIDUAL DISTRIBUTION
    # -----------------------------------------------------------------------

    plt.figure(
        figsize=(9, 6)
    )

    plt.hist(
        df["Error"],
        bins=80,
        alpha=0.85
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=1.5
    )

    plt.xlabel(
        "Residual = Actual - Predicted"
    )

    plt.ylabel(
        "Frequency"
    )

    plt.title(
        f"{model_name} — Residual Distribution"
    )

    plt.tight_layout()

    residual_path = os.path.join(
        output_directory,
        "residual_distribution.png"
    )

    plt.savefig(
        residual_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # -----------------------------------------------------------------------
    # PLOT 3 — RESIDUAL VS PREDICTED
    # -----------------------------------------------------------------------

    plt.figure(
        figsize=(9, 6)
    )

    plt.scatter(
        df["Prediction"],
        df["Error"],
        s=8,
        alpha=0.20
    )

    plt.axhline(
        0,
        linestyle="--",
        linewidth=1.5
    )

    plt.xlabel(
        "Predicted AQI(t+1)"
    )

    plt.ylabel(
        "Residual"
    )

    plt.title(
        f"{model_name} — Residual vs Predicted"
    )

    plt.tight_layout()

    residual_predicted_path = os.path.join(
        output_directory,
        "residual_vs_predicted.png"
    )

    plt.savefig(
        residual_predicted_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # -----------------------------------------------------------------------
    # PLOT 4 — ERROR OVER TIME
    # -----------------------------------------------------------------------

    time_df = df.sort_values(
        "Datetime"
    ).copy()

    # Plot absolute error to make periods of large forecasting error visible.
    plt.figure(
        figsize=(12, 6)
    )

    plt.plot(
        time_df["Datetime"],
        time_df["Absolute_Error"],
        linewidth=0.6,
        alpha=0.55
    )

    rolling_window = min(
        500,
        max(
            50,
            len(time_df) // 100
        )
    )

    rolling_error = (
        time_df["Absolute_Error"]
        .rolling(
            rolling_window,
            min_periods=1
        )
        .mean()
    )

    plt.plot(
        time_df["Datetime"],
        rolling_error,
        linewidth=2
    )

    plt.xlabel(
        "Datetime"
    )

    plt.ylabel(
        "Absolute Error (AQI)"
    )

    plt.title(
        f"{model_name} — Forecast Error Over Time"
    )

    plt.tight_layout()

    error_time_path = os.path.join(
        output_directory,
        "error_over_time.png"
    )

    plt.savefig(
        error_time_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # -----------------------------------------------------------------------
    # PLOT 5 — ACTUAL VS PREDICTED BY AQI RANGE
    # -----------------------------------------------------------------------

    df["AQI_Range"] = pd.cut(
        df["Actual_AQI"],
        bins=[
            -np.inf,
            99.999999,
            199.999999,
            299.999999,
            399.999999,
            np.inf
        ],
        labels=[
            "<100",
            "100–199",
            "200–299",
            "300–399",
            "≥400"
        ]
    )

    range_summary = (
        df.groupby(
            "AQI_Range",
            observed=False
        )[
            [
                "Actual_AQI",
                "Prediction"
            ]
        ]
        .mean()
        .reset_index()
    )

    x = np.arange(
        len(range_summary)
    )

    width = 0.38

    plt.figure(
        figsize=(10, 6)
    )

    plt.bar(
        x - width / 2,
        range_summary["Actual_AQI"],
        width,
        label="Actual"
    )

    plt.bar(
        x + width / 2,
        range_summary["Prediction"],
        width,
        label="Predicted"
    )

    plt.xticks(
        x,
        range_summary["AQI_Range"]
    )

    plt.xlabel(
        "Actual AQI Range"
    )

    plt.ylabel(
        "Mean AQI"
    )

    plt.title(
        f"{model_name} — Mean Actual vs Predicted by AQI Range"
    )

    plt.legend()

    plt.tight_layout()

    range_plot_path = os.path.join(
        output_directory,
        "actual_vs_predicted_by_aqi_range.png"
    )

    plt.savefig(
        range_plot_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # -----------------------------------------------------------------------
    # PLOT 6 — AQI CATEGORY CONFUSION MATRIX
    # -----------------------------------------------------------------------

    actual_categories = pd.Categorical(
        df["Actual_Category"],
        categories=category_order,
        ordered=True
    )

    predicted_categories = pd.Categorical(
        df["Predicted_Category"],
        categories=category_order,
        ordered=True
    )

    confusion = pd.crosstab(
        actual_categories,
        predicted_categories,
        rownames=["Actual"],
        colnames=["Predicted"],
        dropna=False
    )

    confusion = confusion.reindex(
        index=category_order,
        columns=category_order,
        fill_value=0
    )

    confusion_path_csv = os.path.join(
        output_directory,
        "confusion_matrix_aqi_category.csv"
    )

    confusion.to_csv(
        confusion_path_csv
    )

    # Plot as percentages row-wise.
    row_totals = confusion.sum(
        axis=1
    ).replace(
        0,
        np.nan
    )

    confusion_percent = (
        confusion
        .div(
            row_totals,
            axis=0
        )
        * 100
    )

    matrix = confusion_percent.to_numpy(
        dtype=float
    )

    plt.figure(
        figsize=(9, 7)
    )

    image = plt.imshow(
        matrix,
        aspect="auto"
    )

    plt.colorbar(
        image,
        label="Actual-category percentage (%)"
    )

    plt.xticks(
        range(len(category_order)),
        category_order,
        rotation=35,
        ha="right"
    )

    plt.yticks(
        range(len(category_order)),
        category_order
    )

    for i in range(
        matrix.shape[0]
    ):

        for j in range(
            matrix.shape[1]
        ):

            value = matrix[i, j]

            if np.isfinite(value):

                plt.text(
                    j,
                    i,
                    f"{value:.1f}",
                    ha="center",
                    va="center",
                    fontsize=8
                )

    plt.xlabel(
        "Predicted AQI Category"
    )

    plt.ylabel(
        "Actual AQI Category"
    )

    plt.title(
        f"{model_name} — AQI Category Confusion Matrix (%)"
    )

    plt.tight_layout()

    confusion_plot_path = os.path.join(
        output_directory,
        "confusion_matrix_aqi_category.png"
    )

    plt.savefig(
        confusion_plot_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # -----------------------------------------------------------------------
    # SAVE FULL DIAGNOSTIC DATA
    # -----------------------------------------------------------------------

    diagnostic_columns = [
        "Datetime",
        "StationId",
        "Actual_AQI",
        "Prediction",
        "Error",
        "Absolute_Error",
        "Actual_Category",
        "Predicted_Category",
        "AQI_Range"
    ]

    diagnostic_path = os.path.join(
        output_directory,
        "prediction_error_diagnostics.parquet"
    )

    df[
        diagnostic_columns
    ].to_parquet(
        diagnostic_path,
        index=False
    )

    return {
        "metrics": metrics.iloc[0].to_dict(),
        "metrics_path": metrics_path,
        "actual_vs_predicted": actual_predicted_path,
        "residual_distribution": residual_path,
        "residual_vs_predicted": residual_predicted_path,
        "error_over_time": error_time_path,
        "actual_vs_predicted_by_range": range_plot_path,
        "confusion_matrix_csv": confusion_path_csv,
        "confusion_matrix_plot": confusion_plot_path,
        "diagnostic_data": diagnostic_path
    }

# ===========================================================================
# 9. RUN DIAGNOSTICS FOR THE THREE FINAL MODELS
# ===========================================================================

all_metrics = []
artifact_records = []

for model_name, config in models.items():

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)

    result = create_model_diagnostics(
        analysis_df,
        model_name,
        config["prediction_column"],
        config["output_directory"]
    )

    all_metrics.append(
        result["metrics"]
    )

    for artifact_name, artifact_path in result.items():

        if artifact_name == "metrics":
            continue

        artifact_records.append({
            "Model": model_name,
            "Artifact": artifact_name,
            "Path": artifact_path,
            "Status": (
                "FOUND"
                if os.path.exists(artifact_path)
                else "MISSING"
            )
        })

    print(
        "MAE :",
        f"{result['metrics']['MAE']:.6f}"
    )

    print(
        "RMSE:",
        f"{result['metrics']['RMSE']:.6f}"
    )

    print(
        "R²  :",
        f"{result['metrics']['R2']:.6f}"
    )

    print(
        "±10 :",
        f"{result['metrics']['Accuracy_±10_AQI_%']:.4f}%"
    )

# ===========================================================================
# 10. MASTER DIAGNOSTIC TABLE
# ===========================================================================

diagnostic_metrics_df = pd.DataFrame(
    all_metrics
)

diagnostic_metrics_df = diagnostic_metrics_df[
    [
        "Model",
        "Evaluation_Set",
        "Records",
        "MAE",
        "RMSE",
        "R2",
        "Accuracy_±5_AQI_%",
        "Accuracy_±10_AQI_%",
        "Accuracy_±20_AQI_%",
        "Mean_Error",
        "Median_Error",
        "P90_Absolute_Error",
        "P95_Absolute_Error",
        "P99_Absolute_Error",
        "Maximum_Absolute_Error"
    ]
]

master_metrics_path = os.path.join(
    MODEL_RESULTS_ROOT,
    "model_diagnostic_summary.csv"
)

diagnostic_metrics_df.to_csv(
    master_metrics_path,
    index=False
)

# ===========================================================================
# 11. MASTER ARTIFACT MANIFEST
# ===========================================================================

artifact_df = pd.DataFrame(
    artifact_records
)

artifact_manifest_path = os.path.join(
    MODEL_RESULTS_ROOT,
    "model_diagnostic_artifact_manifest.csv"
)

artifact_df.to_csv(
    artifact_manifest_path,
    index=False
)

# ===========================================================================
# 12. R.3 METADATA
# ===========================================================================

metadata = {
    "phase": "R.3",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": {
        "phase": "15.12",
        "holdout_directory": FINAL_HOLDOUT_DIR,
        "prediction_file": prediction_file,
        "rows_analyzed": int(len(analysis_df))
    },

    "models": [
        "Tuned Extra Trees",
        "Tuned Random Forest",
        "Optimized Weighted Ensemble"
    ],

    "operations": {
        "model_training": False,
        "model_inference": False,
        "model_tuning": False,
        "new_holdout_evaluation": False
    },

    "category_diagnostic": {
        "purpose": (
            "Derived regression-output AQI category diagnostic"
        ),
        "categories": category_order
    },

    "outputs": {
        "master_metrics": master_metrics_path,
        "artifact_manifest": artifact_manifest_path
    }
}

metadata_path = os.path.join(
    MODEL_RESULTS_ROOT,
    "r3_model_diagnostics_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        metadata,
        f,
        indent=2
    )

# ===========================================================================
# 13. DISPLAY MASTER RESULTS
# ===========================================================================

print("\n" + "=" * 80)
print("R.3 — MASTER DIAGNOSTIC RESULTS")
print("=" * 80)

display(
    diagnostic_metrics_df.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "Accuracy_±5_AQI_%": "{:.4f}",
        "Accuracy_±10_AQI_%": "{:.4f}",
        "Accuracy_±20_AQI_%": "{:.4f}",
        "Mean_Error": "{:.6f}",
        "Median_Error": "{:.6f}",
        "P90_Absolute_Error": "{:.6f}",
        "P95_Absolute_Error": "{:.6f}",
        "P99_Absolute_Error": "{:.6f}",
        "Maximum_Absolute_Error": "{:.6f}"
    })
)

# ===========================================================================
# 14. VERIFY ALL OUTPUTS
# ===========================================================================

print("\n" + "=" * 80)
print("R.3 ARTIFACT VERIFICATION")
print("=" * 80)

verification_df = artifact_df.copy()

display(
    verification_df
)

missing_artifacts = verification_df[
    verification_df["Status"] != "FOUND"
]

if len(missing_artifacts) > 0:

    raise RuntimeError(
        "Some R.3 artifacts are missing:\n"
        + "\n".join(
            missing_artifacts["Path"].tolist()
        )
    )

# ===========================================================================
# 15. FINAL STATUS
# ===========================================================================

print("\n" + "=" * 80)
print("R.3 COMPLETE")
print("=" * 80)

print(
    "Final holdout rows used:",
    f"{len(analysis_df):,}"
)

print(
    "Models processed:",
    len(models)
)

print(
    "Diagnostic plots per model:",
    6
)

print(
    "Confusion matrices:",
    3
)

print("\nMaster metrics:")
print(master_metrics_path)

print("\nArtifact manifest:")
print(artifact_manifest_path)

print("\nMetadata:")
print(metadata_path)

print("\n" + "=" * 80)
print("FROZEN MODEL INTEGRITY")
print("=" * 80)
print("73-feature contract changed : NO")
print("Ensemble weights changed    : NO")
print("Models retrained            : NO")
print("Models retuned              : NO")
print("Holdout re-evaluated        : NO")
print("Final holdout status        : FROZEN / UNTOUCHED")

print("\nNext: R.4 — Existing SHAP/XAI artifact packaging and SHAP visualization organization.")

In [ ]:
# ===========================================================================
# R.4 — SHAP / XAI RESULTS PACKAGING
# ===========================================================================
# Purpose:
#   1. Copy existing Phase 15.11 SHAP/XAI artifacts into FINAL_RESULTS.
#   2. Verify all expected artifacts.
#   3. Create a clean grouped-feature SHAP importance plot from the
#      already-saved SHAP importance table.
#   4. Create a compact XAI summary/manifest for reporting.
#
# IMPORTANT:
#   - No model training
#   - No model inference
#   - No model tuning
#   - No TreeExplainer execution
#   - No new holdout evaluation
#   - Existing SHAP results are reused
# ===========================================================================

import os
import json
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("R.4 — SHAP / XAI RESULTS PACKAGING")
print("=" * 80)

# ===========================================================================
# 1. PATHS
# ===========================================================================

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

SOURCE_XAI_DIR = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_11_xai"
)

SOURCE_XAI_PLOT_DIR = os.path.join(
    SOURCE_XAI_DIR,
    "plots"
)

RESULTS_ROOT = os.path.join(
    BASE,
    "outputs",
    "FINAL_RESULTS"
)

XAI_RESULTS_DIR = os.path.join(
    RESULTS_ROOT,
    "05_SHAP_XAI"
)

os.makedirs(
    XAI_RESULTS_DIR,
    exist_ok=True
)

if not os.path.exists(SOURCE_XAI_DIR):
    raise FileNotFoundError(
        "Phase 15.11 XAI directory not found:\n"
        + SOURCE_XAI_DIR
    )

# ===========================================================================
# 2. EXPECTED SOURCE ARTIFACTS
# ===========================================================================

source_artifacts = {
    "Global SHAP importance":
        os.path.join(
            SOURCE_XAI_DIR,
            "phase15_11_global_shap_importance.csv"
        ),

    "Grouped SHAP importance":
        os.path.join(
            SOURCE_XAI_DIR,
            "phase15_11_grouped_shap_importance.csv"
        ),

    "Ensemble SHAP importance":
        os.path.join(
            SOURCE_XAI_DIR,
            "phase15_11_ensemble_shap_importance.csv"
        ),

    "Local SHAP explanation":
        os.path.join(
            SOURCE_XAI_DIR,
            "phase15_11_local_shap_explanation.csv"
        ),

    "XAI sample metadata":
        os.path.join(
            SOURCE_XAI_DIR,
            "phase15_11_xai_sample_metadata.csv"
        ),

    "SHAP arrays":
        os.path.join(
            SOURCE_XAI_DIR,
            "phase15_11_shap_arrays.npz"
        ),

    "XAI summary":
        os.path.join(
            SOURCE_XAI_DIR,
            "phase15_11_xai_summary.json"
        ),

    "Extra Trees global SHAP":
        os.path.join(
            SOURCE_XAI_PLOT_DIR,
            "phase15_11_extra_trees_global_shap_bar.png"
        ),

    "Random Forest global SHAP":
        os.path.join(
            SOURCE_XAI_PLOT_DIR,
            "phase15_11_random_forest_global_shap_bar.png"
        ),

    "Ensemble global SHAP":
        os.path.join(
            SOURCE_XAI_PLOT_DIR,
            "phase15_11_ensemble_global_shap_bar.png"
        ),

    "Extra Trees SHAP beeswarm":
        os.path.join(
            SOURCE_XAI_PLOT_DIR,
            "phase15_11_extra_trees_shap_beeswarm.png"
        ),

    "Random Forest SHAP beeswarm":
        os.path.join(
            SOURCE_XAI_PLOT_DIR,
            "phase15_11_random_forest_shap_beeswarm.png"
        ),

    "Local ensemble waterfall":
        os.path.join(
            SOURCE_XAI_PLOT_DIR,
            "phase15_11_ensemble_local_waterfall.png"
        )
}

# ===========================================================================
# 3. VERIFY SOURCE ARTIFACTS
# ===========================================================================

verification = []

for label, source_path in source_artifacts.items():

    exists = os.path.exists(
        source_path
    )

    size_mb = (
        os.path.getsize(source_path)
        / (1024 ** 2)
        if exists
        else np.nan
    )

    verification.append({
        "Artifact": label,
        "Source_Path": source_path,
        "Status": "FOUND" if exists else "MISSING",
        "Size_MB": size_mb
    })

verification_df = pd.DataFrame(
    verification
)

print("\n" + "=" * 80)
print("SOURCE XAI ARTIFACT VERIFICATION")
print("=" * 80)

display(
    verification_df.style.format({
        "Size_MB": "{:.3f}"
    })
)

missing = verification_df[
    verification_df["Status"] == "MISSING"
]

if len(missing) > 0:

    raise RuntimeError(
        "Missing Phase 15.11 XAI artifacts:\n"
        + "\n".join(
            missing["Source_Path"].tolist()
        )
    )

# ===========================================================================
# 4. COPY EXISTING ARTIFACTS
# ===========================================================================

copied_records = []

destination_names = {
    "Global SHAP importance":
        "global_shap_importance.csv",

    "Grouped SHAP importance":
        "grouped_shap_importance.csv",

    "Ensemble SHAP importance":
        "ensemble_shap_importance.csv",

    "Local SHAP explanation":
        "local_shap_explanation.csv",

    "XAI sample metadata":
        "xai_sample_metadata.csv",

    "SHAP arrays":
        "shap_arrays.npz",

    "XAI summary":
        "xai_summary.json",

    "Extra Trees global SHAP":
        "extra_trees_global_shap_bar.png",

    "Random Forest global SHAP":
        "random_forest_global_shap_bar.png",

    "Ensemble global SHAP":
        "ensemble_global_shap_bar.png",

    "Extra Trees SHAP beeswarm":
        "extra_trees_shap_beeswarm.png",

    "Random Forest SHAP beeswarm":
        "random_forest_shap_beeswarm.png",

    "Local ensemble waterfall":
        "ensemble_local_waterfall.png"
}

for label, source_path in source_artifacts.items():

    destination_path = os.path.join(
        XAI_RESULTS_DIR,
        destination_names[label]
    )

    shutil.copy2(
        source_path,
        destination_path
    )

    copied_records.append({
        "Artifact": label,
        "Source_Path": source_path,
        "Destination_Path": destination_path,
        "Status": (
            "FOUND"
            if os.path.exists(destination_path)
            else "MISSING"
        ),
        "Size_MB": (
            os.path.getsize(destination_path)
            / (1024 ** 2)
            if os.path.exists(destination_path)
            else np.nan
        )
    })

copied_df = pd.DataFrame(
    copied_records
)

# ===========================================================================
# 5. LOAD GROUPED SHAP IMPORTANCE TABLE
# ===========================================================================

grouped_path = os.path.join(
    XAI_RESULTS_DIR,
    "grouped_shap_importance.csv"
)

grouped_df = pd.read_csv(
    grouped_path
)

required_group_columns = [
    "Original_Feature",
    "Ensemble_Mean_Abs_SHAP"
]

missing_group_columns = [
    c for c in required_group_columns
    if c not in grouped_df.columns
]

if missing_group_columns:

    raise KeyError(
        "Required SHAP columns missing:\n"
        + "\n".join(missing_group_columns)
    )

grouped_df = grouped_df.copy()

grouped_df[
    "Ensemble_Mean_Abs_SHAP"
] = pd.to_numeric(
    grouped_df["Ensemble_Mean_Abs_SHAP"],
    errors="coerce"
)

grouped_df = grouped_df.dropna(
    subset=[
        "Original_Feature",
        "Ensemble_Mean_Abs_SHAP"
    ]
)

grouped_df = grouped_df.sort_values(
    "Ensemble_Mean_Abs_SHAP",
    ascending=False
).reset_index(
    drop=True
)

# ===========================================================================
# 6. GROUPED ENSEMBLE SHAP PLOT
# ===========================================================================
# This is a visualization of the already-generated SHAP results.
# No SHAP calculation occurs here.

top_n = min(
    20,
    len(grouped_df)
)

plot_df = grouped_df.head(
    top_n
).iloc[::-1]

plt.figure(
    figsize=(10, 8)
)

plt.barh(
    plot_df["Original_Feature"],
    plot_df["Ensemble_Mean_Abs_SHAP"]
)

plt.xlabel(
    "Mean |SHAP value|"
)

plt.ylabel(
    "Original Feature"
)

plt.title(
    "Final Ensemble — Global SHAP Importance by Original Feature"
)

plt.tight_layout()

grouped_plot_path = os.path.join(
    XAI_RESULTS_DIR,
    "ensemble_grouped_shap_importance.png"
)

plt.savefig(
    grouped_plot_path,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ===========================================================================
# 7. TOP FEATURE TABLE FOR REPORTING
# ===========================================================================

top_features_df = grouped_df.head(
    20
).copy()

top_features_path = os.path.join(
    XAI_RESULTS_DIR,
    "top_20_ensemble_shap_features.csv"
)

top_features_df.to_csv(
    top_features_path,
    index=False
)

# ===========================================================================
# 8. LOAD EXISTING XAI SUMMARY
# ===========================================================================

xai_summary_source = os.path.join(
    XAI_RESULTS_DIR,
    "xai_summary.json"
)

with open(
    xai_summary_source,
    "r"
) as f:

    xai_summary = json.load(
        f
    )

# ===========================================================================
# 9. CREATE XAI README
# ===========================================================================

readme_path = os.path.join(
    XAI_RESULTS_DIR,
    "README.md"
)

readme_text = f"""# SHAP / XAI Results

This directory contains the explainability artifacts generated during
Phase 15.11 for the frozen final model system.

## Models explained

- Tuned Extra Trees
- Tuned Random Forest
- Optimized Weighted Ensemble

## XAI data source

Phase 15.11 used a reproducible sample from the VALIDATION split.
The FINAL_HOLDOUT was explicitly excluded from XAI analysis.

## Existing SHAP artifacts

- Extra Trees global SHAP bar
- Random Forest global SHAP bar
- Ensemble global SHAP bar
- Extra Trees SHAP beeswarm
- Random Forest SHAP beeswarm
- Local ensemble SHAP waterfall
- Global SHAP importance tables
- Grouped SHAP importance
- Local SHAP explanation
- SHAP arrays
- XAI sample metadata
- XAI summary

## Additional reporting visualization

`ensemble_grouped_shap_importance.png`

This plot is generated directly from the already-saved grouped SHAP
importance table. No new SHAP calculation was performed.

## Frozen-model integrity

Feature contract: 73 features
Ensemble weights: ET 0.594466 / RF 0.405534
Final holdout: FROZEN / UNTOUCHED
Model retraining: NO
Model retuning: NO
New holdout evaluation: NO

Created by R.4 on {datetime.now(timezone.utc).isoformat()}.
"""

with open(
    readme_path,
    "w"
) as f:

    f.write(
        readme_text
    )

# ===========================================================================
# 10. CREATE SHAP ARTIFACT MANIFEST
# ===========================================================================

manifest_records = []

for record in copied_records:

    manifest_records.append({
        "Artifact": record["Artifact"],
        "Path": record["Destination_Path"],
        "Type": (
            os.path.splitext(
                record["Destination_Path"]
            )[1].lower()
        ),
        "Status": record["Status"],
        "Size_MB": record["Size_MB"],
        "Source_Phase": "15.11"
    })

manifest_records.append({
    "Artifact":
        "Ensemble grouped SHAP importance plot",
    "Path":
        grouped_plot_path,
    "Type":
        ".png",
    "Status":
        "FOUND"
        if os.path.exists(grouped_plot_path)
        else "MISSING",
    "Size_MB":
        (
            os.path.getsize(grouped_plot_path)
            / (1024 ** 2)
            if os.path.exists(grouped_plot_path)
            else np.nan
        ),
    "Source_Phase":
        "R.4 visualization packaging"
})

manifest_records.append({
    "Artifact":
        "Top 20 ensemble SHAP features",
    "Path":
        top_features_path,
    "Type":
        ".csv",
    "Status":
        "FOUND"
        if os.path.exists(top_features_path)
        else "MISSING",
    "Size_MB":
        (
            os.path.getsize(top_features_path)
            / (1024 ** 2)
            if os.path.exists(top_features_path)
            else np.nan
        ),
    "Source_Phase":
        "R.4"
})

manifest_df = pd.DataFrame(
    manifest_records
)

manifest_path = os.path.join(
    XAI_RESULTS_DIR,
    "shap_xai_artifact_manifest.csv"
)

manifest_df.to_csv(
    manifest_path,
    index=False
)

# ===========================================================================
# 11. CREATE R.4 METADATA
# ===========================================================================

r4_metadata = {
    "phase": "R.4",
    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "purpose":
        "Package existing Phase 15.11 SHAP/XAI artifacts.",

    "source_phase":
        "15.11",

    "xai_source_split":
        xai_summary.get(
            "xai_data",
            {}
        ).get(
            "source_split",
            "VALIDATION"
        ),

    "final_holdout_used":
        False,

    "model_training":
        False,

    "model_tuning":
        False,

    "new_holdout_evaluation":
        False,

    "models": [
        "Tuned Extra Trees",
        "Tuned Random Forest",
        "Optimized Weighted Ensemble"
    ],

    "ensemble_weights": {
        "Tuned Extra Trees": 0.594466,
        "Tuned Random Forest": 0.405534
    },

    "original_feature_count":
        73,

    "copied_artifacts":
        int(
            len(copied_records)
        ),

    "additional_visualizations":
        1,

    "output_directory":
        XAI_RESULTS_DIR,

    "manifest":
        manifest_path,

    "readme":
        readme_path
}

metadata_path = os.path.join(
    XAI_RESULTS_DIR,
    "r4_shap_xai_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        r4_metadata,
        f,
        indent=2
    )

# ===========================================================================
# 12. FINAL VERIFICATION
# ===========================================================================

print("\n" + "=" * 80)
print("R.4 — FINAL XAI ARTIFACT VERIFICATION")
print("=" * 80)

final_missing = manifest_df[
    manifest_df["Status"] != "FOUND"
]

display(
    manifest_df.style.format({
        "Size_MB": "{:.3f}"
    })
)

if len(final_missing) > 0:

    raise RuntimeError(
        "Some R.4 artifacts are missing."
    )

# ===========================================================================
# 13. DISPLAY TOP SHAP FEATURES
# ===========================================================================

display_columns = [
    c for c in [
        "Original_Feature",
        "Ensemble_Mean_Abs_SHAP",
        "Ensemble_Relative_Importance_%"
    ]
    if c in top_features_df.columns
]

print("\n" + "=" * 80)
print("TOP 20 ENSEMBLE SHAP FEATURES")
print("=" * 80)

display(
    top_features_df[
        display_columns
    ].style.format({
        "Ensemble_Mean_Abs_SHAP": "{:.6f}",
        "Ensemble_Relative_Importance_%": "{:.4f}"
    })
)

# ===========================================================================
# 14. STATUS
# ===========================================================================

print("\n" + "=" * 80)
print("R.4 COMPLETE")
print("=" * 80)

print(
    "Existing SHAP artifacts packaged:",
    len(copied_records)
)

print(
    "Additional grouped SHAP plot:",
    grouped_plot_path
)

print(
    "Top-20 SHAP table:",
    top_features_path
)

print(
    "Manifest:",
    manifest_path
)

print(
    "Metadata:",
    metadata_path
)

print(
    "README:",
    readme_path
)

print("\n" + "=" * 80)
print("FROZEN MODEL INTEGRITY")
print("=" * 80)

print("73-feature contract changed : NO")
print("Ensemble weights changed    : NO")
print("Models retrained            : NO")
print("Models retuned              : NO")
print("Holdout re-evaluated        : NO")
print("Final holdout status        : FROZEN / UNTOUCHED")

print("\n" + "=" * 80)
print("NEXT: R.5 — Station-level + AQI-range + error-analysis reporting")
print("=" * 80)

In [ ]:
# ===========================================================================
# R.4A — LOCATE EXISTING SHAP / XAI ARTIFACTS
# ===========================================================================
# Purpose:
#   Find the already-generated Phase 15.11 SHAP/XAI files anywhere inside
#   the project directory.
#
# IMPORTANT:
#   - No SHAP calculation
#   - No model loading
#   - No training
#   - No holdout evaluation
# ===========================================================================

import os
import pandas as pd

print("=" * 80)
print("R.4A — LOCATE EXISTING SHAP / XAI ARTIFACTS")
print("=" * 80)

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

if not os.path.exists(BASE):
    raise FileNotFoundError(
        f"Project directory not found:\n{BASE}"
    )

# ---------------------------------------------------------------------------
# Exact filenames expected from Phase 15.11
# ---------------------------------------------------------------------------

target_filenames = [
    "phase15_11_global_shap_importance.csv",
    "phase15_11_grouped_shap_importance.csv",
    "phase15_11_ensemble_shap_importance.csv",
    "phase15_11_local_shap_explanation.csv",
    "phase15_11_xai_sample_metadata.csv",
    "phase15_11_shap_arrays.npz",
    "phase15_11_xai_summary.json",
    "phase15_11_extra_trees_global_shap_bar.png",
    "phase15_11_random_forest_global_shap_bar.png",
    "phase15_11_ensemble_global_shap_bar.png",
    "phase15_11_extra_trees_shap_beeswarm.png",
    "phase15_11_random_forest_shap_beeswarm.png",
    "phase15_11_ensemble_local_waterfall.png"
]

target_set = set(target_filenames)

# ---------------------------------------------------------------------------
# Recursive search
# ---------------------------------------------------------------------------

matches = []

for root, dirs, files in os.walk(BASE):

    for filename in files:

        if filename in target_set:

            path = os.path.join(
                root,
                filename
            )

            try:
                size_mb = (
                    os.path.getsize(path)
                    / (1024 ** 2)
                )
            except OSError:
                size_mb = None

            matches.append({
                "Filename": filename,
                "Found": True,
                "Path": path,
                "Size_MB": size_mb
            })

# ---------------------------------------------------------------------------
# Build complete status table
# ---------------------------------------------------------------------------

found_map = {
    row["Filename"]: row
    for row in matches
}

status_rows = []

for filename in target_filenames:

    if filename in found_map:

        row = found_map[filename].copy()

        status_rows.append(row)

    else:

        status_rows.append({
            "Filename": filename,
            "Found": False,
            "Path": None,
            "Size_MB": None
        })

status_df = pd.DataFrame(
    status_rows
)

print("\n" + "=" * 80)
print("SHAP / XAI FILE DISCOVERY")
print("=" * 80)

display(
    status_df.style.format({
        "Size_MB": "{:.3f}"
    })
)

# ---------------------------------------------------------------------------
# Print exact found paths
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("FOUND FILES")
print("=" * 80)

found_df = status_df[
    status_df["Found"]
].copy()

for _, row in found_df.iterrows():

    print(
        f"\n{row['Filename']}"
    )

    print(
        f"  {row['Path']}"
    )

# ---------------------------------------------------------------------------
# Save discovery result
# ---------------------------------------------------------------------------

RESULTS_ROOT = os.path.join(
    BASE,
    "outputs",
    "FINAL_RESULTS"
)

MASTER_DIR = os.path.join(
    RESULTS_ROOT,
    "00_MASTER"
)

os.makedirs(
    MASTER_DIR,
    exist_ok=True
)

discovery_path = os.path.join(
    MASTER_DIR,
    "r4a_shap_file_discovery.csv"
)

status_df.to_csv(
    discovery_path,
    index=False
)

# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("R.4A SUMMARY")
print("=" * 80)

print(
    "Expected SHAP/XAI files:",
    len(target_filenames)
)

print(
    "Found:",
    int(status_df["Found"].sum())
)

print(
    "Missing:",
    int((~status_df["Found"]).sum())
)

print(
    "\nDiscovery report:"
)

print(
    discovery_path
)

print("\n" + "=" * 80)
print("R.4A COMPLETE")
print("=" * 80)

print(
    "No SHAP calculations performed."
)

print(
    "No models loaded."
)

print(
    "No training/tuning performed."
)

print(
    "No holdout evaluation performed."
)

In [ ]:
# ============================================================================
# R.5 — COMPREHENSIVE MODEL COMPARISON + FINAL MODEL DOCUMENTATION
# ============================================================================
# Purpose:
#   Create the complete reporting package for the Delhi AQI project.
#
# Includes:
#   1. All Phase 15 base-model comparison
#   2. Tuned RF / ET / XGB comparison
#   3. 50/50 and optimized ensemble comparison
#   4. Earlier Phase 12–14 model comparison
#   5. Final frozen holdout comparison
#   6. Final model registry
#   7. Final model outline
#   8. Final comparison report
#   9. Report-ready comparison charts
#
# IMPORTANT:
#   - No model training
#   - No model tuning
#   - No model inference
#   - No holdout re-evaluation
#   - Final holdout remains FROZEN
# ============================================================================

import os
import json
import textwrap
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 80)
print("R.5 — COMPREHENSIVE MODEL COMPARISON + FINAL MODEL DOCUMENTATION")
print("=" * 80)

# ============================================================================
# 1. PROJECT PATHS
# ============================================================================

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

RESULTS_ROOT = os.path.join(
    BASE,
    "outputs",
    "FINAL_RESULTS"
)

MASTER_DIR = os.path.join(
    RESULTS_ROOT,
    "00_MASTER"
)

TRAINING_DIR = os.path.join(
    RESULTS_ROOT,
    "01_TRAINING_AND_VALIDATION"
)

COMPARISON_DIR = os.path.join(
    RESULTS_ROOT,
    "03_MODEL_COMPARISON"
)

HOLDOUT_DIR = os.path.join(
    RESULTS_ROOT,
    "04_FINAL_HOLDOUT"
)

PROTOCOL_DIR = os.path.join(
    RESULTS_ROOT,
    "07_PROTOCOL_AND_FREEZE"
)

REPORT_DIR = os.path.join(
    RESULTS_ROOT,
    "08_REPORTS"
)

for directory in [
    MASTER_DIR,
    TRAINING_DIR,
    COMPARISON_DIR,
    HOLDOUT_DIR,
    PROTOCOL_DIR,
    REPORT_DIR
]:
    os.makedirs(
        directory,
        exist_ok=True
    )

# ============================================================================
# 2. FINAL MODEL PATHS
# ============================================================================

FINAL_ET_MODEL = os.path.join(
    BASE,
    "models",
    "phase15_10_final_tuned_extra_trees_trainval.joblib"
)

FINAL_RF_MODEL = os.path.join(
    BASE,
    "models",
    "phase15_10_final_tuned_random_forest_trainval.joblib"
)

FINAL_ENSEMBLE_CONFIG = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_10_final_ensemble_config.json"
)

FINAL_FREEZE_JSON = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_15_final_freeze",
    "phase15_15_final_freeze.json"
)

FEATURE_CONTRACT = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_15_final_freeze",
    "phase15_15_feature_contract.csv"
)

SPLIT_PROTOCOL = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_1_evaluation_protocol.json"
)

TIME_SPLIT_FILE = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_1_time_split_assignments.parquet"
)

# ============================================================================
# 3. LOAD FINAL CONFIGURATION
# ============================================================================

ensemble_weight_et = 0.594466
ensemble_weight_rf = 0.405534

final_config = None

if os.path.exists(FINAL_ENSEMBLE_CONFIG):

    with open(
        FINAL_ENSEMBLE_CONFIG,
        "r"
    ) as f:

        final_config = json.load(f)

    try:

        components = final_config[
            "component_models"
        ]

        for component in components:

            model_name = component.get("model")

            weight = float(
                component.get("weight")
            )

            if model_name == "Tuned Extra Trees":
                ensemble_weight_et = weight

            elif model_name == "Tuned Random Forest":
                ensemble_weight_rf = weight

    except Exception:

        pass

if not np.isclose(
    ensemble_weight_et + ensemble_weight_rf,
    1.0,
    atol=1e-6
):

    raise ValueError(
        "Final ensemble weights do not sum to 1."
    )

# ============================================================================
# 4. MODEL FILE REGISTRY
# ============================================================================

def file_size_mb(
    path
):

    if not os.path.exists(path):
        return np.nan

    return (
        os.path.getsize(path)
        / (1024 ** 2)
    )

final_model_registry = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees",
        "Role": "Final component model",
        "Stored": os.path.exists(FINAL_ET_MODEL),
        "Path": FINAL_ET_MODEL,
        "Size_MB": file_size_mb(FINAL_ET_MODEL),
        "Final_Ensemble_Weight": ensemble_weight_et
    },
    {
        "Model": "Tuned Random Forest",
        "Role": "Final component model",
        "Stored": os.path.exists(FINAL_RF_MODEL),
        "Path": FINAL_RF_MODEL,
        "Size_MB": file_size_mb(FINAL_RF_MODEL),
        "Final_Ensemble_Weight": ensemble_weight_rf
    },
    {
        "Model": "Optimized Weighted Ensemble",
        "Role": "FINAL MODEL",
        "Stored": os.path.exists(FINAL_ENSEMBLE_CONFIG),
        "Path": FINAL_ENSEMBLE_CONFIG,
        "Size_MB": file_size_mb(FINAL_ENSEMBLE_CONFIG),
        "Final_Ensemble_Weight": 1.0
    }
])

model_registry_path = os.path.join(
    MASTER_DIR,
    "final_model_registry.csv"
)

final_model_registry.to_csv(
    model_registry_path,
    index=False
)

# ============================================================================
# 5. PHASE 15 BASE MODELS
# ============================================================================
# Values are included only where they were retained reliably.
#
# Known Phase 15 validation RMSE:
#   Extra Trees          8.538368
#   HistGradientBoosting 8.898831
#   XGBoost              8.989854
#   Random Forest        9.504760
#
# Other Phase 15 base-model values were not reliably retained in the
# available reporting artifacts, so they remain NaN.

phase15_base_models = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "Model_Family": "Linear",
        "Stage": "Phase 15 Base Model",
        "Validation_RMSE": np.nan,
        "Metric_Status": "Not reliably retained"
    },
    {
        "Model": "Decision Tree",
        "Model_Family": "Tree",
        "Stage": "Phase 15 Base Model",
        "Validation_RMSE": np.nan,
        "Metric_Status": "Not reliably retained"
    },
    {
        "Model": "Random Forest",
        "Model_Family": "Bagging Tree",
        "Stage": "Phase 15 Base Model",
        "Validation_RMSE": 9.504760,
        "Metric_Status": "Recorded"
    },
    {
        "Model": "Extra Trees",
        "Model_Family": "Randomized Tree Ensemble",
        "Stage": "Phase 15 Base Model",
        "Validation_RMSE": 8.538368,
        "Metric_Status": "Recorded"
    },
    {
        "Model": "Gradient Boosting",
        "Model_Family": "Boosting",
        "Stage": "Phase 15 Base Model",
        "Validation_RMSE": np.nan,
        "Metric_Status": "Not reliably retained"
    },
    {
        "Model": "HistGradientBoosting",
        "Model_Family": "Boosting",
        "Stage": "Phase 15 Base Model",
        "Validation_RMSE": 8.898831,
        "Metric_Status": "Recorded"
    },
    {
        "Model": "AdaBoost",
        "Model_Family": "Boosting",
        "Stage": "Phase 15 Base Model",
        "Validation_RMSE": np.nan,
        "Metric_Status": "Not reliably retained"
    },
    {
        "Model": "XGBoost",
        "Model_Family": "Boosting",
        "Stage": "Phase 15 Base Model",
        "Validation_RMSE": 8.989854,
        "Metric_Status": "Recorded"
    }
])

base_models_path = os.path.join(
    TRAINING_DIR,
    "phase15_base_models_comparison.csv"
)

phase15_base_models.to_csv(
    base_models_path,
    index=False
)

# ============================================================================
# 6. TUNED MODEL COMPARISON
# ============================================================================

tuned_models = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees",
        "Stage": "Phase 15 Tuned",
        "Search_RMSE": 8.340532,
        "Refit_Validation_RMSE": 8.286682,
        "Selected_for_Final_Ensemble": True,
        "Final_Holdout_RMSE": 7.716191,
        "Final_Holdout_MAE": 3.284691,
        "Final_Holdout_R2": 0.996583,
        "Final_Holdout_±10_%": 94.395190,
        "Final_Holdout_±20_%": 98.023467
    },
    {
        "Model": "Tuned Random Forest",
        "Stage": "Phase 15 Tuned",
        "Search_RMSE": 8.342435,
        "Refit_Validation_RMSE": 8.331690,
        "Selected_for_Final_Ensemble": True,
        "Final_Holdout_RMSE": 7.690637,
        "Final_Holdout_MAE": 3.205527,
        "Final_Holdout_R2": 0.996605,
        "Final_Holdout_±10_%": 94.493212,
        "Final_Holdout_±20_%": 98.105396
    },
    {
        "Model": "Tuned XGBoost",
        "Stage": "Phase 15 Tuned",
        "Search_RMSE": 9.102745,
        "Refit_Validation_RMSE": 8.901796,
        "Selected_for_Final_Ensemble": False,
        "Final_Holdout_RMSE": np.nan,
        "Final_Holdout_MAE": np.nan,
        "Final_Holdout_R2": np.nan,
        "Final_Holdout_±10_%": np.nan,
        "Final_Holdout_±20_%": np.nan
    }
])

tuned_models_path = os.path.join(
    TRAINING_DIR,
    "phase15_tuned_models_comparison.csv"
)

tuned_models.to_csv(
    tuned_models_path,
    index=False
)

# ============================================================================
# 7. ENSEMBLE DEVELOPMENT COMPARISON
# ============================================================================

ensemble_development = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees",
        "Stage": "Top-two component",
        "Validation_RMSE": 8.286682,
        "Weight": ensemble_weight_et
    },
    {
        "Model": "Tuned Random Forest",
        "Stage": "Top-two component",
        "Validation_RMSE": 8.331690,
        "Weight": ensemble_weight_rf
    },
    {
        "Model": "50/50 Ensemble",
        "Stage": "Ensemble experiment",
        "Validation_RMSE": 8.249447,
        "Weight": 0.5
    },
    {
        "Model": "Optimized Weighted Ensemble",
        "Stage": "FINAL MODEL",
        "Validation_RMSE": 8.247305,
        "Weight": np.nan
    }
])

ensemble_dev_path = os.path.join(
    TRAINING_DIR,
    "ensemble_development_comparison.csv"
)

ensemble_development.to_csv(
    ensemble_dev_path,
    index=False
)

# ============================================================================
# 8. EARLIER PHASE 12–14 COMPARISON
# ============================================================================
# These are the retained serious-model results from the earlier
# final-10-station experiments.

historical_models = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "Stage": "Phase 12–14",
        "MAE": 4.301847,
        "RMSE": 10.198616,
        "R2": 0.993241,
        "Accuracy_±10_%": np.nan,
        "Source_Status": "Recorded"
    },
    {
        "Model": "Decision Tree",
        "Stage": "Phase 12–14",
        "MAE": 4.505284,
        "RMSE": 10.108943,
        "R2": 0.993360,
        "Accuracy_±10_%": np.nan,
        "Source_Status": "Recorded"
    },
    {
        "Model": "Random Forest",
        "Stage": "Phase 12–14",
        "MAE": 3.497741,
        "RMSE": 8.457054,
        "R2": 0.995353,
        "Accuracy_±10_%": np.nan,
        "Source_Status": "Recorded"
    },
    {
        "Model": "XGBoost",
        "Stage": "Phase 12–14",
        "MAE": 4.010653,
        "RMSE": 8.827806,
        "R2": 0.994936,
        "Accuracy_±10_%": np.nan,
        "Source_Status": "Recorded"
    },
    {
        "Model": "Persistence",
        "Stage": "Phase 12–14",
        "MAE": 3.917885,
        "RMSE": 9.418282,
        "R2": 0.994236,
        "Accuracy_±10_%": np.nan,
        "Source_Status": "Recorded"
    },
    {
        "Model": "LSTM",
        "Stage": "Phase 12–14",
        "MAE": 8.095820,
        "RMSE": 13.862160,
        "R2": 0.987523,
        "Accuracy_±10_%": np.nan,
        "Source_Status": "Recorded"
    },
    {
        "Model": "CNN-BiLSTM",
        "Stage": "Phase 12–14",
        "MAE": 8.407335,
        "RMSE": 13.849509,
        "R2": 0.987546,
        "Accuracy_±10_%": np.nan,
        "Source_Status": "Recorded"
    }
])

historical_path = os.path.join(
    COMPARISON_DIR,
    "phase12_14_historical_model_comparison.csv"
)

historical_models.to_csv(
    historical_path,
    index=False
)

# ============================================================================
# 9. FINAL FROZEN HOLDOUT RESULTS
# ============================================================================

final_holdout = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees",
        "Stage": "FINAL HOLDOUT",
        "Model_Role": "Final component",
        "Records": 68352,
        "MAE": 3.284691,
        "RMSE": 7.716191,
        "R2": 0.996583,
        "Accuracy_±5_%": 84.699800,
        "Accuracy_±10_%": 94.395190,
        "Accuracy_±20_%": 98.023467
    },
    {
        "Model": "Tuned Random Forest",
        "Stage": "FINAL HOLDOUT",
        "Model_Role": "Final component",
        "Records": 68352,
        "MAE": 3.205527,
        "RMSE": 7.690637,
        "R2": 0.996605,
        "Accuracy_±5_%": 85.405000,
        "Accuracy_±10_%": 94.493212,
        "Accuracy_±20_%": 98.105396
    },
    {
        "Model": "Optimized Weighted Ensemble",
        "Stage": "FINAL HOLDOUT",
        "Model_Role": "FINAL MODEL",
        "Records": 68352,
        "MAE": 3.216121,
        "RMSE": 7.658390,
        "R2": 0.996634,
        "Accuracy_±5_%": 85.249883,
        "Accuracy_±10_%": 94.605864,
        "Accuracy_±20_%": 98.090765
    },
    {
        "Model": "Persistence",
        "Stage": "FINAL HOLDOUT",
        "Model_Role": "Benchmark",
        "Records": 68285,
        "MAE": 3.855634,
        "RMSE": 8.840011,
        "R2": 0.995516,
        "Accuracy_±5_%": np.nan,
        "Accuracy_±10_%": 92.261844,
        "Accuracy_±20_%": 97.532401
    }
])

final_holdout_path = os.path.join(
    HOLDOUT_DIR,
    "final_frozen_holdout_comparison.csv"
)

final_holdout.to_csv(
    final_holdout_path,
    index=False
)

# ============================================================================
# 10. COMPREHENSIVE MASTER COMPARISON
# ============================================================================

master_rows = []

# Phase 12–14
for _, row in historical_models.iterrows():

    master_rows.append({
        "Model": row["Model"],
        "Comparison_Group": "Phase 12–14 Historical Comparison",
        "Stage": row["Stage"],
        "MAE": row["MAE"],
        "RMSE": row["RMSE"],
        "R2": row["R2"],
        "Accuracy_±10_%": row["Accuracy_±10_%"],
        "Accuracy_±20_%": np.nan,
        "Selected_Final": False
    })

# Phase 15 base
for _, row in phase15_base_models.iterrows():

    master_rows.append({
        "Model": row["Model"],
        "Comparison_Group": "Phase 15 Base Models",
        "Stage": row["Stage"],
        "MAE": np.nan,
        "RMSE": row["Validation_RMSE"],
        "R2": np.nan,
        "Accuracy_±10_%": np.nan,
        "Accuracy_±20_%": np.nan,
        "Selected_Final": False
    })

# Tuned models
for _, row in tuned_models.iterrows():

    master_rows.append({
        "Model": row["Model"],
        "Comparison_Group": "Phase 15 Tuned Models",
        "Stage": row["Stage"],
        "MAE": row["Final_Holdout_MAE"],
        "RMSE": (
            row["Final_Holdout_RMSE"]
            if pd.notna(row["Final_Holdout_RMSE"])
            else row["Refit_Validation_RMSE"]
        ),
        "R2": row["Final_Holdout_R2"],
        "Accuracy_±10_%": row["Final_Holdout_±10_%"],
        "Accuracy_±20_%": row["Final_Holdout_±20_%"],
        "Selected_Final": bool(
            row["Selected_for_Final_Ensemble"]
        )
    })

# Ensemble development
for _, row in ensemble_development.iterrows():

    master_rows.append({
        "Model": row["Model"],
        "Comparison_Group": "Ensemble Development",
        "Stage": row["Stage"],
        "MAE": np.nan,
        "RMSE": row["Validation_RMSE"],
        "R2": np.nan,
        "Accuracy_±10_%": np.nan,
        "Accuracy_±20_%": np.nan,
        "Selected_Final": row["Model"] == "Optimized Weighted Ensemble"
    })

# Final holdout
for _, row in final_holdout.iterrows():

    master_rows.append({
        "Model": row["Model"],
        "Comparison_Group": "Final Frozen Holdout",
        "Stage": row["Stage"],
        "MAE": row["MAE"],
        "RMSE": row["RMSE"],
        "R2": row["R2"],
        "Accuracy_±10_%": row["Accuracy_±10_%"],
        "Accuracy_±20_%": row["Accuracy_±20_%"],
        "Selected_Final": (
            row["Model"] == "Optimized Weighted Ensemble"
        )
    })

comprehensive = pd.DataFrame(
    master_rows
)

comprehensive_path = os.path.join(
    COMPARISON_DIR,
    "comprehensive_model_comparison.csv"
)

comprehensive.to_csv(
    comprehensive_path,
    index=False
)

# ============================================================================
# 11. PLOT — PHASE 15 VALIDATION RMSE FOR KNOWN BASE/TUNED/ENSEMBLE RESULTS
# ============================================================================

validation_plot_rows = []

for _, row in phase15_base_models.iterrows():

    if pd.notna(
        row["Validation_RMSE"]
    ):

        validation_plot_rows.append({
            "Model": row["Model"],
            "RMSE": row["Validation_RMSE"],
            "Group": "Base"
        })

for _, row in tuned_models.iterrows():

    validation_plot_rows.append({
        "Model": row["Model"],
        "RMSE": row["Refit_Validation_RMSE"],
        "Group": "Tuned"
    })

validation_plot_rows.extend([
    {
        "Model": "50/50 Ensemble",
        "RMSE": 8.249447,
        "Group": "Ensemble"
    },
    {
        "Model": "Optimized Weighted Ensemble",
        "RMSE": 8.247305,
        "Group": "Ensemble"
    }
])

validation_plot_df = pd.DataFrame(
    validation_plot_rows
).sort_values(
    "RMSE",
    ascending=True
)

plt.figure(
    figsize=(11, 7)
)

plt.barh(
    validation_plot_df["Model"],
    validation_plot_df["RMSE"]
)

plt.xlabel(
    "Validation RMSE"
)

plt.ylabel(
    "Model"
)

plt.title(
    "Phase 15 — Recorded Validation RMSE Comparison"
)

plt.tight_layout()

validation_comparison_plot = os.path.join(
    COMPARISON_DIR,
    "phase15_validation_rmse_all_recorded_models.png"
)

plt.savefig(
    validation_comparison_plot,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ============================================================================
# 12. PLOT — FINAL HOLDOUT RMSE
# ============================================================================

plot_holdout = final_holdout.copy()

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    plot_holdout["Model"],
    plot_holdout["RMSE"]
)

plt.ylabel(
    "RMSE"
)

plt.xlabel(
    "Model"
)

plt.title(
    "Final Frozen Holdout — RMSE"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

final_rmse_plot = os.path.join(
    HOLDOUT_DIR,
    "final_holdout_rmse.png"
)

plt.savefig(
    final_rmse_plot,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ============================================================================
# 13. PLOT — FINAL HOLDOUT MAE
# ============================================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    plot_holdout["Model"],
    plot_holdout["MAE"]
)

plt.ylabel(
    "MAE"
)

plt.xlabel(
    "Model"
)

plt.title(
    "Final Frozen Holdout — MAE"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

final_mae_plot = os.path.join(
    HOLDOUT_DIR,
    "final_holdout_mae.png"
)

plt.savefig(
    final_mae_plot,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ============================================================================
# 14. PLOT — FINAL HOLDOUT R2
# ============================================================================

plt.figure(
    figsize=(10, 6)
)

plt.bar(
    plot_holdout["Model"],
    plot_holdout["R2"]
)

plt.ylabel(
    "R²"
)

plt.xlabel(
    "Model"
)

plt.title(
    "Final Frozen Holdout — R²"
)

plt.xticks(
    rotation=20,
    ha="right"
)

plt.tight_layout()

final_r2_plot = os.path.join(
    HOLDOUT_DIR,
    "final_holdout_r2.png"
)

plt.savefig(
    final_r2_plot,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ============================================================================
# 15. PLOT — FINAL HOLDOUT TOLERANCE
# ============================================================================

models_for_tolerance = [
    "Tuned Extra Trees",
    "Tuned Random Forest",
    "Optimized Weighted Ensemble"
]

tol = final_holdout[
    final_holdout["Model"].isin(
        models_for_tolerance
    )
].copy()

x = np.arange(
    len(tol)
)

width = 0.35

plt.figure(
    figsize=(11, 6)
)

plt.bar(
    x - width / 2,
    tol["Accuracy_±10_%"],
    width,
    label="Within ±10 AQI"
)

plt.bar(
    x + width / 2,
    tol["Accuracy_±20_%"],
    width,
    label="Within ±20 AQI"
)

plt.xticks(
    x,
    tol["Model"],
    rotation=20,
    ha="right"
)

plt.ylabel(
    "Percentage (%)"
)

plt.xlabel(
    "Model"
)

plt.title(
    "Final Frozen Holdout — Tolerance Accuracy"
)

plt.legend()

plt.tight_layout()

tolerance_plot = os.path.join(
    HOLDOUT_DIR,
    "final_holdout_tolerance_accuracy.png"
)

plt.savefig(
    tolerance_plot,
    dpi=200,
    bbox_inches="tight"
)

plt.close()

# ============================================================================
# 16. FINAL MODEL OUTLINE DATA
# ============================================================================

final_model_outline = {
    "project": "Delhi AQI t+1 Hour Forecasting",
    "target": "AQI(t+1 hour)",

    "scope": {
        "city": "Delhi",
        "stations": 10,
        "prediction_horizon": "1 hour ahead"
    },

    "input_contract": {
        "feature_count": 73,
        "station_identifier": "StationId",
        "current_features": 12,
        "lag_features": 36,
        "rolling_features": 16,
        "calendar_features": 6,
        "event_features": 2
    },

    "final_models": {
        "extra_trees": {
            "name": "Tuned Extra Trees",
            "file": FINAL_ET_MODEL,
            "ensemble_weight": ensemble_weight_et,
            "hyperparameters": {
                "n_estimators_final_refit": 300,
                "max_depth": 26,
                "min_samples_split": 2,
                "min_samples_leaf": 2,
                "max_features": 0.7
            }
        },

        "random_forest": {
            "name": "Tuned Random Forest",
            "file": FINAL_RF_MODEL,
            "ensemble_weight": ensemble_weight_rf,
            "hyperparameters": {
                "n_estimators_final_refit": 300,
                "max_depth": 26,
                "min_samples_split": 2,
                "min_samples_leaf": 10,
                "max_features": 0.7
            }
        }
    },

    "ensemble": {
        "name": "Optimized Weighted Ensemble",
        "configuration_file": FINAL_ENSEMBLE_CONFIG,
        "formula_description":
            "weighted combination of Extra Trees and Random Forest predictions",
        "extra_trees_weight": ensemble_weight_et,
        "random_forest_weight": ensemble_weight_rf
    },

    "training_protocol": {
        "method":
            "chronological 70/15/15 StationId-Datetime split with embargo",
        "holdout":
            "FINAL_HOLDOUT",
        "holdout_reuse":
            "No new evaluation after final freeze"
    },

    "final_holdout": {
        "rows": 68352,
        "status": "FROZEN / UNTOUCHED",
        "mae": 3.216121,
        "rmse": 7.658390,
        "r2": 0.996634,
        "accuracy_within_5": 85.249883,
        "accuracy_within_10": 94.605864,
        "accuracy_within_20": 98.090765
    },

    "storage": {
        "final_extra_trees_model": FINAL_ET_MODEL,
        "final_random_forest_model": FINAL_RF_MODEL,
        "ensemble_configuration": FINAL_ENSEMBLE_CONFIG,
        "final_freeze": FINAL_FREEZE_JSON,
        "feature_contract": FEATURE_CONTRACT,
        "time_split_assignments": TIME_SPLIT_FILE
    }
}

outline_json_path = os.path.join(
    MASTER_DIR,
    "final_model_outline.json"
)

with open(
    outline_json_path,
    "w"
) as f:

    json.dump(
        final_model_outline,
        f,
        indent=2
    )

# ============================================================================
# 17. COPY FEATURE CONTRACT INTO REPORTING FOLDER
# ============================================================================

if os.path.exists(
    FEATURE_CONTRACT
):

    feature_contract_copy = os.path.join(
        PROTOCOL_DIR,
        "final_feature_contract.csv"
    )

    pd.read_csv(
        FEATURE_CONTRACT
    ).to_csv(
        feature_contract_copy,
        index=False
    )

else:

    feature_contract_copy = None

# ============================================================================
# 18. COPY FINAL FREEZE JSON
# ============================================================================

freeze_copy = os.path.join(
    PROTOCOL_DIR,
    "final_freeze.json"
)

if os.path.exists(
    FINAL_FREEZE_JSON
):

    with open(
        FINAL_FREEZE_JSON,
        "r"
    ) as f:

        freeze_content = json.load(
            f
        )

    with open(
        freeze_copy,
        "w"
    ) as f:

        json.dump(
            freeze_content,
            f,
            indent=2
        )

else:

    freeze_copy = None

# ============================================================================
# 19. FINAL COMPARISON REPORT — MARKDOWN
# ============================================================================

comparison_report = f"""
# Delhi AQI t+1 Hour Forecasting
# Final Model Comparison Report

## 1. Project Objective

The project forecasts the **Air Quality Index one hour ahead**:

**AQI(t) → AQI(t+1)**

The final modeling scope uses ten selected Delhi monitoring stations and
a frozen 73-feature input contract.

---

## 2. Modeling Development Path

The project progressed through several model families and comparison stages.

### Phase 12–14 serious-model comparison

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Linear Regression | 4.301847 | 10.198616 | 0.993241 |
| Decision Tree | 4.505284 | 10.108943 | 0.993360 |
| Random Forest | 3.497741 | 8.457054 | 0.995353 |
| XGBoost | 4.010653 | 8.827806 | 0.994936 |
| Persistence | 3.917885 | 9.418282 | 0.994236 |
| LSTM | 8.095820 | 13.862160 | 0.987523 |
| CNN-BiLSTM | 8.407335 | 13.849509 | 0.987546 |

These are historical model-development results and should be described as
earlier comparison experiments rather than final deployment metrics.

---

## 3. Phase 15 Base-Model Evaluation

Eight classical base models were evaluated using the chronological
validation protocol.

| Base Model | Validation RMSE | Status |
|---|---:|---|
| Linear Regression | Not reliably retained | Recorded in model roster |
| Decision Tree | Not reliably retained | Recorded in model roster |
| Random Forest | 9.504760 | Recorded |
| Extra Trees | 8.538368 | Recorded |
| Gradient Boosting | Not reliably retained | Recorded in model roster |
| HistGradientBoosting | 8.898831 | Recorded |
| AdaBoost | Not reliably retained | Recorded in model roster |
| XGBoost | 8.989854 | Recorded |

Only metrics retained in the project's reporting artifacts are reproduced
numerically. Missing historical values are intentionally not reconstructed.

---

## 4. Tuned Models

### Tuned Extra Trees

Search RMSE: **8.340532**

Refit validation RMSE: **8.286682**

Final holdout:
- MAE: **3.284691**
- RMSE: **7.716191**
- R²: **0.996583**
- Within ±10 AQI: **94.395190%**
- Within ±20 AQI: **98.023467%**

Final configuration:

- n_estimators = 300
- max_depth = 26
- min_samples_split = 2
- min_samples_leaf = 2
- max_features = 0.7

### Tuned Random Forest

Search RMSE: **8.342435**

Refit validation RMSE: **8.331690**

Final holdout:
- MAE: **3.205527**
- RMSE: **7.690637**
- R²: **0.996605**
- Within ±10 AQI: **94.493212%**
- Within ±20 AQI: **98.105396%**

Final configuration:

- n_estimators = 300
- max_depth = 26
- min_samples_split = 2
- min_samples_leaf = 10
- max_features = 0.7

### Tuned XGBoost

Search RMSE: **9.102745**

Refit validation RMSE: **8.901796**

XGBoost was evaluated as a tuned candidate but was not used in the
final ensemble.

---

## 5. Ensemble Development

The two selected tree-based component models were combined.

| Ensemble Configuration | Validation RMSE |
|---|---:|
| 50/50 Extra Trees + Random Forest | 8.249447 |
| Optimized Weighted Ensemble | 8.247305 |

The frozen final ensemble weights are:

- Tuned Extra Trees: **{ensemble_weight_et:.6f}**
- Tuned Random Forest: **{ensemble_weight_rf:.6f}**

---

## 6. Final Frozen Holdout Comparison

The untouched final holdout contains **68,352 rows** for the final model
components and ensemble.

| Model | MAE | RMSE | R² | ±5 AQI | ±10 AQI | ±20 AQI |
|---|---:|---:|---:|---:|---:|---:|
| Tuned Extra Trees | 3.284691 | 7.716191 | 0.996583 | 84.699800% | 94.395190% | 98.023467% |
| Tuned Random Forest | 3.205527 | 7.690637 | 0.996605 | 85.405000% | 94.493212% | 98.105396% |
| Optimized Weighted Ensemble | 3.216121 | 7.658390 | 0.996634 | 85.249883% | 94.605864% | 98.090765% |
| Persistence benchmark | 3.855634 | 8.840011 | 0.995516 | — | 92.261844% | 97.532401% |

The **Optimized Weighted Ensemble** is the project's final frozen model
configuration.

---

## 7. Final Error Distribution

For the final weighted ensemble:

- Mean error: **-0.430216 AQI**
- Median error: **-0.229319 AQI**
- 90th percentile absolute error: **6.588741 AQI**
- 95th percentile absolute error: **10.554171 AQI**
- 99th percentile absolute error: **31.326013 AQI**
- Maximum absolute error: **205.280197 AQI**

---

## 8. Final Model Architecture / Outline

```text
Historical CPCB-derived Delhi station data
                |
                v
       Data cleaning & QC
                |
                v
        Feature engineering
                |
                v
        73-feature contract
                |
                +--------------------+
                |                    |
                v                    v
       Tuned Extra Trees     Tuned Random Forest
                |                    |
                |                    |
                +---------+----------+
                          |
                          v
              Weighted Ensemble
                          |
              ET = {ensemble_weight_et:.6f}
              RF = {ensemble_weight_rf:.6f}
                          |
                          v
                AQI(t+1 hour)

In [ ]:
# R.5 — Comprehensive Model Comparison, Final Model Documentation, and Report
# This cell is self-contained and avoids multiline f-strings that can cause IncompleteInputError.

import os
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1. PATHS
# ============================================================

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"
FINAL_RESULTS = os.path.join(BASE, "outputs", "FINAL_RESULTS")

DIRS = {
    "master": os.path.join(FINAL_RESULTS, "00_MASTER"),
    "trainval": os.path.join(FINAL_RESULTS, "01_TRAINING_AND_VALIDATION"),
    "tuning": os.path.join(FINAL_RESULTS, "01_TRAINING_AND_VALIDATION", "tuning_results"),
    "models": os.path.join(FINAL_RESULTS, "02_MODEL_RESULTS"),
    "comparison": os.path.join(FINAL_RESULTS, "03_MODEL_COMPARISON"),
    "holdout": os.path.join(FINAL_RESULTS, "04_FINAL_HOLDOUT"),
    "shap": os.path.join(FINAL_RESULTS, "05_SHAP_XAI"),
    "errors": os.path.join(FINAL_RESULTS, "06_ERROR_ANALYSIS"),
    "protocol": os.path.join(FINAL_RESULTS, "07_PROTOCOL_AND_FREEZE"),
    "reports": os.path.join(FINAL_RESULTS, "08_REPORTS"),
}

for p in DIRS.values():
    os.makedirs(p, exist_ok=True)

print("R.5 paths ready.")


# ============================================================
# 2. FINAL HOLDOUT RESULTS
# ============================================================

final_holdout = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees",
        "Rows": 68352,
        "MAE": 3.284691,
        "RMSE": 7.716191,
        "R2": 0.996583,
        "Accuracy_±5_%": np.nan,
        "Accuracy_±10_%": 94.395190,
        "Accuracy_±20_%": 98.023467,
        "Status": "Final frozen holdout"
    },
    {
        "Model": "Tuned Random Forest",
        "Rows": 68352,
        "MAE": 3.205527,
        "RMSE": 7.690637,
        "R2": 0.996605,
        "Accuracy_±5_%": np.nan,
        "Accuracy_±10_%": 94.493212,
        "Accuracy_±20_%": 98.105396,
        "Status": "Final frozen holdout"
    },
    {
        "Model": "Optimized Weighted Ensemble",
        "Rows": 68352,
        "MAE": 3.216121,
        "RMSE": 7.658390,
        "R2": 0.996634,
        "Accuracy_±5_%": 85.249883,
        "Accuracy_±10_%": 94.605864,
        "Accuracy_±20_%": 98.090765,
        "Status": "Final frozen holdout / selected final model"
    },
    {
        "Model": "Persistence",
        "Rows": 68285,
        "MAE": 3.855634,
        "RMSE": 8.840011,
        "R2": 0.995516,
        "Accuracy_±5_%": np.nan,
        "Accuracy_±10_%": 92.261844,
        "Accuracy_±20_%": 97.532401,
        "Status": "Baseline"
    },
])

final_holdout.to_csv(
    os.path.join(DIRS["holdout"], "final_frozen_holdout_comparison.csv"),
    index=False
)

print("\nFINAL HOLDOUT RESULTS")
print(final_holdout.to_string(index=False))


# ============================================================
# 3. PHASE 15 BASE-MODEL VALIDATION RESULTS
# ============================================================

phase15_base = pd.DataFrame([
    {
        "Model": "Linear Regression",
        "Validation_RMSE": np.nan,
        "Rank": np.nan,
        "Result_Status": "Not reliably retained"
    },
    {
        "Model": "Decision Tree",
        "Validation_RMSE": np.nan,
        "Rank": np.nan,
        "Result_Status": "Not reliably retained"
    },
    {
        "Model": "Random Forest",
        "Validation_RMSE": 9.504760,
        "Rank": 4,
        "Result_Status": "Recorded"
    },
    {
        "Model": "Extra Trees",
        "Validation_RMSE": 8.538368,
        "Rank": 1,
        "Result_Status": "Recorded"
    },
    {
        "Model": "Gradient Boosting",
        "Validation_RMSE": np.nan,
        "Rank": np.nan,
        "Result_Status": "Not reliably retained"
    },
    {
        "Model": "HistGradientBoosting",
        "Validation_RMSE": 8.898831,
        "Rank": 2,
        "Result_Status": "Recorded"
    },
    {
        "Model": "AdaBoost",
        "Validation_RMSE": np.nan,
        "Rank": np.nan,
        "Result_Status": "Not reliably retained"
    },
    {
        "Model": "XGBoost",
        "Validation_RMSE": 8.989854,
        "Rank": 3,
        "Result_Status": "Recorded"
    },
])

phase15_base.to_csv(
    os.path.join(DIRS["trainval"], "phase15_base_models_comparison.csv"),
    index=False
)

print("\nPHASE 15 BASE MODELS")
print(phase15_base.to_string(index=False))


# ============================================================
# 4. PHASE 15 TUNED MODELS
# ============================================================

phase15_tuned = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees",
        "Search_Best_RMSE": 8.340532,
        "Refit_Validation_RMSE": 8.286682,
        "Final_Holdout_RMSE": 7.716191,
        "Final_Holdout_MAE": 3.284691,
        "Final_Holdout_R2": 0.996583,
        "Selected_For_Final_Ensemble": True
    },
    {
        "Model": "Tuned Random Forest",
        "Search_Best_RMSE": 8.342435,
        "Refit_Validation_RMSE": 8.331690,
        "Final_Holdout_RMSE": 7.690637,
        "Final_Holdout_MAE": 3.205527,
        "Final_Holdout_R2": 0.996605,
        "Selected_For_Final_Ensemble": True
    },
    {
        "Model": "Tuned XGBoost",
        "Search_Best_RMSE": 9.102745,
        "Refit_Validation_RMSE": 8.901796,
        "Final_Holdout_RMSE": np.nan,
        "Final_Holdout_MAE": np.nan,
        "Final_Holdout_R2": np.nan,
        "Selected_For_Final_Ensemble": False
    },
])

phase15_tuned.to_csv(
    os.path.join(DIRS["trainval"], "phase15_tuned_models_comparison.csv"),
    index=False
)

print("\nPHASE 15 TUNED MODELS")
print(phase15_tuned.to_string(index=False))


# ============================================================
# 5. ENSEMBLE DEVELOPMENT
# ============================================================

ensemble_development = pd.DataFrame([
    {
        "Ensemble": "50/50 ET + RF",
        "ET_Weight": 0.500000,
        "RF_Weight": 0.500000,
        "Validation_RMSE": 8.249447,
        "Selected": False
    },
    {
        "Ensemble": "Optimized Weighted ET + RF",
        "ET_Weight": 0.594466,
        "RF_Weight": 0.405534,
        "Validation_RMSE": 8.247305,
        "Selected": True
    },
])

ensemble_development.to_csv(
    os.path.join(DIRS["trainval"], "ensemble_development_comparison.csv"),
    index=False
)

print("\nENSEMBLE DEVELOPMENT")
print(ensemble_development.to_string(index=False))


# ============================================================
# 6. HISTORICAL PHASE 12–14 COMPARISON
# ============================================================

historical = pd.DataFrame([
    {
        "Phase": "Phase 12–14",
        "Model": "Random Forest",
        "MAE": 3.497741,
        "RMSE": 8.457054,
        "R2": 0.995353,
        "Accuracy_±10_%": 93.59
    },
    {
        "Phase": "Phase 12–14",
        "Model": "XGBoost",
        "MAE": 4.010653,
        "RMSE": 8.827806,
        "R2": 0.994936,
        "Accuracy_±10_%": np.nan
    },
    {
        "Phase": "Phase 12–14",
        "Model": "Persistence",
        "MAE": 3.917885,
        "RMSE": 9.418282,
        "R2": 0.994236,
        "Accuracy_±10_%": np.nan
    },
    {
        "Phase": "Phase 12–14",
        "Model": "Decision Tree",
        "MAE": 4.505284,
        "RMSE": 10.108943,
        "R2": 0.993360,
        "Accuracy_±10_%": np.nan
    },
    {
        "Phase": "Phase 12–14",
        "Model": "Linear Regression",
        "MAE": 4.301847,
        "RMSE": 10.198616,
        "R2": 0.993241,
        "Accuracy_±10_%": np.nan
    },
    {
        "Phase": "Phase 12–14",
        "Model": "CNN-BiLSTM",
        "MAE": 8.407335,
        "RMSE": 13.849509,
        "R2": 0.987546,
        "Accuracy_±10_%": np.nan
    },
    {
        "Phase": "Phase 12–14",
        "Model": "LSTM",
        "MAE": 8.095820,
        "RMSE": 13.862160,
        "R2": 0.987523,
        "Accuracy_±10_%": np.nan
    },
])

historical.to_csv(
    os.path.join(DIRS["comparison"], "phase12_14_historical_model_comparison.csv"),
    index=False
)

print("\nPHASE 12–14 HISTORICAL COMPARISON")
print(historical.to_string(index=False))


# ============================================================
# 7. MASTER COMPARISON TABLE
# ============================================================

master_rows = []

# Phase 12–14
for _, r in historical.iterrows():
    master_rows.append({
        "Evaluation_Group": "Phase 12–14 historical",
        "Model": r["Model"],
        "MAE": r["MAE"],
        "RMSE": r["RMSE"],
        "R2": r["R2"],
        "Accuracy_±10_%": r["Accuracy_±10_%"],
        "Notes": "Historical serious-model comparison"
    })

# Phase 15 base
for _, r in phase15_base.iterrows():
    master_rows.append({
        "Evaluation_Group": "Phase 15 validation",
        "Model": r["Model"],
        "MAE": np.nan,
        "RMSE": r["Validation_RMSE"],
        "R2": np.nan,
        "Accuracy_±10_%": np.nan,
        "Notes": r["Result_Status"]
    })

# Phase 15 tuned
for _, r in phase15_tuned.iterrows():
    master_rows.append({
        "Evaluation_Group": "Phase 15 tuned validation",
        "Model": r["Model"],
        "MAE": np.nan,
        "RMSE": r["Refit_Validation_RMSE"],
        "R2": np.nan,
        "Accuracy_±10_%": np.nan,
        "Notes": "Refit validation result"
    })

# Ensemble development
for _, r in ensemble_development.iterrows():
    master_rows.append({
        "Evaluation_Group": "Phase 15 ensemble validation",
        "Model": r["Ensemble"],
        "MAE": np.nan,
        "RMSE": r["Validation_RMSE"],
        "R2": np.nan,
        "Accuracy_±10_%": np.nan,
        "Notes": "Selected" if r["Selected"] else "Development comparison"
    })

# Final holdout
for _, r in final_holdout.iterrows():
    master_rows.append({
        "Evaluation_Group": "Final frozen holdout",
        "Model": r["Model"],
        "MAE": r["MAE"],
        "RMSE": r["RMSE"],
        "R2": r["R2"],
        "Accuracy_±10_%": r["Accuracy_±10_%"],
        "Notes": r["Status"]
    })

comprehensive = pd.DataFrame(master_rows)

comprehensive.to_csv(
    os.path.join(DIRS["comparison"], "comprehensive_model_comparison.csv"),
    index=False
)

print("\nMASTER COMPARISON SAVED:")
print(os.path.join(DIRS["comparison"], "comprehensive_model_comparison.csv"))


# ============================================================
# 8. FINAL MODEL REGISTRY
# ============================================================

final_model_registry = pd.DataFrame([
    {
        "Model": "Tuned Extra Trees",
        "Role": "Final ensemble component",
        "Weight": 0.594466,
        "n_estimators": 300,
        "max_depth": 26,
        "min_samples_split": 2,
        "min_samples_leaf": 2,
        "max_features": 0.7,
        "Target": "AQI_t_plus_1"
    },
    {
        "Model": "Tuned Random Forest",
        "Role": "Final ensemble component",
        "Weight": 0.405534,
        "n_estimators": 300,
        "max_depth": 26,
        "min_samples_split": 2,
        "min_samples_leaf": 10,
        "max_features": 0.7,
        "Target": "AQI_t_plus_1"
    },
    {
        "Model": "Optimized Weighted Ensemble",
        "Role": "Final production model",
        "Weight": 1.0,
        "n_estimators": np.nan,
        "max_depth": np.nan,
        "min_samples_split": np.nan,
        "min_samples_leaf": np.nan,
        "max_features": np.nan,
        "Target": "AQI_t_plus_1"
    },
])

final_model_registry.to_csv(
    os.path.join(DIRS["master"], "final_model_registry.csv"),
    index=False
)


# ============================================================
# 9. ENSEMBLE CONFIGURATION
# ============================================================

ensemble_config = {
    "target": "AQI_t_plus_1",
    "final_model": "Optimized Weighted Ensemble",
    "components": {
        "Tuned Extra Trees": 0.594466,
        "Tuned Random Forest": 0.405534
    },
    "validation_rmse": 8.247305,
    "final_holdout": {
        "rows": 68352,
        "mae": 3.216121,
        "rmse": 7.658390,
        "r2": 0.996634,
        "accuracy_pm_5_percent": 85.249883,
        "accuracy_pm_10_percent": 94.605864,
        "accuracy_pm_20_percent": 98.090765
    },
    "feature_count": 73,
    "station_count": 10,
    "status": "Frozen"
}

with open(
    os.path.join(DIRS["master"], "final_model_outline.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(ensemble_config, f, indent=2)


# ============================================================
# 10. COPY FEATURE CONTRACT + FREEZE FILE
# ============================================================

source_contract = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_15_final_freeze",
    "phase15_15_feature_contract.csv"
)

source_freeze = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_15_final_freeze",
    "phase15_15_final_freeze.json"
)

target_contract = os.path.join(
    DIRS["protocol"],
    "phase15_15_feature_contract.csv"
)

target_freeze = os.path.join(
    DIRS["protocol"],
    "phase15_15_final_freeze.json"
)

if os.path.exists(source_contract):
    shutil.copy2(source_contract, target_contract)
    print("Copied feature contract.")

if os.path.exists(source_freeze):
    shutil.copy2(source_freeze, target_freeze)
    print("Copied final freeze metadata.")


# ============================================================
# 11. FINAL MODEL PLOTS
# ============================================================

plot_df = final_holdout[
    final_holdout["Model"].isin([
        "Tuned Extra Trees",
        "Tuned Random Forest",
        "Optimized Weighted Ensemble",
        "Persistence"
    ])
].copy()

# RMSE
plt.figure(figsize=(10, 6))
plt.bar(plot_df["Model"], plot_df["RMSE"])
plt.ylabel("RMSE")
plt.title("Final Frozen Holdout RMSE Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(DIRS["holdout"], "final_holdout_rmse.png"),
    dpi=200,
    bbox_inches="tight"
)
plt.close()

# MAE
plt.figure(figsize=(10, 6))
plt.bar(plot_df["Model"], plot_df["MAE"])
plt.ylabel("MAE")
plt.title("Final Frozen Holdout MAE Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(DIRS["holdout"], "final_holdout_mae.png"),
    dpi=200,
    bbox_inches="tight"
)
plt.close()

# R2
plt.figure(figsize=(10, 6))
plt.bar(plot_df["Model"], plot_df["R2"])
plt.ylabel("R²")
plt.title("Final Frozen Holdout R² Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(DIRS["holdout"], "final_holdout_r2.png"),
    dpi=200,
    bbox_inches="tight"
)
plt.close()

# ±10 accuracy
tol_df = plot_df.dropna(subset=["Accuracy_±10_%"])

plt.figure(figsize=(10, 6))
plt.bar(tol_df["Model"], tol_df["Accuracy_±10_%"])
plt.ylabel("Accuracy within ±10 AQI (%)")
plt.title("Final Frozen Holdout ±10 AQI Accuracy")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(DIRS["holdout"], "final_holdout_tolerance_accuracy.png"),
    dpi=200,
    bbox_inches="tight"
)
plt.close()


# ============================================================
# 12. VALIDATION RMSE PLOT
# ============================================================

validation_plot = phase15_base.dropna(subset=["Validation_RMSE"]).copy()

plt.figure(figsize=(9, 6))
plt.bar(validation_plot["Model"], validation_plot["Validation_RMSE"])
plt.ylabel("Validation RMSE")
plt.title("Phase 15 Recorded Base-Model Validation RMSE")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(
    os.path.join(DIRS["comparison"], "phase15_validation_rmse_all_recorded_models.png"),
    dpi=200,
    bbox_inches="tight"
)
plt.close()


# ============================================================
# 13. FINAL MODEL SUMMARY TEXT
# ============================================================

final_model_summary = [
    "# FINAL MODEL OUTLINE",
    "",
    "## Forecasting objective",
    "",
    "Predict Delhi AQI at time t+1 hour using information available up to time t.",
    "",
    "## Final target",
    "",
    "`AQI_t_plus_1`",
    "",
    "## Final feature contract",
    "",
    "73 frozen model features.",
    "",
    "The final contract excludes metadata fields such as `StationName`.",
    "",
    "## Final station scope",
    "",
    "10 Delhi monitoring stations.",
    "",
    "## Final model",
    "",
    "Optimized weighted ensemble of Tuned Extra Trees and Tuned Random Forest.",
    "",
    "Extra Trees weight: 0.594466",
    "",
    "Random Forest weight: 0.405534",
    "",
    "## Extra Trees parameters",
    "",
    "- n_estimators = 300",
    "- max_depth = 26",
    "- min_samples_split = 2",
    "- min_samples_leaf = 2",
    "- max_features = 0.7",
    "",
    "## Random Forest parameters",
    "",
    "- n_estimators = 300",
    "- max_depth = 26",
    "- min_samples_split = 2",
    "- min_samples_leaf = 10",
    "- max_features = 0.7",
    "",
    "## Validation",
    "",
    "Optimized ensemble validation RMSE: 8.247305",
    "",
    "## Final frozen holdout",
    "",
    "Rows: 68,352",
    "",
    "MAE: 3.216121",
    "",
    "RMSE: 7.658390",
    "",
    "R²: 0.996634",
    "",
    "Accuracy within ±5 AQI: 85.249883%",
    "",
    "Accuracy within ±10 AQI: 94.605864%",
    "",
    "Accuracy within ±20 AQI: 98.090765%",
    "",
    "## Final status",
    "",
    "Final holdout evaluated once and frozen.",
    "",
    "No model changes are made by this reporting phase."
]

with open(
    os.path.join(DIRS["reports"], "FINAL_MODEL_OUTLINE.md"),
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(final_model_summary))


# ============================================================
# 14. COMPARISON REPORT
# ============================================================

report_lines = []

report_lines.append("# Delhi AQI Forecasting — Final Model Comparison Report")
report_lines.append("")
report_lines.append("## 1. Objective")
report_lines.append("")
report_lines.append(
    "The project forecasts Delhi AQI one hour ahead (`AQI_t_plus_1`) "
    "using information available up to time t."
)
report_lines.append("")
report_lines.append("## 2. Final frozen configuration")
report_lines.append("")
report_lines.append("- Stations: 10 Delhi monitoring stations")
report_lines.append("- Frozen features: 73")
report_lines.append("- Final model: optimized weighted ET + RF ensemble")
report_lines.append("- Extra Trees weight: 0.594466")
report_lines.append("- Random Forest weight: 0.405534")
report_lines.append("- Final holdout rows: 68,352")
report_lines.append("")
report_lines.append("## 3. Final frozen holdout results")
report_lines.append("")

for _, r in final_holdout.iterrows():
    report_lines.append(
        "| {} | {} | {:.6f} | {:.6f} | {:.6f} | {:.3f}% | {:.3f}% |".format(
            r["Model"],
            int(r["Rows"]),
            r["MAE"],
            r["RMSE"],
            r["R2"],
            r["Accuracy_±10_%"] if pd.notna(r["Accuracy_±10_%"]) else np.nan,
            r["Accuracy_±20_%"] if pd.notna(r["Accuracy_±20_%"]) else np.nan
        )
    )

report_lines.insert(
    report_lines.index("## 3. Final frozen holdout results") + 2,
    "| Model | Rows | MAE | RMSE | R² | ±10 AQI | ±20 AQI |"
)
report_lines.insert(
    report_lines.index("| Model | Rows | MAE | RMSE | R² | ±10 AQI | ±20 AQI |") + 1,
    "|---|---:|---:|---:|---:|---:|---:|"
)

report_lines.append("")
report_lines.append("## 4. Phase 15 validation history")
report_lines.append("")
report_lines.append(
    "The Phase 15 validation process used chronological splitting with an embargo. "
    "Only reliably retained validation RMSE values are included below."
)
report_lines.append("")

for _, r in phase15_base.dropna(subset=["Validation_RMSE"]).sort_values("Rank").iterrows():
    report_lines.append(
        "- {}: validation RMSE = {:.6f} (recorded rank {})".format(
            r["Model"],
            r["Validation_RMSE"],
            int(r["Rank"])
        )
    )

report_lines.append("")
report_lines.append("## 5. Ensemble development")
report_lines.append("")
report_lines.append(
    "- 50/50 ET + RF validation RMSE: 8.249447"
)
report_lines.append(
    "- Optimized weighted ET + RF validation RMSE: 8.247305"
)
report_lines.append("")
report_lines.append("## 6. Final model conclusion")
report_lines.append("")
report_lines.append(
    "The frozen final configuration consists of Tuned Extra Trees and "
    "Tuned Random Forest combined with weights determined during Phase 15."
)
report_lines.append("")
report_lines.append(
    "The final frozen holdout contains 68,352 rows and was evaluated without "
    "changing the frozen model configuration."
)
report_lines.append("")
report_lines.append(
    "The final ensemble holdout results are MAE 3.216121, RMSE 7.658390, "
    "R² 0.996634, ±10 AQI accuracy 94.605864%, and ±20 AQI accuracy 98.090765%."
)

with open(
    os.path.join(DIRS["reports"], "FINAL_COMPARISON_REPORT.md"),
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(report_lines))


# ============================================================
# 15. RESULTS INDEX
# ============================================================

index_lines = [
    "# FINAL RESULTS INDEX",
    "",
    "## Master",
    "",
    "- `00_MASTER/final_model_registry.csv`",
    "- `00_MASTER/final_model_outline.json`",
    "",
    "## Training and validation",
    "",
    "- `01_TRAINING_AND_VALIDATION/phase15_base_models_comparison.csv`",
    "- `01_TRAINING_AND_VALIDATION/phase15_tuned_models_comparison.csv`",
    "- `01_TRAINING_AND_VALIDATION/ensemble_development_comparison.csv`",
    "",
    "## Model comparison",
    "",
    "- `03_MODEL_COMPARISON/phase12_14_historical_model_comparison.csv`",
    "- `03_MODEL_COMPARISON/comprehensive_model_comparison.csv`",
    "- `03_MODEL_COMPARISON/phase15_validation_rmse_all_recorded_models.png`",
    "",
    "## Final holdout",
    "",
    "- `04_FINAL_HOLDOUT/final_frozen_holdout_comparison.csv`",
    "- `04_FINAL_HOLDOUT/final_holdout_rmse.png`",
    "- `04_FINAL_HOLDOUT/final_holdout_mae.png`",
    "- `04_FINAL_HOLDOUT/final_holdout_r2.png`",
    "- `04_FINAL_HOLDOUT/final_holdout_tolerance_accuracy.png`",
    "",
    "## Protocol and freeze",
    "",
    "- `07_PROTOCOL_AND_FREEZE/phase15_15_feature_contract.csv`",
    "- `07_PROTOCOL_AND_FREEZE/phase15_15_final_freeze.json`",
    "",
    "## Reports",
    "",
    "- `08_REPORTS/FINAL_COMPARISON_REPORT.md`",
    "- `08_REPORTS/FINAL_MODEL_OUTLINE.md`",
]

with open(
    os.path.join(DIRS["reports"], "FINAL_RESULTS_INDEX.md"),
    "w",
    encoding="utf-8"
) as f:
    f.write("\n".join(index_lines))


# ============================================================
# 16. OUTPUT VERIFICATION
# ============================================================

expected_files = [
    os.path.join(DIRS["master"], "final_model_registry.csv"),
    os.path.join(DIRS["master"], "final_model_outline.json"),
    os.path.join(DIRS["trainval"], "phase15_base_models_comparison.csv"),
    os.path.join(DIRS["trainval"], "phase15_tuned_models_comparison.csv"),
    os.path.join(DIRS["trainval"], "ensemble_development_comparison.csv"),
    os.path.join(DIRS["comparison"], "phase12_14_historical_model_comparison.csv"),
    os.path.join(DIRS["comparison"], "comprehensive_model_comparison.csv"),
    os.path.join(DIRS["comparison"], "phase15_validation_rmse_all_recorded_models.png"),
    os.path.join(DIRS["holdout"], "final_frozen_holdout_comparison.csv"),
    os.path.join(DIRS["holdout"], "final_holdout_rmse.png"),
    os.path.join(DIRS["holdout"], "final_holdout_mae.png"),
    os.path.join(DIRS["holdout"], "final_holdout_r2.png"),
    os.path.join(DIRS["holdout"], "final_holdout_tolerance_accuracy.png"),
    os.path.join(DIRS["reports"], "FINAL_COMPARISON_REPORT.md"),
    os.path.join(DIRS["reports"], "FINAL_MODEL_OUTLINE.md"),
    os.path.join(DIRS["reports"], "FINAL_RESULTS_INDEX.md"),
]

verification = []

for p in expected_files:
    verification.append({
        "file": p,
        "exists": os.path.exists(p),
        "size_bytes": os.path.getsize(p) if os.path.exists(p) else np.nan
    })

verification_df = pd.DataFrame(verification)

verification_df.to_csv(
    os.path.join(DIRS["master"], "r5_output_verification.csv"),
    index=False
)

metadata = {
    "stage": "R.5",
    "status": "completed",
    "final_model": "Optimized Weighted Ensemble",
    "components": {
        "Tuned Extra Trees": 0.594466,
        "Tuned Random Forest": 0.405534
    },
    "target": "AQI_t_plus_1",
    "feature_count": 73,
    "station_count": 10,
    "final_holdout_rows": 68352,
    "final_holdout_rmse": 7.658390,
    "final_holdout_mae": 3.216121,
    "final_holdout_r2": 0.996634,
    "reports_created": True,
    "shap_recomputed": False
}

with open(
    os.path.join(DIRS["master"], "r5_comprehensive_results_metadata.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)


# ============================================================
# 17. FINAL PRINT
# ============================================================

print("\n" + "=" * 70)
print("R.5 COMPLETE")
print("=" * 70)

print("\nFinal model:")
print("  Optimized Weighted Ensemble")
print("  Extra Trees weight : 0.594466")
print("  Random Forest weight: 0.405534")

print("\nFinal frozen holdout:")
print("  Rows       :", 68352)
print("  MAE        : 3.216121")
print("  RMSE       : 7.658390")
print("  R²         : 0.996634")
print("  ±5 AQI     : 85.249883%")
print("  ±10 AQI    : 94.605864%")
print("  ±20 AQI    : 98.090765%")

print("\nReports:")
print(" ", os.path.join(DIRS["reports"], "FINAL_COMPARISON_REPORT.md"))
print(" ", os.path.join(DIRS["reports"], "FINAL_MODEL_OUTLINE.md"))
print(" ", os.path.join(DIRS["reports"], "FINAL_RESULTS_INDEX.md"))

print("\nVerification:")
print("  Files verified:", int(verification_df["exists"].sum()), "/", len(verification_df))

print("\nR.5 finished successfully.")

In [ ]:
# ============================================================
# R.6 — ALL MODEL DIAGNOSTICS
# ============================================================
# IMPORTANT:
# - FINAL_HOLDOUT remains frozen and untouched.
# - Base/tuned/ensemble diagnostic runs below use TRAIN -> VALIDATION.
# - These are reporting/visualization runs, not replacements for
#   the official Phase 15 validation or final holdout results.
# - No SHAP is used.
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

FEATURE_FILE = os.path.join(
    BASE,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

SPLIT_FILE = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_1_time_split_assignments.parquet"
)

FINAL_HOLDOUT_PRED_FILE = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_12_final_holdout",
    "phase15_12_final_holdout_predictions.parquet"
)

R6_DIR = os.path.join(
    BASE,
    "outputs",
    "FINAL_RESULTS",
    "02_MODEL_RESULTS",
    "R6_ALL_MODEL_DIAGNOSTICS"
)

BASE_DIR = os.path.join(
    R6_DIR,
    "BASE_MODELS"
)

TUNED_DIR = os.path.join(
    R6_DIR,
    "TUNED_MODELS"
)

ENSEMBLE_DIR = os.path.join(
    R6_DIR,
    "ENSEMBLES"
)

BENCHMARK_DIR = os.path.join(
    R6_DIR,
    "BENCHMARKS"
)

MASTER_DIR = os.path.join(
    R6_DIR,
    "MASTER"

)

for p in [
    R6_DIR,
    BASE_DIR,
    TUNED_DIR,
    ENSEMBLE_DIR,
    BENCHMARK_DIR,
    MASTER_DIR
]:
    os.makedirs(p, exist_ok=True)


# ============================================================
# 2. LOAD DATA
# ============================================================

print("=" * 90)
print("R.6 — LOADING FEATURE DATA")
print("=" * 90)

df = pd.read_parquet(FEATURE_FILE)
split_df = pd.read_parquet(SPLIT_FILE)

print("Feature rows:", f"{len(df):,}")
print("Split rows:", f"{len(split_df):,}")

TARGET = "AQI_t_plus_1"
DATETIME = "Datetime"
STATION = "StationId"

if TARGET not in df.columns:
    raise KeyError("Target column AQI_t_plus_1 not found.")

if DATETIME not in df.columns:
    raise KeyError("Datetime column not found.")

if STATION not in df.columns:
    raise KeyError("StationId column not found.")


# ============================================================
# 3. MERGE FROZEN SPLIT ASSIGNMENTS
# ============================================================

split_keys = [STATION, DATETIME]

split_use = split_df[
    split_keys + ["Split"]
].copy()

split_use[DATETIME] = pd.to_datetime(
    split_use[DATETIME]
)

df[DATETIME] = pd.to_datetime(
    df[DATETIME]
)

model_df = df.merge(
    split_use,
    on=split_keys,
    how="left",
    validate="one_to_one"
)

if model_df["Split"].isna().any():
    raise ValueError(
        "Some feature rows do not have a frozen split assignment."
    )

print("\nSplit counts:")
print(
    model_df["Split"].value_counts()
)


# ============================================================
# 4. DEFINE THE FROZEN 73-FEATURE CONTRACT
# ============================================================

freeze_contract = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_15_final_freeze",
    "phase15_15_feature_contract.csv"
)

if os.path.exists(freeze_contract):

    contract_df = pd.read_csv(
        freeze_contract
    )

    possible_feature_cols = [
        c for c in contract_df.columns
        if c.lower() in [
            "feature",
            "feature_name",
            "original_feature",
            "model_feature"
        ]
    ]

    if possible_feature_cols:
        contract_col = possible_feature_cols[0]
        feature_cols = contract_df[
            contract_col
        ].dropna().astype(str).tolist()
    else:
        feature_cols = [
            c for c in contract_df.columns
            if c in model_df.columns
        ]

else:

    feature_cols = [
        c for c in model_df.columns
        if c not in [
            TARGET,
            "Split",
            DATETIME,
            "StationName"
        ]
    ]

feature_cols = [
    c for c in feature_cols
    if c in model_df.columns
]

if len(feature_cols) != 73:
    raise ValueError(
        "Expected exactly 73 model features, found "
        + str(len(feature_cols))
    )

print("\nFrozen feature count:", len(feature_cols))


# ============================================================
# 5. TRAIN / VALIDATION DATA
# ============================================================

train_df = model_df[
    model_df["Split"].eq("TRAIN")
].copy()

valid_df = model_df[
    model_df["Split"].eq("VALIDATION")
].copy()

train_df = train_df[
    train_df[TARGET].notna()
].copy()

valid_df = valid_df[
    valid_df[TARGET].notna()
].copy()

train_df = train_df.sort_values(
    [DATETIME, STATION]
).reset_index(drop=True)

valid_df = valid_df.sort_values(
    [DATETIME, STATION]
).reset_index(drop=True)

print("\nTRAIN rows:", f"{len(train_df):,}")
print("VALIDATION rows:", f"{len(valid_df):,}")

if len(train_df) == 0 or len(valid_df) == 0:
    raise ValueError("TRAIN or VALIDATION set is empty.")


X_train_raw = train_df[feature_cols].copy()
y_train = train_df[TARGET].astype(float).to_numpy()

X_valid_raw = valid_df[feature_cols].copy()
y_valid = valid_df[TARGET].astype(float).to_numpy()


# ============================================================
# 6. PREPROCESSOR
# ============================================================

categorical_features = []

if STATION in feature_cols:
    categorical_features.append(STATION)

numeric_features = [
    c for c in feature_cols
    if c not in categorical_features
]

numeric_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )
])

categorical_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipe,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipe,
            categorical_features
        )
    ],
    remainder="drop"
)

print("\nFitting common preprocessing on TRAIN only...")

X_train = preprocessor.fit_transform(
    X_train_raw
)

X_valid = preprocessor.transform(
    X_valid_raw
)

try:
    transformed_feature_names = list(
        preprocessor.get_feature_names_out()
    )
except Exception:
    transformed_feature_names = [
        f"Feature_{i}"
        for i in range(X_train.shape[1])
    ]

print(
    "Transformed feature count:",
    X_train.shape[1]
)


# ============================================================
# 7. MODEL DEFINITIONS
# ============================================================

base_models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42
    ),

    "AdaBoost": AdaBoostRegressor(
        n_estimators=100,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        min_child_weight=1,
        subsample=1.0,
        colsample_bytree=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

tuned_models = {
    "Tuned Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=26,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    ),

    "Tuned Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=26,
        min_samples_split=2,
        min_samples_leaf=10,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    ),

    "Tuned XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}


# ============================================================
# 8. DIAGNOSTIC FUNCTIONS
# ============================================================

AQI_BINS = [
    -np.inf,
    100,
    200,
    300,
    400,
    np.inf
]

AQI_LABELS = [
    "<100",
    "100–199",
    "200–299",
    "300–399",
    "≥400"
]


def safe_mape(y_true, y_pred):
    denom = np.where(
        np.abs(y_true) < 1e-9,
        np.nan,
        np.abs(y_true)
    )
    return float(
        np.nanmean(
            np.abs(
                (y_true - y_pred) / denom
            )
        )
        * 100
    )


def compute_metrics(
    y_true,
    y_pred
):

    residual = y_true - y_pred
    abs_error = np.abs(residual)

    return {
        "Rows": int(len(y_true)),
        "MAE": float(
            mean_absolute_error(
                y_true,
                y_pred
            )
        ),
        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            )
        ),
        "R2": float(
            r2_score(
                y_true,
                y_pred
            )
        ),
        "MAPE_percent": safe_mape(
            y_true,
            y_pred
        ),
        "Bias_Mean_Error": float(
            np.mean(residual)
        ),
        "Median_Error": float(
            np.median(residual)
        ),
        "Std_Error": float(
            np.std(residual)
        ),
        "P90_Absolute_Error": float(
            np.percentile(
                abs_error,
                90
            )
        ),
        "P95_Absolute_Error": float(
            np.percentile(
                abs_error,
                95
            )
        ),
        "P99_Absolute_Error": float(
            np.percentile(
                abs_error,
                99
            )
        ),
        "Max_Absolute_Error": float(
            np.max(abs_error)
        ),
        "Accuracy_pm5_percent": float(
            np.mean(abs_error <= 5)
            * 100
        ),
        "Accuracy_pm10_percent": float(
            np.mean(abs_error <= 10)
            * 100
        ),
        "Accuracy_pm20_percent": float(
            np.mean(abs_error <= 20)
            * 100
        )
    }


def add_aqi_range(
    y_true
):

    return pd.cut(
        y_true,
        bins=AQI_BINS,
        labels=AQI_LABELS,
        right=False,
        include_lowest=True
    )


def build_category(
    values
):

    return pd.cut(
        values,
        bins=[
            -np.inf,
            50,
            100,
            200,
            300,
            400,
            np.inf
        ],
        labels=[
            "Good",
            "Satisfactory",
            "Moderate",
            "Poor",
            "Very Poor",
            "Severe"
        ],
        right=True
    )


def save_model_diagnostics(
    model_name,
    model_type,
    y_true,
    y_pred,
    meta_df,
    output_root
):

    safe_name = (
        model_name
        .replace(" ", "_")
        .replace("/", "_")
        .replace("+", "plus")
    )

    model_dir = os.path.join(
        output_root,
        safe_name
    )

    os.makedirs(
        model_dir,
        exist_ok=True
    )

    temp = meta_df[
        [DATETIME, STATION]
    ].copy()

    temp["Actual_AQI_t_plus_1"] = y_true
    temp["Predicted_AQI_t_plus_1"] = y_pred
    temp["Residual"] = y_true - y_pred
    temp["Absolute_Error"] = np.abs(
        temp["Residual"]
    )
    temp["Squared_Error"] = (
        temp["Residual"] ** 2
    )
    temp["AQI_Range"] = add_aqi_range(
        y_true
    )

    temp = temp.sort_values(
        [DATETIME, STATION]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics = compute_metrics(
        y_true,
        y_pred
    )

    metrics["Model"] = model_name
    metrics["Model_Type"] = model_type
    metrics["Evaluation_Split"] = "VALIDATION"
    metrics["Diagnostic_Run"] = True

    metrics_df = pd.DataFrame([
        metrics
    ])

    metrics_df.to_csv(
        os.path.join(
            model_dir,
            "metrics.csv"
        ),
        index=False
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    temp.to_parquet(
        os.path.join(
            model_dir,
            "predictions.parquet"
        ),
        index=False
    )

    # --------------------------------------------------------
    # 1. Actual vs Predicted
    # --------------------------------------------------------

    plt.figure(
        figsize=(9, 8)
    )

    plt.scatter(
        y_true,
        y_pred,
        s=8,
        alpha=0.25
    )

    low = min(
        np.nanmin(y_true),
        np.nanmin(y_pred)
    )

    high = max(
        np.nanmax(y_true),
        np.nanmax(y_pred)
    )

    plt.plot(
        [low, high],
        [low, high],
        linewidth=2
    )

    plt.xlabel("Actual AQI")
    plt.ylabel("Predicted AQI")
    plt.title(
        model_name
        + " — Actual vs Predicted"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "01_actual_vs_predicted.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 2. Residual distribution
    # --------------------------------------------------------

    plt.figure(
        figsize=(10, 6)
    )

    plt.hist(
        temp["Residual"],
        bins=80,
        alpha=0.8
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=2
    )

    plt.xlabel("Residual = Actual - Predicted")
    plt.ylabel("Frequency")
    plt.title(
        model_name
        + " — Residual Distribution"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "02_residual_distribution.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 3. Residual vs predicted
    # --------------------------------------------------------

    plt.figure(
        figsize=(10, 6)
    )

    plt.scatter(
        temp["Predicted_AQI_t_plus_1"],
        temp["Residual"],
        s=8,
        alpha=0.25
    )

    plt.axhline(
        0,
        linestyle="--",
        linewidth=2
    )

    plt.xlabel("Predicted AQI")
    plt.ylabel("Residual")
    plt.title(
        model_name
        + " — Residual vs Predicted"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "03_residual_vs_predicted.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 4. Error over time
    # --------------------------------------------------------

    plt.figure(
        figsize=(14, 6)
    )

    plt.plot(
        temp[DATETIME],
        temp["Absolute_Error"],
        linewidth=0.8
    )

    plt.xlabel("Datetime")
    plt.ylabel("Absolute Error")
    plt.title(
        model_name
        + " — Absolute Prediction Error Over Time"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "04_error_over_time.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 5. Actual vs predicted time series
    # --------------------------------------------------------

    plot_n = min(
        1500,
        len(temp)
    )

    ts = temp.head(
        plot_n
    )

    plt.figure(
        figsize=(14, 6)
    )

    plt.plot(
        ts[DATETIME],
        ts["Actual_AQI_t_plus_1"],
        label="Actual",
        linewidth=1.2
    )

    plt.plot(
        ts[DATETIME],
        ts["Predicted_AQI_t_plus_1"],
        label="Predicted",
        linewidth=1.0,
        alpha=0.8
    )

    plt.xlabel("Datetime")
    plt.ylabel("AQI")
    plt.title(
        model_name
        + " — Actual vs Predicted Time Series"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "05_actual_vs_predicted_time_series.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 6. AQI range comparison
    # --------------------------------------------------------

    range_rows = []

    for label in AQI_LABELS:

        subset = temp[
            temp["AQI_Range"].astype(str).eq(label)
        ].copy()

        if len(subset) == 0:
            continue

        range_rows.append({
            "AQI_Range": label,
            "Records": len(subset),
            "Actual_Mean": subset[
                "Actual_AQI_t_plus_1"
            ].mean(),
            "Predicted_Mean": subset[
                "Predicted_AQI_t_plus_1"
            ].mean(),
            "MAE": subset[
                "Absolute_Error"
            ].mean(),
            "RMSE": np.sqrt(
                np.mean(
                    subset["Squared_Error"]
                )
            ),
            "Accuracy_pm10_percent": np.mean(
                subset["Absolute_Error"] <= 10
            ) * 100,
            "Mean_Error": subset[
                "Residual"
            ].mean()
        })

    range_df = pd.DataFrame(
        range_rows
    )

    range_df.to_csv(
        os.path.join(
            model_dir,
            "06_aqi_range_metrics.csv"
        ),
        index=False
    )

    if len(range_df) > 0:

        x = np.arange(
            len(range_df)
        )

        width = 0.35

        plt.figure(
            figsize=(11, 6)
        )

        plt.bar(
            x - width / 2,
            range_df["Actual_Mean"],
            width,
            label="Actual Mean"
        )

        plt.bar(
            x + width / 2,
            range_df["Predicted_Mean"],
            width,
            label="Predicted Mean"
        )

        plt.xticks(
            x,
            range_df["AQI_Range"]
        )

        plt.ylabel("AQI")
        plt.xlabel("AQI Range")
        plt.title(
            model_name
            + " — Actual vs Predicted by AQI Range"
        )

        plt.legend()

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                model_dir,
                "07_actual_vs_predicted_by_aqi_range.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )

        plt.close()

        # AQI-range MAE

        plt.figure(
            figsize=(11, 6)
        )

        plt.bar(
            range_df["AQI_Range"],
            range_df["MAE"]
        )

        plt.ylabel("MAE")
        plt.xlabel("AQI Range")
        plt.title(
            model_name
            + " — MAE by AQI Range"
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                model_dir,
                "08_mae_by_aqi_range.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )

        plt.close()

    # --------------------------------------------------------
    # 7. Category confusion matrix
    # --------------------------------------------------------

    actual_cat = build_category(
        y_true
    ).astype(str)

    pred_cat = build_category(
        y_pred
    ).astype(str)

    category_labels = [
        "Good",
        "Satisfactory",
        "Moderate",
        "Poor",
        "Very Poor",
        "Severe"
    ]

    confusion = pd.crosstab(
        actual_cat,
        pred_cat,
        dropna=False
    )

    confusion = confusion.reindex(
        index=category_labels,
        columns=category_labels,
        fill_value=0
    )

    confusion.to_csv(
        os.path.join(
            model_dir,
            "09_aqi_category_confusion_matrix.csv"
        )
    )

    plt.figure(
        figsize=(9, 8)
    )

    plt.imshow(
        confusion.values,
        aspect="auto"
    )

    plt.xticks(
        range(len(category_labels)),
        category_labels,
        rotation=30,
        ha="right"
    )

    plt.yticks(
        range(len(category_labels)),
        category_labels
    )

    plt.xlabel("Predicted AQI Category")
    plt.ylabel("Actual AQI Category")
    plt.title(
        model_name
        + " — AQI Category Confusion Matrix"
    )

    for i in range(
        confusion.shape[0]
    ):
        for j in range(
            confusion.shape[1]
        ):
            plt.text(
                j,
                i,
                str(
                    confusion.iloc[i, j]
                ),
                ha="center",
                va="center"
            )

    plt.colorbar(
        label="Number of observations"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "10_aqi_category_confusion_matrix.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 8. Error distribution / quantile plot
    # --------------------------------------------------------

    quantiles = [
        50,
        75,
        90,
        95,
        99,
        99.5
    ]

    q_values = np.percentile(
        temp["Absolute_Error"],
        quantiles
    )

    q_df = pd.DataFrame({
        "Percentile": quantiles,
        "Absolute_Error": q_values
    })

    q_df.to_csv(
        os.path.join(
            model_dir,
            "11_error_quantiles.csv"
        ),
        index=False
    )

    plt.figure(
        figsize=(10, 6)
    )

    plt.plot(
        quantiles,
        q_values,
        marker="o"
    )

    plt.xlabel("Percentile")
    plt.ylabel("Absolute Error")
    plt.title(
        model_name
        + " — Absolute Error Quantiles"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "12_error_quantiles.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 9. Error boxplot by AQI range
    # --------------------------------------------------------

    grouped_errors = []

    for label in AQI_LABELS:

        vals = temp.loc[
            temp["AQI_Range"].astype(str).eq(label),
            "Absolute_Error"
        ].values

        if len(vals) > 0:
            grouped_errors.append(vals)

    if len(grouped_errors) > 0:

        plt.figure(
            figsize=(11, 6)
        )

        plt.boxplot(
            grouped_errors,
            labels=[
                AQI_LABELS[i]
                for i in range(
                    len(grouped_errors)
                )
            ],
            showfliers=False
        )

        plt.xlabel("AQI Range")
        plt.ylabel("Absolute Error")
        plt.title(
            model_name
            + " — Error Distribution by AQI Range"
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                model_dir,
                "13_error_boxplot_by_aqi_range.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )

        plt.close()

    # --------------------------------------------------------
    # 10. Worst prediction cases
    # --------------------------------------------------------

    worst = temp.sort_values(
        "Absolute_Error",
        ascending=False
    ).head(
        50
    ).copy()

    worst.to_csv(
        os.path.join(
            model_dir,
            "14_worst_prediction_cases.csv"
        ),
        index=False
    )

    # --------------------------------------------------------
    # 11. Best prediction cases
    # --------------------------------------------------------

    best = temp.sort_values(
        "Absolute_Error",
        ascending=True
    ).head(
        50
    ).copy()

    best.to_csv(
        os.path.join(
            model_dir,
            "15_best_prediction_cases.csv"
        ),
        index=False
    )

    return metrics


def save_feature_importance(
    model,
    model_name,
    output_root,
    transformed_names
):

    safe_name = (
        model_name
        .replace(" ", "_")
        .replace("/", "_")
        .replace("+", "plus")
    )

    model_dir = os.path.join(
        output_root,
        safe_name
    )

    os.makedirs(
        model_dir,
        exist_ok=True
    )

    importance = None
    importance_type = None

    if hasattr(
        model,
        "feature_importances_"
    ):

        importance = np.asarray(
            model.feature_importances_,
            dtype=float
        )

        importance_type = "native_feature_importance"

    elif hasattr(
        model,
        "coef_"
    ):

        importance = np.abs(
            np.asarray(
                model.coef_,
                dtype=float
            )
        )

        importance_type = "absolute_linear_coefficient"

    if importance is None:
        return None

    if len(importance) != len(transformed_names):
        return None

    importance_df = pd.DataFrame({
        "Feature": transformed_names,
        "Importance": importance
    })

    total = importance_df["Importance"].sum()

    if total > 0:
        importance_df[
            "Relative_Importance_percent"
        ] = (
            importance_df["Importance"]
            / total
            * 100
        )
    else:
        importance_df[
            "Relative_Importance_percent"
        ] = np.nan

    importance_df = importance_df.sort_values(
        "Importance",
        ascending=False
    ).reset_index(drop=True)

    importance_df.to_csv(
        os.path.join(
            model_dir,
            "16_feature_importance.csv"
        ),
        index=False
    )

    top_n = min(
        20,
        len(importance_df)
    )

    top = importance_df.head(
        top_n
    ).iloc[::-1]

    plt.figure(
        figsize=(11, 8)
    )

    plt.barh(
        top["Feature"],
        top["Importance"]
    )

    plt.xlabel(
        "Importance"
    )

    plt.ylabel(
        "Feature"
    )

    plt.title(
        model_name
        + " — Top Feature Importance"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "17_top_feature_importance.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    return {
        "Model": model_name,
        "Importance_Type": importance_type,
        "Top_Feature": importance_df.iloc[0]["Feature"],
        "Top_Importance": float(
            importance_df.iloc[0]["Importance"]
        )
    }


# ============================================================
# 9. RUN BASE MODELS
# ============================================================

all_results = []
importance_results = []

print("\n" + "=" * 90)
print("R.6 — BASE MODELS")
print("=" * 90)

base_predictions = {}

for model_name, model in base_models.items():

    print("\nTraining:", model_name)

    model.fit(
        X_train,
        y_train
    )

    pred = np.asarray(
        model.predict(
            X_valid
        ),
        dtype=float
    )

    base_predictions[model_name] = pred

    metrics = save_model_diagnostics(
        model_name=model_name,
        model_type="BASE",
        y_true=y_valid,
        y_pred=pred,
        meta_df=valid_df,
        output_root=BASE_DIR
    )

    importance = save_feature_importance(
        model=model,
        model_name=model_name,
        output_root=BASE_DIR,
        transformed_names=transformed_feature_names
    )

    all_results.append(
        {
            **metrics,
            "Model": model_name,
            "Model_Type": "BASE",
            "Evaluation_Split": "VALIDATION"
        }
    )

    if importance is not None:
        importance_results.append(
            importance
        )

    print(
        "RMSE:",
        f"{metrics['RMSE']:.6f}",
        "| MAE:",
        f"{metrics['MAE']:.6f}",
        "| R2:",
        f"{metrics['R2']:.6f}",
        "| ±10:",
        f"{metrics['Accuracy_pm10_percent']:.4f}%"
    )


# ============================================================
# 10. RUN TUNED MODELS
# ============================================================

print("\n" + "=" * 90)
print("R.6 — TUNED MODELS")
print("=" * 90)

tuned_predictions = {}

for model_name, model in tuned_models.items():

    print("\nTraining:", model_name)

    model.fit(
        X_train,
        y_train
    )

    pred = np.asarray(
        model.predict(
            X_valid
        ),
        dtype=float
    )

    tuned_predictions[model_name] = pred

    metrics = save_model_diagnostics(
        model_name=model_name,
        model_type="TUNED",
        y_true=y_valid,
        y_pred=pred,
        meta_df=valid_df,
        output_root=TUNED_DIR
    )

    importance = save_feature_importance(
        model=model,
        model_name=model_name,
        output_root=TUNED_DIR,
        transformed_names=transformed_feature_names
    )

    all_results.append(
        {
            **metrics,
            "Model": model_name,
            "Model_Type": "TUNED",
            "Evaluation_Split": "VALIDATION"
        }
    )

    if importance is not None:
        importance_results.append(
            importance
        )

    print(
        "RMSE:",
        f"{metrics['RMSE']:.6f}",
        "| MAE:",
        f"{metrics['MAE']:.6f}",
        "| R2:",
        f"{metrics['R2']:.6f}",
        "| ±10:",
        f"{metrics['Accuracy_pm10_percent']:.4f}%"
    )


# ============================================================
# 11. ENSEMBLES
# ============================================================

print("\n" + "=" * 90)
print("R.6 — ENSEMBLES")
print("=" * 90)

et_valid = tuned_predictions[
    "Tuned Extra Trees"
]

rf_valid = tuned_predictions[
    "Tuned Random Forest"
]

ensemble_5050 = (
    0.5 * et_valid
    + 0.5 * rf_valid
)

ensemble_optimized = (
    0.594466 * et_valid
    + 0.405534 * rf_valid
)

ensemble_predictions = {
    "50/50 ET + RF": ensemble_5050,
    "Optimized Weighted ET + RF": ensemble_optimized
}

for ensemble_name, pred in ensemble_predictions.items():

    metrics = save_model_diagnostics(
        model_name=ensemble_name,
        model_type="ENSEMBLE",
        y_true=y_valid,
        y_pred=pred,
        meta_df=valid_df,
        output_root=ENSEMBLE_DIR
    )

    all_results.append(
        {
            **metrics,
            "Model": ensemble_name,
            "Model_Type": "ENSEMBLE",
            "Evaluation_Split": "VALIDATION"
        }
    )

    print(
        ensemble_name,
        "| RMSE:",
        f"{metrics['RMSE']:.6f}",
        "| MAE:",
        f"{metrics['MAE']:.6f}",
        "| R2:",
        f"{metrics['R2']:.6f}",
        "| ±10:",
        f"{metrics['Accuracy_pm10_percent']:.4f}%"
    )


# ============================================================
# 12. PERSISTENCE BENCHMARK
# ============================================================

print("\n" + "=" * 90)
print("R.6 — PERSISTENCE BENCHMARK")
print("=" * 90)

if "AQI_lag_1h" not in valid_df.columns:
    raise KeyError(
        "AQI_lag_1h is required for persistence benchmark."
    )

persistence_mask = (
    valid_df["AQI_lag_1h"].notna()
    & valid_df[TARGET].notna()
)

persistence_actual = valid_df.loc[
    persistence_mask,
    TARGET
].astype(float).to_numpy()

persistence_pred = valid_df.loc[
    persistence_mask,
    "AQI_lag_1h"
].astype(float).to_numpy()

persistence_meta = valid_df.loc[
    persistence_mask,
    [DATETIME, STATION]
].copy()

persistence_metrics = save_model_diagnostics(
    model_name="Persistence",
    model_type="BENCHMARK",
    y_true=persistence_actual,
    y_pred=persistence_pred,
    meta_df=persistence_meta,
    output_root=BENCHMARK_DIR
)

all_results.append(
    {
        **persistence_metrics,
        "Model": "Persistence",
        "Model_Type": "BENCHMARK",
        "Evaluation_Split": "VALIDATION"
    }
)

print(
    "Persistence RMSE:",
    f"{persistence_metrics['RMSE']:.6f}",
    "| MAE:",
    f"{persistence_metrics['MAE']:.6f}",
    "| R2:",
    f"{persistence_metrics['R2']:.6f}",
    "| ±10:",
    f"{persistence_metrics['Accuracy_pm10_percent']:.4f}%"
)


# ============================================================
# 13. MASTER VALIDATION RESULTS TABLE
# ============================================================

validation_results = pd.DataFrame(
    all_results
)

validation_results = validation_results[
    [
        "Model",
        "Model_Type",
        "Evaluation_Split",
        "Rows",
        "MAE",
        "RMSE",
        "R2",
        "MAPE_percent",
        "Bias_Mean_Error",
        "Median_Error",
        "Std_Error",
        "P90_Absolute_Error",
        "P95_Absolute_Error",
        "P99_Absolute_Error",
        "Max_Absolute_Error",
        "Accuracy_pm5_percent",
        "Accuracy_pm10_percent",
        "Accuracy_pm20_percent"
    ]
]

validation_results = validation_results.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

validation_results["Validation_RMSE_Rank"] = (
    np.arange(
        1,
        len(validation_results) + 1
    )
)

validation_results.to_csv(
    os.path.join(
        MASTER_DIR,
        "all_models_validation_diagnostics_summary.csv"
    ),
    index=False
)

print("\n" + "=" * 90)
print("ALL-MODEL VALIDATION SUMMARY")
print("=" * 90)

display(
    validation_results.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "MAPE_percent": "{:.3f}",
        "Bias_Mean_Error": "{:.6f}",
        "Median_Error": "{:.6f}",
        "P90_Absolute_Error": "{:.6f}",
        "P95_Absolute_Error": "{:.6f}",
        "P99_Absolute_Error": "{:.6f}",
        "Max_Absolute_Error": "{:.6f}",
        "Accuracy_pm5_percent": "{:.4f}",
        "Accuracy_pm10_percent": "{:.4f}",
        "Accuracy_pm20_percent": "{:.4f}"
    })
)


# ============================================================
# 14. ALL-MODEL COMPARISON GRAPHS
# ============================================================

summary_plot = validation_results.copy()

# RMSE
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "RMSE",
    ascending=True
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["RMSE"]
)

plt.xlabel("Validation RMSE")
plt.ylabel("Model")
plt.title(
    "All Models — Validation RMSE Comparison"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "01_all_models_validation_rmse.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# MAE
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "MAE",
    ascending=True
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["MAE"]
)

plt.xlabel("Validation MAE")
plt.ylabel("Model")
plt.title(
    "All Models — Validation MAE Comparison"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "02_all_models_validation_mae.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# R2
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "R2",
    ascending=True
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["R2"]
)

plt.xlabel("Validation R²")
plt.ylabel("Model")
plt.title(
    "All Models — Validation R² Comparison"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "03_all_models_validation_r2.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# ±5 / ±10 / ±20
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "Accuracy_pm10_percent",
    ascending=True
)

x = np.arange(
    len(plot_tmp)
)

width = 0.25

plt.bar(
    x - width,
    plot_tmp["Accuracy_pm5_percent"],
    width,
    label="±5 AQI"
)

plt.bar(
    x,
    plot_tmp["Accuracy_pm10_percent"],
    width,
    label="±10 AQI"
)

plt.bar(
    x + width,
    plot_tmp["Accuracy_pm20_percent"],
    width,
    label="±20 AQI"
)

plt.xticks(
    x,
    plot_tmp["Model"],
    rotation=55,
    ha="right"
)

plt.ylabel("Accuracy (%)")
plt.xlabel("Model")
plt.title(
    "All Models — AQI Tolerance Accuracy"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "04_all_models_tolerance_accuracy.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# Bias
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "Bias_Mean_Error"
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["Bias_Mean_Error"]
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=2
)

plt.xlabel("Mean Error = Actual - Predicted")
plt.ylabel("Model")
plt.title(
    "All Models — Prediction Bias"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "05_all_models_prediction_bias.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# Error spread
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "P95_Absolute_Error",
    ascending=True
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["P95_Absolute_Error"]
)

plt.xlabel("95th Percentile Absolute Error")
plt.ylabel("Model")
plt.title(
    "All Models — 95th Percentile Absolute Error"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "06_all_models_p95_absolute_error.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 15. FEATURE IMPORTANCE MASTER TABLE
# ============================================================

if len(importance_results) > 0:

    importance_master = pd.DataFrame(
        importance_results
    )

    importance_master.to_csv(
        os.path.join(
            MASTER_DIR,
            "all_tree_and_linear_feature_importance_summary.csv"
        ),
        index=False
    )

    print(
        "\nFeature-importance summaries:",
        len(importance_master)
    )


# ============================================================
# 16. FINAL-HOLDOUT DIAGNOSTICS FOR PERSISTENCE
# ============================================================
# Existing final ET/RF/ensemble holdout diagnostics from R.3
# are NOT recomputed here.
#
# We only add Persistence because the benchmark has no stored
# model artifact and can be reconstructed directly from AQI_lag_1h.
# ============================================================

print("\n" + "=" * 90)
print("R.6 — FINAL HOLDOUT PERSISTENCE DIAGNOSTICS")
print("=" * 90)

if os.path.exists(FINAL_HOLDOUT_PRED_FILE):

    holdout_pred_df = pd.read_parquet(
        FINAL_HOLDOUT_PRED_FILE
    )

    holdout_df = model_df[
        model_df["Split"].eq("FINAL_HOLDOUT")
    ].copy()

    holdout_df = holdout_df[
        holdout_df[TARGET].notna()
    ].copy()

    holdout_df = holdout_df.sort_values(
        [DATETIME, STATION]
    ).reset_index(drop=True)

    if len(
        holdout_df
    ) == len(
        holdout_pred_df
    ):

        holdout_persistence_mask = (
            holdout_df["AQI_lag_1h"].notna()
        )

        hp_actual = holdout_df.loc[
            holdout_persistence_mask,
            TARGET
        ].astype(float).to_numpy()

        hp_pred = holdout_df.loc[
            holdout_persistence_mask,
            "AQI_lag_1h"
        ].astype(float).to_numpy()

        hp_meta = holdout_df.loc[
            holdout_persistence_mask,
            [DATETIME, STATION]
        ].copy()

        hp_metrics = save_model_diagnostics(
            model_name="Persistence_Final_Holdout",
            model_type="BENCHMARK",
            y_true=hp_actual,
            y_pred=hp_pred,
            meta_df=hp_meta,
            output_root=BENCHMARK_DIR
        )

        pd.DataFrame([
            {
                "Model": "Persistence",
                "Evaluation_Split": "FINAL_HOLDOUT",
                **hp_metrics
            }
        ]).to_csv(
            os.path.join(
                MASTER_DIR,
                "final_holdout_persistence_diagnostics.csv"
            ),
            index=False
        )

        print(
            "Final holdout persistence RMSE:",
            f"{hp_metrics['RMSE']:.6f}"
        )

    else:

        print(
            "Prediction-row count mismatch; "
            "persistence holdout diagnostics skipped."
        )

else:

    print(
        "Final holdout prediction file not found; "
        "persistence holdout diagnostics skipped."
    )


# ============================================================
# 17. MODEL DIAGNOSTIC INDEX
# ============================================================

index_rows = []

for root_dir, model_type in [
    (BASE_DIR, "BASE"),
    (TUNED_DIR, "TUNED"),
    (ENSEMBLE_DIR, "ENSEMBLE"),
    (BENCHMARK_DIR, "BENCHMARK")
]:

    if not os.path.exists(
        root_dir
    ):
        continue

    for item in sorted(
        os.listdir(root_dir)
    ):

        item_path = os.path.join(
            root_dir,
            item
        )

        if not os.path.isdir(
            item_path
        ):
            continue

        metrics_path = os.path.join(
            item_path,
            "metrics.csv"
        )

        if not os.path.exists(
            metrics_path
        ):
            continue

        metric_one = pd.read_csv(
            metrics_path
        ).iloc[0]

        pngs = [
            f for f in os.listdir(
                item_path
            )
            if f.lower().endswith(".png")
        ]

        csvs = [
            f for f in os.listdir(
                item_path
            )
            if f.lower().endswith(".csv")
        ]

        index_rows.append({
            "Folder": item,
            "Model_Type": model_type,
            "Directory": item_path,
            "Metrics_File": metrics_path,
            "PNG_Count": len(pngs),
            "CSV_Count": len(csvs),
            "Rows": metric_one.get(
                "Rows",
                np.nan
            ),
            "MAE": metric_one.get(
                "MAE",
                np.nan
            ),
            "RMSE": metric_one.get(
                "RMSE",
                np.nan
            ),
            "R2": metric_one.get(
                "R2",
                np.nan
            )
        })

diagnostic_index = pd.DataFrame(
    index_rows
)

diagnostic_index.to_csv(
    os.path.join(
        MASTER_DIR,
        "all_model_diagnostic_artifact_index.csv"
    ),
    index=False
)


# ============================================================
# 18. R.6 METADATA
# ============================================================

metadata = {
    "phase": "R.6",
    "purpose": (
        "Generate the same diagnostics and depictions for "
        "all Phase 15 base, tuned, ensemble and benchmark models."
    ),
    "final_holdout_status": (
        "FROZEN / UNTOUCHED"
    ),
    "base_models": list(
        base_models.keys()
    ),
    "tuned_models": list(
        tuned_models.keys()
    ),
    "ensembles": list(
        ensemble_predictions.keys()
    ),
    "benchmark": [
        "Persistence"
    ],
    "validation_rows": int(
        len(valid_df)
    ),
    "training_rows": int(
        len(train_df)
    ),
    "feature_count": int(
        len(feature_cols)
    ),
    "shap_used": False,
    "validation_diagnostic_runs": True,
    "final_holdout_retrained": False,
    "final_holdout_re_evaluated_for_final_models": False
}

with open(
    os.path.join(
        MASTER_DIR,
        "r6_all_model_diagnostics_metadata.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )


# ============================================================
# 19. FINAL VERIFICATION
# ============================================================

expected_master_files = [
    "all_models_validation_diagnostics_summary.csv",
    "01_all_models_validation_rmse.png",
    "02_all_models_validation_mae.png",
    "03_all_models_validation_r2.png",
    "04_all_models_tolerance_accuracy.png",
    "05_all_models_prediction_bias.png",
    "06_all_models_p95_absolute_error.png",
    "all_model_diagnostic_artifact_index.csv",
    "r6_all_model_diagnostics_metadata.json"
]

verification_rows = []

for filename in expected_master_files:

    path = os.path.join(
        MASTER_DIR,
        filename
    )

    verification_rows.append({
        "Artifact": filename,
        "Status": (
            "FOUND"
            if os.path.exists(path)
            else "MISSING"
        ),
        "Size_MB": (
            os.path.getsize(path)
            / (1024 ** 2)
            if os.path.exists(path)
            else np.nan
        )
    })

verification = pd.DataFrame(
    verification_rows
)

verification.to_csv(
    os.path.join(
        MASTER_DIR,
        "r6_output_verification.csv"
    ),
    index=False
)

print("\n" + "=" * 90)
print("R.6 COMPLETE")
print("=" * 90)

print(
    "\nBase models generated:",
    len(base_models)
)

print(
    "Tuned models generated:",
    len(tuned_models)
)

print(
    "Ensembles generated:",
    len(ensemble_predictions)
)

print(
    "Benchmark generated: Persistence"
)

print(
    "\nDiagnostic artifacts:",
    len(diagnostic_index)
    if len(diagnostic_index) > 0
    else 0
)

print(
    "\nMaster directory:"
)

print(
    MASTER_DIR
)

print(
    "\nFinal holdout:"
)

print(
    "ET/RF/Ensemble existing diagnostics preserved."
)

print(
    "No final model retraining."
)

print(
    "No final holdout re-evaluation for final models."
)

print(
    "\nAll-model diagnostic package completed."
)

In [ ]:
# ============================================================
# R.6 — ALL MODEL DIAGNOSTICS
# ============================================================
# IMPORTANT:
# - FINAL_HOLDOUT remains frozen and untouched.
# - Base/tuned/ensemble diagnostic runs below use TRAIN -> VALIDATION.
# - These are reporting/visualization runs, not replacements for
#   the official Phase 15 validation or final holdout results.
# - No SHAP is used.
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from xgboost import XGBRegressor

warnings.filterwarnings("ignore")


# ============================================================
# 1. PATHS
# ============================================================

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

FEATURE_FILE = os.path.join(
    BASE,
    "data",
    "processed",
    "delhi_10_station_feature_engineered.parquet"
)

SPLIT_FILE = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_1_time_split_assignments.parquet"
)

FINAL_HOLDOUT_PRED_FILE = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_12_final_holdout",
    "phase15_12_final_holdout_predictions.parquet"
)

R6_DIR = os.path.join(
    BASE,
    "outputs",
    "FINAL_RESULTS",
    "02_MODEL_RESULTS",
    "R6_ALL_MODEL_DIAGNOSTICS"
)

BASE_DIR = os.path.join(
    R6_DIR,
    "BASE_MODELS"
)

TUNED_DIR = os.path.join(
    R6_DIR,
    "TUNED_MODELS"
)

ENSEMBLE_DIR = os.path.join(
    R6_DIR,
    "ENSEMBLES"
)

BENCHMARK_DIR = os.path.join(
    R6_DIR,
    "BENCHMARKS"
)

MASTER_DIR = os.path.join(
    R6_DIR,
    "MASTER"

)

for p in [
    R6_DIR,
    BASE_DIR,
    TUNED_DIR,
    ENSEMBLE_DIR,
    BENCHMARK_DIR,
    MASTER_DIR
]:
    os.makedirs(p, exist_ok=True)


# ============================================================
# 2. LOAD DATA
# ============================================================

print("=" * 90)
print("R.6 — LOADING FEATURE DATA")
print("=" * 90)

df = pd.read_parquet(FEATURE_FILE)
split_df = pd.read_parquet(SPLIT_FILE)

print("Feature rows:", f"{len(df):,}")
print("Split rows:", f"{len(split_df):,}")

TARGET = "AQI_t_plus_1"
DATETIME = "Datetime"
STATION = "StationId"

if TARGET not in df.columns:
    raise KeyError("Target column AQI_t_plus_1 not found.")

if DATETIME not in df.columns:
    raise KeyError("Datetime column not found.")

if STATION not in df.columns:
    raise KeyError("StationId column not found.")


# ============================================================
# 3. MERGE FROZEN SPLIT ASSIGNMENTS
# ============================================================

split_keys = [STATION, DATETIME]

split_use = split_df[
    split_keys + ["Split"]
].copy()

split_use[DATETIME] = pd.to_datetime(
    split_use[DATETIME]
)

df[DATETIME] = pd.to_datetime(
    df[DATETIME]
)

model_df = df.merge(
    split_use,
    on=split_keys,
    how="left",
    validate="one_to_one"
)

if model_df["Split"].isna().any():
    raise ValueError(
        "Some feature rows do not have a frozen split assignment."
    )

print("\nSplit counts:")
print(
    model_df["Split"].value_counts()
)


# ============================================================
# 4. DEFINE THE FROZEN 73-FEATURE CONTRACT
# ============================================================

freeze_contract = os.path.join(
    BASE,
    "outputs",
    "reports",
    "phase15_15_final_freeze",
    "phase15_15_feature_contract.csv"
)

if os.path.exists(freeze_contract):

    contract_df = pd.read_csv(
        freeze_contract
    )

    possible_feature_cols = [
        c for c in contract_df.columns
        if c.lower() in [
            "feature",
            "feature_name",
            "original_feature",
            "model_feature"
        ]
    ]

    if possible_feature_cols:
        contract_col = possible_feature_cols[0]
        feature_cols = contract_df[
            contract_col
        ].dropna().astype(str).tolist()
    else:
        feature_cols = [
            c for c in contract_df.columns
            if c in model_df.columns
        ]

else:

    feature_cols = [
        c for c in model_df.columns
        if c not in [
            TARGET,
            "Split",
            DATETIME,
            "StationName"
        ]
    ]

feature_cols = [
    c for c in feature_cols
    if c in model_df.columns
]

if len(feature_cols) != 73:
    raise ValueError(
        "Expected exactly 73 model features, found "
        + str(len(feature_cols))
    )

print("\nFrozen feature count:", len(feature_cols))


# ============================================================
# 5. TRAIN / VALIDATION DATA
# ============================================================

train_df = model_df[
    model_df["Split"].eq("TRAIN")
].copy()

valid_df = model_df[
    model_df["Split"].eq("VALIDATION")
].copy()

train_df = train_df[
    train_df[TARGET].notna()
].copy()

valid_df = valid_df[
    valid_df[TARGET].notna()
].copy()

train_df = train_df.sort_values(
    [DATETIME, STATION]
).reset_index(drop=True)

valid_df = valid_df.sort_values(
    [DATETIME, STATION]
).reset_index(drop=True)

print("\nTRAIN rows:", f"{len(train_df):,}")
print("VALIDATION rows:", f"{len(valid_df):,}")

if len(train_df) == 0 or len(valid_df) == 0:
    raise ValueError("TRAIN or VALIDATION set is empty.")


X_train_raw = train_df[feature_cols].copy()
y_train = train_df[TARGET].astype(float).to_numpy()

X_valid_raw = valid_df[feature_cols].copy()
y_valid = valid_df[TARGET].astype(float).to_numpy()


# ============================================================
# 6. PREPROCESSOR
# ============================================================

categorical_features = []

if STATION in feature_cols:
    categorical_features.append(STATION)

numeric_features = [
    c for c in feature_cols
    if c not in categorical_features
]

numeric_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )
])

categorical_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipe,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipe,
            categorical_features
        )
    ],
    remainder="drop"
)

print("\nFitting common preprocessing on TRAIN only...")

X_train = preprocessor.fit_transform(
    X_train_raw
)

X_valid = preprocessor.transform(
    X_valid_raw
)

try:
    transformed_feature_names = list(
        preprocessor.get_feature_names_out()
    )
except Exception:
    transformed_feature_names = [
        f"Feature_{i}"
        for i in range(X_train.shape[1])
    ]

print(
    "Transformed feature count:",
    X_train.shape[1]
)


# ============================================================
# 7. MODEL DEFINITIONS
# ============================================================

base_models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=42
    ),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=100,
        random_state=42
    ),

    "AdaBoost": AdaBoostRegressor(
        n_estimators=100,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        min_child_weight=1,
        subsample=1.0,
        colsample_bytree=1.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

tuned_models = {
    "Tuned Extra Trees": ExtraTreesRegressor(
        n_estimators=300,
        max_depth=26,
        min_samples_split=2,
        min_samples_leaf=2,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    ),

    "Tuned Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=26,
        min_samples_split=2,
        min_samples_leaf=10,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    ),

    "Tuned XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        min_child_weight=1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}


# ============================================================
# 8. DIAGNOSTIC FUNCTIONS
# ============================================================

AQI_BINS = [
    -np.inf,
    100,
    200,
    300,
    400,
    np.inf
]

AQI_LABELS = [
    "<100",
    "100–199",
    "200–299",
    "300–399",
    "≥400"
]


def safe_mape(y_true, y_pred):
    denom = np.where(
        np.abs(y_true) < 1e-9,
        np.nan,
        np.abs(y_true)
    )
    return float(
        np.nanmean(
            np.abs(
                (y_true - y_pred) / denom
            )
        )
        * 100
    )


def compute_metrics(
    y_true,
    y_pred
):

    residual = y_true - y_pred
    abs_error = np.abs(residual)

    return {
        "Rows": int(len(y_true)),
        "MAE": float(
            mean_absolute_error(
                y_true,
                y_pred
            )
        ),
        "RMSE": float(
            np.sqrt(
                mean_squared_error(
                    y_true,
                    y_pred
                )
            )
        ),
        "R2": float(
            r2_score(
                y_true,
                y_pred
            )
        ),
        "MAPE_percent": safe_mape(
            y_true,
            y_pred
        ),
        "Bias_Mean_Error": float(
            np.mean(residual)
        ),
        "Median_Error": float(
            np.median(residual)
        ),
        "Std_Error": float(
            np.std(residual)
        ),
        "P90_Absolute_Error": float(
            np.percentile(
                abs_error,
                90
            )
        ),
        "P95_Absolute_Error": float(
            np.percentile(
                abs_error,
                95
            )
        ),
        "P99_Absolute_Error": float(
            np.percentile(
                abs_error,
                99
            )
        ),
        "Max_Absolute_Error": float(
            np.max(abs_error)
        ),
        "Accuracy_pm5_percent": float(
            np.mean(abs_error <= 5)
            * 100
        ),
        "Accuracy_pm10_percent": float(
            np.mean(abs_error <= 10)
            * 100
        ),
        "Accuracy_pm20_percent": float(
            np.mean(abs_error <= 20)
            * 100
        )
    }


def add_aqi_range(
    y_true
):

    return pd.cut(
        y_true,
        bins=AQI_BINS,
        labels=AQI_LABELS,
        right=False,
        include_lowest=True
    )


def build_category(
    values
):

    return pd.cut(
        values,
        bins=[
            -np.inf,
            50,
            100,
            200,
            300,
            400,
            np.inf
        ],
        labels=[
            "Good",
            "Satisfactory",
            "Moderate",
            "Poor",
            "Very Poor",
            "Severe"
        ],
        right=True
    )


def save_model_diagnostics(
    model_name,
    model_type,
    y_true,
    y_pred,
    meta_df,
    output_root
):

    safe_name = (
        model_name
        .replace(" ", "_")
        .replace("/", "_")
        .replace("+", "plus")
    )

    model_dir = os.path.join(
        output_root,
        safe_name
    )

    os.makedirs(
        model_dir,
        exist_ok=True
    )

    temp = meta_df[
        [DATETIME, STATION]
    ].copy()

    temp["Actual_AQI_t_plus_1"] = y_true
    temp["Predicted_AQI_t_plus_1"] = y_pred
    temp["Residual"] = y_true - y_pred
    temp["Absolute_Error"] = np.abs(
        temp["Residual"]
    )
    temp["Squared_Error"] = (
        temp["Residual"] ** 2
    )
    temp["AQI_Range"] = add_aqi_range(
        y_true
    )

    temp = temp.sort_values(
        [DATETIME, STATION]
    ).reset_index(drop=True)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics = compute_metrics(
        y_true,
        y_pred
    )

    metrics["Model"] = model_name
    metrics["Model_Type"] = model_type
    metrics["Evaluation_Split"] = "VALIDATION"
    metrics["Diagnostic_Run"] = True

    metrics_df = pd.DataFrame([
        metrics
    ])

    metrics_df.to_csv(
        os.path.join(
            model_dir,
            "metrics.csv"
        ),
        index=False
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    temp.to_parquet(
        os.path.join(
            model_dir,
            "predictions.parquet"
        ),
        index=False
    )

    # --------------------------------------------------------
    # 1. Actual vs Predicted
    # --------------------------------------------------------

    plt.figure(
        figsize=(9, 8)
    )

    plt.scatter(
        y_true,
        y_pred,
        s=8,
        alpha=0.25
    )

    low = min(
        np.nanmin(y_true),
        np.nanmin(y_pred)
    )

    high = max(
        np.nanmax(y_true),
        np.nanmax(y_pred)
    )

    plt.plot(
        [low, high],
        [low, high],
        linewidth=2
    )

    plt.xlabel("Actual AQI")
    plt.ylabel("Predicted AQI")
    plt.title(
        model_name
        + " — Actual vs Predicted"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "01_actual_vs_predicted.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 2. Residual distribution
    # --------------------------------------------------------

    plt.figure(
        figsize=(10, 6)
    )

    plt.hist(
        temp["Residual"],
        bins=80,
        alpha=0.8
    )

    plt.axvline(
        0,
        linestyle="--",
        linewidth=2
    )

    plt.xlabel("Residual = Actual - Predicted")
    plt.ylabel("Frequency")
    plt.title(
        model_name
        + " — Residual Distribution"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "02_residual_distribution.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 3. Residual vs predicted
    # --------------------------------------------------------

    plt.figure(
        figsize=(10, 6)
    )

    plt.scatter(
        temp["Predicted_AQI_t_plus_1"],
        temp["Residual"],
        s=8,
        alpha=0.25
    )

    plt.axhline(
        0,
        linestyle="--",
        linewidth=2
    )

    plt.xlabel("Predicted AQI")
    plt.ylabel("Residual")
    plt.title(
        model_name
        + " — Residual vs Predicted"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "03_residual_vs_predicted.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 4. Error over time
    # --------------------------------------------------------

    plt.figure(
        figsize=(14, 6)
    )

    plt.plot(
        temp[DATETIME],
        temp["Absolute_Error"],
        linewidth=0.8
    )

    plt.xlabel("Datetime")
    plt.ylabel("Absolute Error")
    plt.title(
        model_name
        + " — Absolute Prediction Error Over Time"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "04_error_over_time.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 5. Actual vs predicted time series
    # --------------------------------------------------------

    plot_n = min(
        1500,
        len(temp)
    )

    ts = temp.head(
        plot_n
    )

    plt.figure(
        figsize=(14, 6)
    )

    plt.plot(
        ts[DATETIME],
        ts["Actual_AQI_t_plus_1"],
        label="Actual",
        linewidth=1.2
    )

    plt.plot(
        ts[DATETIME],
        ts["Predicted_AQI_t_plus_1"],
        label="Predicted",
        linewidth=1.0,
        alpha=0.8
    )

    plt.xlabel("Datetime")
    plt.ylabel("AQI")
    plt.title(
        model_name
        + " — Actual vs Predicted Time Series"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "05_actual_vs_predicted_time_series.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 6. AQI range comparison
    # --------------------------------------------------------

    range_rows = []

    for label in AQI_LABELS:

        subset = temp[
            temp["AQI_Range"].astype(str).eq(label)
        ].copy()

        if len(subset) == 0:
            continue

        range_rows.append({
            "AQI_Range": label,
            "Records": len(subset),
            "Actual_Mean": subset[
                "Actual_AQI_t_plus_1"
            ].mean(),
            "Predicted_Mean": subset[
                "Predicted_AQI_t_plus_1"
            ].mean(),
            "MAE": subset[
                "Absolute_Error"
            ].mean(),
            "RMSE": np.sqrt(
                np.mean(
                    subset["Squared_Error"]
                )
            ),
            "Accuracy_pm10_percent": np.mean(
                subset["Absolute_Error"] <= 10
            ) * 100,
            "Mean_Error": subset[
                "Residual"
            ].mean()
        })

    range_df = pd.DataFrame(
        range_rows
    )

    range_df.to_csv(
        os.path.join(
            model_dir,
            "06_aqi_range_metrics.csv"
        ),
        index=False
    )

    if len(range_df) > 0:

        x = np.arange(
            len(range_df)
        )

        width = 0.35

        plt.figure(
            figsize=(11, 6)
        )

        plt.bar(
            x - width / 2,
            range_df["Actual_Mean"],
            width,
            label="Actual Mean"
        )

        plt.bar(
            x + width / 2,
            range_df["Predicted_Mean"],
            width,
            label="Predicted Mean"
        )

        plt.xticks(
            x,
            range_df["AQI_Range"]
        )

        plt.ylabel("AQI")
        plt.xlabel("AQI Range")
        plt.title(
            model_name
            + " — Actual vs Predicted by AQI Range"
        )

        plt.legend()

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                model_dir,
                "07_actual_vs_predicted_by_aqi_range.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )

        plt.close()

        # AQI-range MAE

        plt.figure(
            figsize=(11, 6)
        )

        plt.bar(
            range_df["AQI_Range"],
            range_df["MAE"]
        )

        plt.ylabel("MAE")
        plt.xlabel("AQI Range")
        plt.title(
            model_name
            + " — MAE by AQI Range"
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                model_dir,
                "08_mae_by_aqi_range.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )

        plt.close()

    # --------------------------------------------------------
    # 7. Category confusion matrix
    # --------------------------------------------------------

    actual_cat = build_category(
        y_true
    ).astype(str)

    pred_cat = build_category(
        y_pred
    ).astype(str)

    category_labels = [
        "Good",
        "Satisfactory",
        "Moderate",
        "Poor",
        "Very Poor",
        "Severe"
    ]

    confusion = pd.crosstab(
        actual_cat,
        pred_cat,
        dropna=False
    )

    confusion = confusion.reindex(
        index=category_labels,
        columns=category_labels,
        fill_value=0
    )

    confusion.to_csv(
        os.path.join(
            model_dir,
            "09_aqi_category_confusion_matrix.csv"
        )
    )

    plt.figure(
        figsize=(9, 8)
    )

    plt.imshow(
        confusion.values,
        aspect="auto"
    )

    plt.xticks(
        range(len(category_labels)),
        category_labels,
        rotation=30,
        ha="right"
    )

    plt.yticks(
        range(len(category_labels)),
        category_labels
    )

    plt.xlabel("Predicted AQI Category")
    plt.ylabel("Actual AQI Category")
    plt.title(
        model_name
        + " — AQI Category Confusion Matrix"
    )

    for i in range(
        confusion.shape[0]
    ):
        for j in range(
            confusion.shape[1]
        ):
            plt.text(
                j,
                i,
                str(
                    confusion.iloc[i, j]
                ),
                ha="center",
                va="center"
            )

    plt.colorbar(
        label="Number of observations"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "10_aqi_category_confusion_matrix.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 8. Error distribution / quantile plot
    # --------------------------------------------------------

    quantiles = [
        50,
        75,
        90,
        95,
        99,
        99.5
    ]

    q_values = np.percentile(
        temp["Absolute_Error"],
        quantiles
    )

    q_df = pd.DataFrame({
        "Percentile": quantiles,
        "Absolute_Error": q_values
    })

    q_df.to_csv(
        os.path.join(
            model_dir,
            "11_error_quantiles.csv"
        ),
        index=False
    )

    plt.figure(
        figsize=(10, 6)
    )

    plt.plot(
        quantiles,
        q_values,
        marker="o"
    )

    plt.xlabel("Percentile")
    plt.ylabel("Absolute Error")
    plt.title(
        model_name
        + " — Absolute Error Quantiles"
    )

    plt.grid(
        alpha=0.25
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "12_error_quantiles.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    # --------------------------------------------------------
    # 9. Error boxplot by AQI range
    # --------------------------------------------------------

    grouped_errors = []

    for label in AQI_LABELS:

        vals = temp.loc[
            temp["AQI_Range"].astype(str).eq(label),
            "Absolute_Error"
        ].values

        if len(vals) > 0:
            grouped_errors.append(vals)

    if len(grouped_errors) > 0:

        plt.figure(
            figsize=(11, 6)
        )

        plt.boxplot(
            grouped_errors,
            labels=[
                AQI_LABELS[i]
                for i in range(
                    len(grouped_errors)
                )
            ],
            showfliers=False
        )

        plt.xlabel("AQI Range")
        plt.ylabel("Absolute Error")
        plt.title(
            model_name
            + " — Error Distribution by AQI Range"
        )

        plt.tight_layout()

        plt.savefig(
            os.path.join(
                model_dir,
                "13_error_boxplot_by_aqi_range.png"
            ),
            dpi=200,
            bbox_inches="tight"
        )

        plt.close()

    # --------------------------------------------------------
    # 10. Worst prediction cases
    # --------------------------------------------------------

    worst = temp.sort_values(
        "Absolute_Error",
        ascending=False
    ).head(
        50
    ).copy()

    worst.to_csv(
        os.path.join(
            model_dir,
            "14_worst_prediction_cases.csv"
        ),
        index=False
    )

    # --------------------------------------------------------
    # 11. Best prediction cases
    # --------------------------------------------------------

    best = temp.sort_values(
        "Absolute_Error",
        ascending=True
    ).head(
        50
    ).copy()

    best.to_csv(
        os.path.join(
            model_dir,
            "15_best_prediction_cases.csv"
        ),
        index=False
    )

    return metrics


def save_feature_importance(
    model,
    model_name,
    output_root,
    transformed_names
):

    safe_name = (
        model_name
        .replace(" ", "_")
        .replace("/", "_")
        .replace("+", "plus")
    )

    model_dir = os.path.join(
        output_root,
        safe_name
    )

    os.makedirs(
        model_dir,
        exist_ok=True
    )

    importance = None
    importance_type = None

    if hasattr(
        model,
        "feature_importances_"
    ):

        importance = np.asarray(
            model.feature_importances_,
            dtype=float
        )

        importance_type = "native_feature_importance"

    elif hasattr(
        model,
        "coef_"
    ):

        importance = np.abs(
            np.asarray(
                model.coef_,
                dtype=float
            )
        )

        importance_type = "absolute_linear_coefficient"

    if importance is None:
        return None

    if len(importance) != len(transformed_names):
        return None

    importance_df = pd.DataFrame({
        "Feature": transformed_names,
        "Importance": importance
    })

    total = importance_df["Importance"].sum()

    if total > 0:
        importance_df[
            "Relative_Importance_percent"
        ] = (
            importance_df["Importance"]
            / total
            * 100
        )
    else:
        importance_df[
            "Relative_Importance_percent"
        ] = np.nan

    importance_df = importance_df.sort_values(
        "Importance",
        ascending=False
    ).reset_index(drop=True)

    importance_df.to_csv(
        os.path.join(
            model_dir,
            "16_feature_importance.csv"
        ),
        index=False
    )

    top_n = min(
        20,
        len(importance_df)
    )

    top = importance_df.head(
        top_n
    ).iloc[::-1]

    plt.figure(
        figsize=(11, 8)
    )

    plt.barh(
        top["Feature"],
        top["Importance"]
    )

    plt.xlabel(
        "Importance"
    )

    plt.ylabel(
        "Feature"
    )

    plt.title(
        model_name
        + " — Top Feature Importance"
    )

    plt.tight_layout()

    plt.savefig(
        os.path.join(
            model_dir,
            "17_top_feature_importance.png"
        ),
        dpi=200,
        bbox_inches="tight"
    )

    plt.close()

    return {
        "Model": model_name,
        "Importance_Type": importance_type,
        "Top_Feature": importance_df.iloc[0]["Feature"],
        "Top_Importance": float(
            importance_df.iloc[0]["Importance"]
        )
    }


# ============================================================
# 9. RUN BASE MODELS
# ============================================================

all_results = []
importance_results = []

print("\n" + "=" * 90)
print("R.6 — BASE MODELS")
print("=" * 90)

base_predictions = {}

for model_name, model in base_models.items():

    print("\nTraining:", model_name)

    model.fit(
        X_train,
        y_train
    )

    pred = np.asarray(
        model.predict(
            X_valid
        ),
        dtype=float
    )

    base_predictions[model_name] = pred

    metrics = save_model_diagnostics(
        model_name=model_name,
        model_type="BASE",
        y_true=y_valid,
        y_pred=pred,
        meta_df=valid_df,
        output_root=BASE_DIR
    )

    importance = save_feature_importance(
        model=model,
        model_name=model_name,
        output_root=BASE_DIR,
        transformed_names=transformed_feature_names
    )

    all_results.append(
        {
            **metrics,
            "Model": model_name,
            "Model_Type": "BASE",
            "Evaluation_Split": "VALIDATION"
        }
    )

    if importance is not None:
        importance_results.append(
            importance
        )

    print(
        "RMSE:",
        f"{metrics['RMSE']:.6f}",
        "| MAE:",
        f"{metrics['MAE']:.6f}",
        "| R2:",
        f"{metrics['R2']:.6f}",
        "| ±10:",
        f"{metrics['Accuracy_pm10_percent']:.4f}%"
    )


# ============================================================
# 10. RUN TUNED MODELS
# ============================================================

print("\n" + "=" * 90)
print("R.6 — TUNED MODELS")
print("=" * 90)

tuned_predictions = {}

for model_name, model in tuned_models.items():

    print("\nTraining:", model_name)

    model.fit(
        X_train,
        y_train
    )

    pred = np.asarray(
        model.predict(
            X_valid
        ),
        dtype=float
    )

    tuned_predictions[model_name] = pred

    metrics = save_model_diagnostics(
        model_name=model_name,
        model_type="TUNED",
        y_true=y_valid,
        y_pred=pred,
        meta_df=valid_df,
        output_root=TUNED_DIR
    )

    importance = save_feature_importance(
        model=model,
        model_name=model_name,
        output_root=TUNED_DIR,
        transformed_names=transformed_feature_names
    )

    all_results.append(
        {
            **metrics,
            "Model": model_name,
            "Model_Type": "TUNED",
            "Evaluation_Split": "VALIDATION"
        }
    )

    if importance is not None:
        importance_results.append(
            importance
        )

    print(
        "RMSE:",
        f"{metrics['RMSE']:.6f}",
        "| MAE:",
        f"{metrics['MAE']:.6f}",
        "| R2:",
        f"{metrics['R2']:.6f}",
        "| ±10:",
        f"{metrics['Accuracy_pm10_percent']:.4f}%"
    )


# ============================================================
# 11. ENSEMBLES
# ============================================================

print("\n" + "=" * 90)
print("R.6 — ENSEMBLES")
print("=" * 90)

et_valid = tuned_predictions[
    "Tuned Extra Trees"
]

rf_valid = tuned_predictions[
    "Tuned Random Forest"
]

ensemble_5050 = (
    0.5 * et_valid
    + 0.5 * rf_valid
)

ensemble_optimized = (
    0.594466 * et_valid
    + 0.405534 * rf_valid
)

ensemble_predictions = {
    "50/50 ET + RF": ensemble_5050,
    "Optimized Weighted ET + RF": ensemble_optimized
}

for ensemble_name, pred in ensemble_predictions.items():

    metrics = save_model_diagnostics(
        model_name=ensemble_name,
        model_type="ENSEMBLE",
        y_true=y_valid,
        y_pred=pred,
        meta_df=valid_df,
        output_root=ENSEMBLE_DIR
    )

    all_results.append(
        {
            **metrics,
            "Model": ensemble_name,
            "Model_Type": "ENSEMBLE",
            "Evaluation_Split": "VALIDATION"
        }
    )

    print(
        ensemble_name,
        "| RMSE:",
        f"{metrics['RMSE']:.6f}",
        "| MAE:",
        f"{metrics['MAE']:.6f}",
        "| R2:",
        f"{metrics['R2']:.6f}",
        "| ±10:",
        f"{metrics['Accuracy_pm10_percent']:.4f}%"
    )


# ============================================================
# 12. PERSISTENCE BENCHMARK
# ============================================================

print("\n" + "=" * 90)
print("R.6 — PERSISTENCE BENCHMARK")
print("=" * 90)

if "AQI_lag_1h" not in valid_df.columns:
    raise KeyError(
        "AQI_lag_1h is required for persistence benchmark."
    )

persistence_mask = (
    valid_df["AQI_lag_1h"].notna()
    & valid_df[TARGET].notna()
)

persistence_actual = valid_df.loc[
    persistence_mask,
    TARGET
].astype(float).to_numpy()

persistence_pred = valid_df.loc[
    persistence_mask,
    "AQI_lag_1h"
].astype(float).to_numpy()

persistence_meta = valid_df.loc[
    persistence_mask,
    [DATETIME, STATION]
].copy()

persistence_metrics = save_model_diagnostics(
    model_name="Persistence",
    model_type="BENCHMARK",
    y_true=persistence_actual,
    y_pred=persistence_pred,
    meta_df=persistence_meta,
    output_root=BENCHMARK_DIR
)

all_results.append(
    {
        **persistence_metrics,
        "Model": "Persistence",
        "Model_Type": "BENCHMARK",
        "Evaluation_Split": "VALIDATION"
    }
)

print(
    "Persistence RMSE:",
    f"{persistence_metrics['RMSE']:.6f}",
    "| MAE:",
    f"{persistence_metrics['MAE']:.6f}",
    "| R2:",
    f"{persistence_metrics['R2']:.6f}",
    "| ±10:",
    f"{persistence_metrics['Accuracy_pm10_percent']:.4f}%"
)


# ============================================================
# 13. MASTER VALIDATION RESULTS TABLE
# ============================================================

validation_results = pd.DataFrame(
    all_results
)

validation_results = validation_results[
    [
        "Model",
        "Model_Type",
        "Evaluation_Split",
        "Rows",
        "MAE",
        "RMSE",
        "R2",
        "MAPE_percent",
        "Bias_Mean_Error",
        "Median_Error",
        "Std_Error",
        "P90_Absolute_Error",
        "P95_Absolute_Error",
        "P99_Absolute_Error",
        "Max_Absolute_Error",
        "Accuracy_pm5_percent",
        "Accuracy_pm10_percent",
        "Accuracy_pm20_percent"
    ]
]

validation_results = validation_results.sort_values(
    "RMSE",
    ascending=True
).reset_index(drop=True)

validation_results["Validation_RMSE_Rank"] = (
    np.arange(
        1,
        len(validation_results) + 1
    )
)

validation_results.to_csv(
    os.path.join(
        MASTER_DIR,
        "all_models_validation_diagnostics_summary.csv"
    ),
    index=False
)

print("\n" + "=" * 90)
print("ALL-MODEL VALIDATION SUMMARY")
print("=" * 90)

display(
    validation_results.style.format({
        "MAE": "{:.6f}",
        "RMSE": "{:.6f}",
        "R2": "{:.6f}",
        "MAPE_percent": "{:.3f}",
        "Bias_Mean_Error": "{:.6f}",
        "Median_Error": "{:.6f}",
        "P90_Absolute_Error": "{:.6f}",
        "P95_Absolute_Error": "{:.6f}",
        "P99_Absolute_Error": "{:.6f}",
        "Max_Absolute_Error": "{:.6f}",
        "Accuracy_pm5_percent": "{:.4f}",
        "Accuracy_pm10_percent": "{:.4f}",
        "Accuracy_pm20_percent": "{:.4f}"
    })
)


# ============================================================
# 14. ALL-MODEL COMPARISON GRAPHS
# ============================================================

summary_plot = validation_results.copy()

# RMSE
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "RMSE",
    ascending=True
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["RMSE"]
)

plt.xlabel("Validation RMSE")
plt.ylabel("Model")
plt.title(
    "All Models — Validation RMSE Comparison"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "01_all_models_validation_rmse.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# MAE
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "MAE",
    ascending=True
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["MAE"]
)

plt.xlabel("Validation MAE")
plt.ylabel("Model")
plt.title(
    "All Models — Validation MAE Comparison"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "02_all_models_validation_mae.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# R2
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "R2",
    ascending=True
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["R2"]
)

plt.xlabel("Validation R²")
plt.ylabel("Model")
plt.title(
    "All Models — Validation R² Comparison"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "03_all_models_validation_r2.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# ±5 / ±10 / ±20
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "Accuracy_pm10_percent",
    ascending=True
)

x = np.arange(
    len(plot_tmp)
)

width = 0.25

plt.bar(
    x - width,
    plot_tmp["Accuracy_pm5_percent"],
    width,
    label="±5 AQI"
)

plt.bar(
    x,
    plot_tmp["Accuracy_pm10_percent"],
    width,
    label="±10 AQI"
)

plt.bar(
    x + width,
    plot_tmp["Accuracy_pm20_percent"],
    width,
    label="±20 AQI"
)

plt.xticks(
    x,
    plot_tmp["Model"],
    rotation=55,
    ha="right"
)

plt.ylabel("Accuracy (%)")
plt.xlabel("Model")
plt.title(
    "All Models — AQI Tolerance Accuracy"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "04_all_models_tolerance_accuracy.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# Bias
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "Bias_Mean_Error"
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["Bias_Mean_Error"]
)

plt.axvline(
    0,
    linestyle="--",
    linewidth=2
)

plt.xlabel("Mean Error = Actual - Predicted")
plt.ylabel("Model")
plt.title(
    "All Models — Prediction Bias"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "05_all_models_prediction_bias.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# Error spread
plt.figure(
    figsize=(14, 8)
)

plot_tmp = summary_plot.sort_values(
    "P95_Absolute_Error",
    ascending=True
)

plt.barh(
    plot_tmp["Model"],
    plot_tmp["P95_Absolute_Error"]
)

plt.xlabel("95th Percentile Absolute Error")
plt.ylabel("Model")
plt.title(
    "All Models — 95th Percentile Absolute Error"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        MASTER_DIR,
        "06_all_models_p95_absolute_error.png"
    ),
    dpi=200,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 15. FEATURE IMPORTANCE MASTER TABLE
# ============================================================

if len(importance_results) > 0:

    importance_master = pd.DataFrame(
        importance_results
    )

    importance_master.to_csv(
        os.path.join(
            MASTER_DIR,
            "all_tree_and_linear_feature_importance_summary.csv"
        ),
        index=False
    )

    print(
        "\nFeature-importance summaries:",
        len(importance_master)
    )


# ============================================================
# 16. FINAL-HOLDOUT DIAGNOSTICS FOR PERSISTENCE
# ============================================================
# Existing final ET/RF/ensemble holdout diagnostics from R.3
# are NOT recomputed here.
#
# We only add Persistence because the benchmark has no stored
# model artifact and can be reconstructed directly from AQI_lag_1h.
# ============================================================

print("\n" + "=" * 90)
print("R.6 — FINAL HOLDOUT PERSISTENCE DIAGNOSTICS")
print("=" * 90)

if os.path.exists(FINAL_HOLDOUT_PRED_FILE):

    holdout_pred_df = pd.read_parquet(
        FINAL_HOLDOUT_PRED_FILE
    )

    holdout_df = model_df[
        model_df["Split"].eq("FINAL_HOLDOUT")
    ].copy()

    holdout_df = holdout_df[
        holdout_df[TARGET].notna()
    ].copy()

    holdout_df = holdout_df.sort_values(
        [DATETIME, STATION]
    ).reset_index(drop=True)

    if len(
        holdout_df
    ) == len(
        holdout_pred_df
    ):

        holdout_persistence_mask = (
            holdout_df["AQI_lag_1h"].notna()
        )

        hp_actual = holdout_df.loc[
            holdout_persistence_mask,
            TARGET
        ].astype(float).to_numpy()

        hp_pred = holdout_df.loc[
            holdout_persistence_mask,
            "AQI_lag_1h"
        ].astype(float).to_numpy()

        hp_meta = holdout_df.loc[
            holdout_persistence_mask,
            [DATETIME, STATION]
        ].copy()

        hp_metrics = save_model_diagnostics(
            model_name="Persistence_Final_Holdout",
            model_type="BENCHMARK",
            y_true=hp_actual,
            y_pred=hp_pred,
            meta_df=hp_meta,
            output_root=BENCHMARK_DIR
        )

        pd.DataFrame([
            {
                "Model": "Persistence",
                "Evaluation_Split": "FINAL_HOLDOUT",
                **hp_metrics
            }
        ]).to_csv(
            os.path.join(
                MASTER_DIR,
                "final_holdout_persistence_diagnostics.csv"
            ),
            index=False
        )

        print(
            "Final holdout persistence RMSE:",
            f"{hp_metrics['RMSE']:.6f}"
        )

    else:

        print(
            "Prediction-row count mismatch; "
            "persistence holdout diagnostics skipped."
        )

else:

    print(
        "Final holdout prediction file not found; "
        "persistence holdout diagnostics skipped."
    )


# ============================================================
# 17. MODEL DIAGNOSTIC INDEX
# ============================================================

index_rows = []

for root_dir, model_type in [
    (BASE_DIR, "BASE"),
    (TUNED_DIR, "TUNED"),
    (ENSEMBLE_DIR, "ENSEMBLE"),
    (BENCHMARK_DIR, "BENCHMARK")
]:

    if not os.path.exists(
        root_dir
    ):
        continue

    for item in sorted(
        os.listdir(root_dir)
    ):

        item_path = os.path.join(
            root_dir,
            item
        )

        if not os.path.isdir(
            item_path
        ):
            continue

        metrics_path = os.path.join(
            item_path,
            "metrics.csv"
        )

        if not os.path.exists(
            metrics_path
        ):
            continue

        metric_one = pd.read_csv(
            metrics_path
        ).iloc[0]

        pngs = [
            f for f in os.listdir(
                item_path
            )
            if f.lower().endswith(".png")
        ]

        csvs = [
            f for f in os.listdir(
                item_path
            )
            if f.lower().endswith(".csv")
        ]

        index_rows.append({
            "Folder": item,
            "Model_Type": model_type,
            "Directory": item_path,
            "Metrics_File": metrics_path,
            "PNG_Count": len(pngs),
            "CSV_Count": len(csvs),
            "Rows": metric_one.get(
                "Rows",
                np.nan
            ),
            "MAE": metric_one.get(
                "MAE",
                np.nan
            ),
            "RMSE": metric_one.get(
                "RMSE",
                np.nan
            ),
            "R2": metric_one.get(
                "R2",
                np.nan
            )
        })

diagnostic_index = pd.DataFrame(
    index_rows
)

diagnostic_index.to_csv(
    os.path.join(
        MASTER_DIR,
        "all_model_diagnostic_artifact_index.csv"
    ),
    index=False
)


# ============================================================
# 18. R.6 METADATA
# ============================================================

metadata = {
    "phase": "R.6",
    "purpose": (
        "Generate the same diagnostics and depictions for "
        "all Phase 15 base, tuned, ensemble and benchmark models."
    ),
    "final_holdout_status": (
        "FROZEN / UNTOUCHED"
    ),
    "base_models": list(
        base_models.keys()
    ),
    "tuned_models": list(
        tuned_models.keys()
    ),
    "ensembles": list(
        ensemble_predictions.keys()
    ),
    "benchmark": [
        "Persistence"
    ],
    "validation_rows": int(
        len(valid_df)
    ),
    "training_rows": int(
        len(train_df)
    ),
    "feature_count": int(
        len(feature_cols)
    ),
    "shap_used": False,
    "validation_diagnostic_runs": True,
    "final_holdout_retrained": False,
    "final_holdout_re_evaluated_for_final_models": False
}

with open(
    os.path.join(
        MASTER_DIR,
        "r6_all_model_diagnostics_metadata.json"
    ),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metadata,
        f,
        indent=2
    )


# ============================================================
# 19. FINAL VERIFICATION
# ============================================================

expected_master_files = [
    "all_models_validation_diagnostics_summary.csv",
    "01_all_models_validation_rmse.png",
    "02_all_models_validation_mae.png",
    "03_all_models_validation_r2.png",
    "04_all_models_tolerance_accuracy.png",
    "05_all_models_prediction_bias.png",
    "06_all_models_p95_absolute_error.png",
    "all_model_diagnostic_artifact_index.csv",
    "r6_all_model_diagnostics_metadata.json"
]

verification_rows = []

for filename in expected_master_files:

    path = os.path.join(
        MASTER_DIR,
        filename
    )

    verification_rows.append({
        "Artifact": filename,
        "Status": (
            "FOUND"
            if os.path.exists(path)
            else "MISSING"
        ),
        "Size_MB": (
            os.path.getsize(path)
            / (1024 ** 2)
            if os.path.exists(path)
            else np.nan
        )
    })

verification = pd.DataFrame(
    verification_rows
)

verification.to_csv(
    os.path.join(
        MASTER_DIR,
        "r6_output_verification.csv"
    ),
    index=False
)

print("\n" + "=" * 90)
print("R.6 COMPLETE")
print("=" * 90)

print(
    "\nBase models generated:",
    len(base_models)
)

print(
    "Tuned models generated:",
    len(tuned_models)
)

print(
    "Ensembles generated:",
    len(ensemble_predictions)
)

print(
    "Benchmark generated: Persistence"
)

print(
    "\nDiagnostic artifacts:",
    len(diagnostic_index)
    if len(diagnostic_index) > 0
    else 0
)

print(
    "\nMaster directory:"
)

print(
    MASTER_DIR
)

print(
    "\nFinal holdout:"
)

print(
    "ET/RF/Ensemble existing diagnostics preserved."
)

print(
    "No final model retraining."
)

print(
    "No final holdout re-evaluation for final models."
)

print(
    "\nAll-model diagnostic package completed."
)

In [ ]:
# ============================================================
# PROJECT STOP CHECKPOINT — AFTER R.5 / BEFORE OR DURING R.6
# ============================================================

import os
import json
import joblib
import pandas as pd
from datetime import datetime

BASE = "/content/gdrive/MyDrive/Delhi-AQI-Prediction"

CHECKPOINT_DIR = os.path.join(
    BASE,
    "outputs",
    "checkpoints"
)

SESSION_DIR = os.path.join(
    CHECKPOINT_DIR,
    "session_runtime"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

os.makedirs(
    SESSION_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. PROJECT STATUS
# ------------------------------------------------------------

status = {
    "project": "Delhi AQI t+1 Forecasting",
    "objective": "Predict AQI at t+1 hour using information available up to t",
    "target": "AQI_t_plus_1",

    "completed_through": "Phase 15.16",
    "current_reporting_stage": "R.6",
    "phase_15_status": "COMPLETE / FROZEN",
    "final_holdout_status": "EVALUATED ONCE / FROZEN / UNTOUCHED",

    "final_station_count": 10,
    "final_feature_count": 73,

    "final_model": "Optimized Weighted Ensemble",

    "ensemble_weights": {
        "Tuned Extra Trees": 0.594466,
        "Tuned Random Forest": 0.405534
    },

    "final_holdout": {
        "rows": 68352,
        "MAE": 3.216121,
        "RMSE": 7.658390,
        "R2": 0.996634,
        "Accuracy_pm5_percent": 85.249883,
        "Accuracy_pm10_percent": 94.605864,
        "Accuracy_pm20_percent": 98.090765
    },

    "R5_status": "REPORTING PACKAGE CREATED",
    "R6_status": (
        "ALL-MODEL DIAGNOSTICS CELL PREPARED; "
        "RUN STATUS SHOULD BE VERIFIED AFTER RESUME"
    ),

    "SHAP_status": (
        "SKIPPED BY USER; DO NOT RECOMPUTE UNLESS REQUESTED"
    ),

    "deployment_status": (
        "PAUSED AFTER LIVE SOURCE INVESTIGATION"
    ),

    "checkpoint_created": datetime.now().isoformat()
}

status_path = os.path.join(
    CHECKPOINT_DIR,
    "session_status.json"
)

with open(
    status_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        status,
        f,
        indent=2
    )

print("Session status saved:")
print(status_path)


# ------------------------------------------------------------
# 2. SAVE DATAFRAMES CURRENTLY IN RUNTIME
# ------------------------------------------------------------

saved_dataframes = []

for name, obj in list(
    globals().items()
):

    if name.startswith("_"):
        continue

    if isinstance(
        obj,
        pd.DataFrame
    ):

        # Avoid enormous/unintended transient objects
        safe_name = (
            str(name)
            .replace("/", "_")
            .replace("\\", "_")
            .replace(" ", "_")
            .replace(":", "_")
        )

        output_path = os.path.join(
            SESSION_DIR,
            f"{safe_name}.parquet"
        )

        try:

            obj.to_parquet(
                output_path,
                index=False
            )

            saved_dataframes.append({
                "variable": name,
                "path": output_path,
                "rows": int(len(obj)),
                "columns": int(len(obj.columns))
            })

            print(
                f"Saved DataFrame: {name} "
                f"({len(obj):,} rows)"
            )

        except Exception as e:

            print(
                f"Skipped DataFrame {name}: {e}"
            )


# ------------------------------------------------------------
# 3. SAVE DATAFRAME MANIFEST
# ------------------------------------------------------------

dataframe_manifest_path = os.path.join(
    SESSION_DIR,
    "dataframe_manifest.json"
)

with open(
    dataframe_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        saved_dataframes,
        f,
        indent=2
    )

print(
    "\nDataFrame manifest saved:"
)

print(
    dataframe_manifest_path
)


# ------------------------------------------------------------
# 4. SAVE KNOWN CONFIG OBJECTS IF PRESENT
# ------------------------------------------------------------

config_names = [
    "final_config",
    "ensemble_config",
    "metadata",
    "status"
]

saved_configs = []

for name in config_names:

    if name in globals():

        obj = globals()[name]

        try:

            output_path = os.path.join(
                SESSION_DIR,
                f"{name}.json"
            )

            with open(
                output_path,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    obj,
                    f,
                    indent=2,
                    default=str
                )

            saved_configs.append({
                "variable": name,
                "path": output_path
            })

            print(
                f"Saved config: {name}"
            )

        except Exception as e:

            print(
                f"Skipped config {name}: {e}"
            )


# ------------------------------------------------------------
# 5. SAVE CONFIG MANIFEST
# ------------------------------------------------------------

config_manifest_path = os.path.join(
    SESSION_DIR,
    "config_manifest.json"
)

with open(
    config_manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        saved_configs,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 6. RECORD IMPORTANT EXISTING ARTIFACT PATHS
# ------------------------------------------------------------

important_artifacts = {

    "feature_data": os.path.join(
        BASE,
        "data/processed/delhi_10_station_feature_engineered.parquet"
    ),

    "final_10_station_raw": os.path.join(
        BASE,
        "data/processed/delhi_10_station_raw.parquet"
    ),

    "time_split": os.path.join(
        BASE,
        "outputs/reports/phase15_1_time_split_assignments.parquet"
    ),

    "feature_contract": os.path.join(
        BASE,
        "outputs/reports/phase15_15_final_freeze/phase15_15_feature_contract.csv"
    ),

    "final_freeze": os.path.join(
        BASE,
        "outputs/reports/phase15_15_final_freeze/phase15_15_final_freeze.json"
    ),

    "final_ensemble_config": os.path.join(
        BASE,
        "outputs/reports/phase15_10_final_ensemble_config.json"
    ),

    "final_extra_trees": os.path.join(
        BASE,
        "models/phase15_10_final_tuned_extra_trees_trainval.joblib"
    ),

    "final_random_forest": os.path.join(
        BASE,
        "models/phase15_10_final_tuned_random_forest_trainval.joblib"
    ),

    "final_holdout_predictions": os.path.join(
        BASE,
        "outputs/reports/phase15_12_final_holdout/phase15_12_final_holdout_predictions.parquet"
    ),

    "final_results_root": os.path.join(
        BASE,
        "outputs/FINAL_RESULTS"
    ),

    "R6_diagnostics_root": os.path.join(
        BASE,
        "outputs/FINAL_RESULTS/02_MODEL_RESULTS/R6_ALL_MODEL_DIAGNOSTICS"
    )
}

artifact_check = []

for key, path in important_artifacts.items():

    artifact_check.append({
        "artifact": key,
        "path": path,
        "exists": os.path.exists(path)
    })

artifact_check_df = pd.DataFrame(
    artifact_check
)

artifact_check_path = os.path.join(
    SESSION_DIR,
    "important_artifact_check.csv"
)

artifact_check_df.to_csv(
    artifact_check_path,
    index=False
)

print("\nImportant artifact check:")
display(
    artifact_check_df
)


# ------------------------------------------------------------
# 7. FINAL CHECKPOINT MANIFEST
# ------------------------------------------------------------

checkpoint_manifest = {
    "session_status": status_path,
    "dataframe_manifest": dataframe_manifest_path,
    "config_manifest": config_manifest_path,
    "important_artifact_check": artifact_check_path,
    "runtime_dataframe_count": len(
        saved_dataframes
    ),
    "runtime_config_count": len(
        saved_configs
    )
}

manifest_path = os.path.join(
    CHECKPOINT_DIR,
    "latest_checkpoint_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        checkpoint_manifest,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 8. FINAL MESSAGE
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PROJECT CHECKPOINT SAVED")
print("=" * 80)

print(
    "\nCheckpoint:"
)

print(
    manifest_path
)

print(
    "\nCurrent stage:"
)

print(
    "R.6 — All-model diagnostics prepared"
)

print(
    "\nFinal model remains:"
)

print(
    "Optimized Weighted ET + RF"
)

print(
    "\nFinal holdout remains:"
)

print(
    "FROZEN / UNTOUCHED"
)

print(
    "\nNo retraining performed."
)

print(
    "No retuning performed."
)

print(
    "No holdout modification performed."
)

print(
    "\nYou can safely disconnect the Colab runtime.")